In [ ]:
# -*- coding: utf-8 -*-
# =====================================================================================
#  [학생용] 결과기 개발 기본 틀 — 1번 셀
# =====================================================================================
#  이 셀은 완성된 결과기가 아닙니다. 1번 셀에 팀별 결과기를 구현한 뒤 사용합니다.
#  결과기 코랩은 아래 두 셀을 위에서 아래로 한 번 실행할 수 있어야 합니다.
#
#    1번 셀: 팀별 결과기 구현 — 이 파일의 코드
#    2번 셀: 공개 10문항 공통 러너 — 운영진 배포본, 팀 식별자 한 줄 외 수정 금지
# =====================================================================================


# -------------------------------------------------------------------------------------
# 0. 고정 기준 — 문서명과 생성 모델 계열
# -------------------------------------------------------------------------------------
OFFICIAL_DOCUMENT_NAMES = (
    "카카오계정 약관",
    "카카오 위치정보 이용약관",
    "카카오 통합서비스약관",
    "카카오 통합 약관",
)
REQUIRED_GENERATION_MODEL_FAMILY = "Qwen2.5-Instruct"


# =====================================================================================
# 1. 팀별 자유 구현 영역 — BM25 + 리랭커 하이브리드 검색 + Qwen2.5 생성
# =====================================================================================
import json as _json
import math as _math
import re as _re
import subprocess as _subprocess
import sys as _sys
import time as _time
import unicodedata as _unicodedata


def _pip_install(*pkgs):
    _subprocess.run([_sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)


def _pip_install_no_deps(*pkgs):
    """의존성 해석 없이 설치한다.

    bitsandbytes는 metadata에 torch를 요구사항으로 달고 있어 그냥 설치하면
    Colab의 torch를 다른 버전으로 갈아끼울 수 있다. 대회 규정상 torch 재설치는
    금지이고 런타임이 깨지면 0점이므로 --no-deps로 wheel만 올린다.
    (bitsandbytes wheel은 자체 CUDA 바이너리를 들고 있어 torch 버전에 비종속적이다.)
    """
    _subprocess.run(
        [_sys.executable, "-m", "pip", "install", "-q", "--no-deps", *pkgs],
        check=True,
    )


_pip_install("rank_bm25", "sentence-transformers")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("[환경]", _DEVICE, "|", torch.cuda.get_device_name(0) if _DEVICE == "cuda" else "GPU 없음")

# bitsandbytes는 7B를 4bit로 올릴 때만 쓰이고 CUDA 전용이다.
# GPU 없는 개발 환경에는 설치 가능한 wheel이 없을 수 있으므로 조건부로 설치하고,
# 실패해도 셀을 죽이지 않는다(아래 로딩부가 3B fp16으로 폴백한다).
_HAS_BNB = False

if _DEVICE == "cuda":
    try:
        _pip_install_no_deps("bitsandbytes")
        import bitsandbytes as _bnb

        _HAS_BNB = True
        print(f"[환경] bitsandbytes {_bnb.__version__}")
    except Exception as _exc:
        print(
            f"[경고] bitsandbytes 준비 실패({type(_exc).__name__}: {_exc}) — "
            "생성 모델은 3B fp16으로 폴백합니다."
        )

import gc

# 같은 Colab 런타임에서 셀을 다시 실행했을 때
# 이전 모델이 GPU 메모리에 남아 OOM이 나는 것을 방지한다.
for _old_name in ("_MODEL", "_RERANKER"):
    _old_obj = globals().pop(_old_name, None)

    if _old_obj is not None:
        del _old_obj

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(
    "[환경]",
    _DEVICE,
    "|",
    torch.cuda.get_device_name(0) if _DEVICE == "cuda" else "GPU 없음",
)

if _DEVICE == "cuda":
    free_mem, total_mem = torch.cuda.mem_get_info()

    print(
        f"[GPU 메모리] "
        f"free={free_mem / 1024**3:.2f} GB / "
        f"total={total_mem / 1024**3:.2f} GB"
    )

# -------------------------------------------------------------------------------------
# 1-1. 약관 원문 — 조 단위, 요약 없이 원문 그대로 노트북에 직접 포함한다.
#      외부 약관 파일 다운로드 없이 내장 원문을 사용한다. (72개 조항: 카카오계정 17 /
#      위치정보 16 / 통합서비스 18 / 통합 21. 출제 기준일 2026-08-04 기준 각 약관 공식 시행본)
# -------------------------------------------------------------------------------------
_ARTICLES_JSON_FALLBACK = r'''[{"doc":"카카오계정 약관","article_no":1,"title":"목적","text":"주식회사 카카오(이하 ‘회사’)가 제공하는 서비스를 이용해 주셔서 감사합니다. 회사는 여러분이 다양한 인터넷과 모바일 서비스를 좀 더 편리하게 이용할 수 있도록 회사 또는 관계사의 개별 서비스에 모두 접속 가능한 통합로그인계정 체계를 만들고 그에 적용되는 '카카오계정 약관(이하 '본 약관')을 마련하였습니다. 본 약관은 여러분이 카카오계정 서비스를 이용하는 데 필요한 권리, 의무 및 책임사항, 이용조건 및 절차 등 기본적인 사항을 규정하고 있으므로 조금만 시간을 내서 주의 깊게 읽어주시기 바랍니다."},{"doc":"카카오계정 약관","article_no":2,"title":"약관의 효력 및 변경","text":"①본 약관의 내용은 카카오계정 웹사이트 또는 개별 서비스의 화면에 게시하거나 기타의 방법으로 공지하고, 본 약관에 동의한 여러분 모두에게 그 효력이 발생합니다.\n②회사는 필요한 경우 관련법령을 위배하지 않는 범위 내에서 본 약관을 변경할 수 있습니다. 본 약관이 변경되는 경우 회사는 변경사항을 시행일자 15일 전부터 여러분에게 서비스 공지사항에서 공지 또는 통지하는 것을 원칙으로 하며, 피치 못하게 여러분에게 불리한 내용으로 변경할 경우에는 그 시행일자 30일 전부터 카카오계정에 등록된 이메일 주소로 이메일(이메일주소가 없는 경우 서비스 내 전자쪽지 발송, 서비스 내 알림 메시지를 띄우는 등의 별도의 전자적 수단) 발송 또는 여러분이 등록한 휴대폰번호로 카카오톡 메시지 또는 문자메시지 발송하는 방법 등으로 개별적으로 알려 드리겠습니다.\n③회사가 전항에 따라 공지 또는 통지를 하면서 공지 또는 통지일로부터 개정약관 시행일 7일 후까지 거부의사를 표시하지 아니하면 승인한 것으로 본다는 뜻을 명확하게 고지하였음에도 여러분의 의사표시가 없는 경우에는 변경된 약관을 승인한 것으로 봅니다. 여러분이 개정약관에 동의하지 않을 경우 여러분은 이용계약을 해지할 수 있습니다."},{"doc":"카카오계정 약관","article_no":3,"title":"약관 외 준칙","text":"본 약관에 규정되지 않은 사항에 대해서는 관련법령 또는 회사가 정한 개별 서비스의 이용약관, 운영정책 및 규칙 등(이하 ‘세부지침’)의 규정에 따릅니다."},{"doc":"카카오계정 약관","article_no":4,"title":"용어의 정의","text":"①본 약관에서 사용하는 용어의 정의는 다음과 같습니다.\n1.카카오계정: 회사 또는 관계사가 제공하는 개별 서비스를 하나의 로그인계정과 비밀번호로 회원 인증, 회원정보 변경, 회원 가입 및 탈퇴 등을 관리할 수 있도록 회사가 정한 로그인계정 정책을 말합니다.\n2.회원: 카카오계정이 적용된 개별 서비스 또는 카카오계정 웹사이트에서 본 약관에 동의하고, 카카오계정을 이용하는 자를 말합니다.\n3.관계사: 회사와 제휴 관계를 맺고 카카오계정을 공동 제공하기로 합의한 법인을 말합니다. 개별 관계사는 카카오 기업사이트에서 확인할 수 있고 추후 추가/변동될 수 있으며 관계사가 추가/변동될 때에는 카카오 기업사이트에 변경 사항을 게시합니다.\n4.개별 서비스: 카카오계정을 이용하여 접속 가능한 회사 또는 관계사가 제공하는 서비스를 말합니다. 개별 서비스는 추후 추가/변동될 수 있으며 서비스가 추가/변동될 때에는 카카오 기업사이트에 변경 사항을 게시합니다.\n5.카카오계정 웹사이트: 회원이 온라인을 통해 카카오계정 정보를 조회 및 수정할 수 있는 인터넷 사이트를 말합니다.\n6.카카오계정 정보 : 카카오계정을 이용하기 위해 회사가 정한 필수 내지 선택 입력 정보로서 카카오계정 웹사이트 또는 개별 서비스 내 카카오계정 설정 화면을 통해 정보 확인, 변경 처리 등을 관리할 수 있는 회원정보 항목을 말합니다.\n7.이용기관 : 제11조에 따른 디지털카드서비스와 관련하여 디지털카드를 제출받거나 확인하여 자신의 업무, 영업에 활용하는 제3자를 말합니다."},{"doc":"카카오계정 약관","article_no":5,"title":"계약의 성립","text":"①카카오계정 이용 신청은 개별 서비스 또는 카카오계정 웹사이트 회원가입 화면에서 여러분이 카카오계정 정보에 일정 정보를 입력하는 방식으로 이루어집니다.\n②카카오계정 이용계약은 여러분이 본 약관의 내용에 동의한 후 본 조 제1항에서 정한 이용신청을 하면 회사가 입력된 일정 정보를 인증한 후 가입을 승낙함으로써 체결됩니다."},{"doc":"카카오계정 약관","article_no":6,"title":"카카오계정 이용의 제한","text":"①제5조에 따른 가입 신청자에게 회사는 원칙적으로 카카오계정의 이용을 승낙합니다. 다만, 회사는 아래 각 호의 경우에는 그 사유가 해소될 때까지 승낙을 유보하거나 승낙하지 않을 수 있습니다. 특히, 여러분이 만 14세 미만인 경우에는 부모님 등 법정대리인의 동의가 있는 경우에만 카카오계정을 생성할 수 있습니다.\n1.회사가 본 약관 또는 세부지침에 의해 여러분의 카카오계정을 삭제하였던 경우\n2.여러분이 다른 사람의 명의나 이메일 주소 등 개인정보를 이용하여 카카오계정을 생성하려 한 경우\n3.카카오계정 생성 시 필요한 정보를 입력하지 않거나 허위의 정보를 입력한 경우\n4.제공 서비스 설비 용량에 현실적인 여유가 없는 경우\n5.서비스 제공을 위한 기술적인 부분에 문제가 있다고 판단되는 경우\n6.기타 회사가 재정적, 기술적으로 필요하다고 인정하는 경우\n7.회사로부터 회원자격정지 조치 등을 받은 회원이 그 조치기간에 이용계약을 임의로 해지하고 재이용을 신청하는 경우\n8.기타 관련법령에 위배되거나 세부지침 등 회사가 정한 기준에 반하는 경우\n②만약, 여러분이 위 조건에 위반하여 카카오계정을 생성한 것으로 판명된 때에는 회사는 즉시 여러분의 카카오계정 이용을 정지시키거나 카카오계정을 삭제하는 등 적절한 제한을 할 수 있습니다."},{"doc":"카카오계정 약관","article_no":7,"title":"카카오계정 제공","text":"①회사가 개별 서비스와 연동하여 카카오계정에서 제공하는 서비스(이하 “카카오계정 서비스” 또는 “서비스”) 내용은 아래와 같습니다.\n1.통합로그인 : 카카오계정이 적용된 개별 서비스에서 하나의 카카오계정과 비밀번호로 로그인할 수 있는 통합 회원 인증 서비스를 이용할 수 있습니다.\n2.SSO(Single Sign On): 웹브라우저나 특정 모바일 기기에서 카카오계정 1회 로그인으로 여러분이 이용 중인 개별 서비스간 추가 로그인 없이 자동 접속 서비스를 이용할 수 있습니다.\n3.카카오계정 정보 통합 관리 : 개별 서비스 이용을 위해 카카오계정 정보를 통합 관리합니다. 또한, 여러분이 이용하고자 하는 개별 서비스의 유형에 따라 전문기관을 통한 실명확인 및 본인인증을 요청할 수 있고, 이를 카카오계정 정보로 저장합니다.\n4.사업자/단체 카카오계정 : 사업자/단체 명의로 카카오 서비스를 이용하기 위해 만들어진 카카오계정으로서 해당 사업자/단체의 책임 하에 권한을 위임받은 담당자가 이용, 관리할 수 있는 계정 서비스입니다.\n5.기타 회사가 제공하는 서비스\n②회사는 더 나은 카카오계정 서비스의 제공을 위하여 여러분에게 서비스의 이용과 관련된 각종 고지, 관리 메시지 및 기타 광고를 비롯한 다양한 정보를 서비스화면 내에 표시하거나 여러분의 이메일로 전송할 수 있습니다. 광고성 정보 전송의 경우에는 사전에 수신에 동의한 경우에만 전송합니다."},{"doc":"카카오계정 약관","article_no":8,"title":"카카오계정 서비스의 변경 및 종료","text":"①회사는 카카오계정 서비스를 365일, 24시간 쉬지 않고 제공하기 위하여 최선의 노력을 다합니다. 다만, 아래 각 호의 경우 카카오계정 서비스의 전부 또는 일부를 제한하거나 중지할 수 있습니다.\n1.카카오계정 서비스용 설비의 유지·보수 등을 위한 정기 또는 임시 점검의 경우\n2.정전, 제반 설비의 장애 또는 이용량의 폭주 등으로 정상적인 카카오계정 이용에 지장이 있는 경우\n3.관계사와의 계약 종료, 정부의 명령/규제, 서비스/회원 정책 변경 등 회사의 제반 사정으로 카카오계정 서비스를 유지할 수 없는 경우\n4.기타 천재지변, 국가비상사태 등 불가항력적 사유가 있는 경우\n②전항에 의한 카카오계정 서비스 중단의 경우에는 미리 제14조에서 정한 방법으로 여러분에게 통지 내지 공지하겠습니다. 이 때 원만한 서비스 및 정책 변경 등을 위하여 서비스 이용 시 재로그인 또는 추가적인 동의 절차 등이 필요할 수 있습니다. 다만, 회사로서도 예측할 수 없거나 통제할 수 없는 사유(회사의 과실이 없는 디스크 내지 서버 장애, 시스템 다운 등)로 서비스가 중단된 경우에는 사전 통지 내지 공지를 할 수 없습니다. 이러한 경우에도 회사가 상황을 파악하는 즉시 최대한 빠른 시일 내에 서비스를 복구하도록 노력하되, 2시간 이상 복구가 지연될 시 카카오 서비스 공지사항, 카카오 고객센터 공지사항 등에 게시하여 알려 드리겠습니다."},{"doc":"카카오계정 약관","article_no":9,"title":"카카오계정 관리","text":"①카카오계정은 여러분 본인만 이용할 수 있으며, 다른 사람이 여러분의 카카오계정을 이용하도록 허락할 수 없습니다. 그리고 여러분은 다른 사람이 여러분의 카카오계정을 무단으로 사용할 수 없도록 직접 비밀번호를 관리하여야 합니다. 회사는 다른 사람이 여러분의 카카오계정을 무단으로 사용하는 것을 막기 위하여 비밀번호 입력 및 추가적인 본인 확인 절차를 거치도록 할 수 있습니다. 만약 무단 사용이 발견된다면, 고객센터를 통하여 회사에게 알려주시기 바라며, 회사는 무단 사용을 막기 위한 방법을 여러분에게 안내하도록 하겠습니다.\n②여러분은 카카오계정 웹사이트 또는 개별 서비스 내 카카오계정 설정 화면을 통하여 여러분의 카카오계정 정보를 열람하고 수정할 수 있습니다. 다만, 카카오계정 서비스의 제공 및 관리를 위해 필요한 카카오계정, 전화번호, 단말기 식별번호, 기타 본인확인정보 등 일부 정보는 수정이 불가능할 수 있으며, 수정하는 경우에는 추가적인 본인 확인 절차가 필요할 수 있습니다.\n③여러분이 이용 신청 시 알려주신 내용에 변동이 있을 때, 전항에 따라 직접 수정하시거나, 고객센터를 통하여 회사에 알려 주시기 바랍니다. 여러분이 카카오계정 정보를 적시에 수정하지 않아 발생하는 문제에 대하여 회사의 고의 또는 과실이 없는 한 회사는 책임을 부담하지 아니합니다."},{"doc":"카카오계정 약관","article_no":10,"title":"사업자/단체 카카오계정","text":"①사업자/단체 카카오계정은 사업자등록번호 또는 고유번호가 있는 사업자/단체가 권한을 위임받은 담당자(이하 본조에서 ‘담당자’)를 통해 만들어 이용할 수 있습니다. 사업자/단체 카카오계정의 이용 및 관리에 관한 책임은 해당 사업자/단체에 있으며, 회사는 이와 관련한 책임을 지지 않습니다.\n②사업자/단체 카카오계정은 계정 정보에 등록된 담당자 1인만 이용할 수 있으며, 이를 다른 사람에게 공유하는 것은 금지됩니다.\n③사업자/단체 카카오계정은 개인 카카오계정으로 전환할 수 없고, 다른 개인 또는 법인 등 제3자에게 양도할 수 없습니다.\n④사업자/단체 카카오계정은 일부 카카오 서비스의 가입 및 이용이 제한되며, 가입 및 이용이 제한되는 서비스는 정책에 따라 변경될 수 있습니다.\n⑤사업자/단체 카카오계정의 정보 변경 또는 담당자 변경 요청에 대해 회사는 해당 계정에 대한 정당한 권한이 있는지 확인하기 위하여 일정한 증빙서류를 요청할 수 있습니다.\n⑥사업자/단체 카카오계정은 사업자/단체에 귀속되는 것으로, 담당자는 해당 계정에 대해 권리를 주장할 수 없습니다.\n⑦본 조에서 정하고 있는 내용 외에 사업자/단체 카카오계정과 관련된 상세한 사항은 사업자/단체 카카오계정 운영정책에 따르며, 회사는 게시판 및 고객센터 도움말 페이지 등을 통하여 회원에게 안내합니다."},{"doc":"카카오계정 약관","article_no":11,"title":"디지털카드 서비스","text":"①회사는 회사를 포함한 제휴 발급기관의 요청에 따라 회원의 카카오계정에 자격증명, 티켓, 아이템 등의 디지털카드를 발급하고 이를 제휴 이용기관에 제출하는 등의 활용을 할 수 있도록 하는 서비스(이하 ‘디지털카드 서비스’라 합니다)를 제공합니다.\n②디지털카드는 회원 본인의 신청에 따라 발급되거나, 이용자가 일정한 조건을 충족한 경우 또는 발급기관의 요청을 받은 경우에는 자동으로 발급될 수 있습니다. 단, 회사는 디지털카드 발급 과정에서 추가적인 인증 또는 이용 등의 동의를 요청할 수 있고, 해외 거주 또는 외국인 회원의 경우 디지털카드의 발급 및 이용이 제한 될 수 있습니다.\n③회사는 제휴 발급기관이 제공하는 정보를 디지털카드에 담거나 표시할 뿐, 제휴 발급기관이 발급한 디지털카드의 내용에 대한 검증 및 적법성 등에 대한 보증을 하지 않습니다.\n④회사, 발급기관 또는 이용기관(이하 발급기관과 이용기관을 통칭하여 ‘제휴사’라 합니다)이 제공하는 서비스에서 회원의 디지털카드 정보를 조회하거나 표시, 노출할 수 있습니다.\n⑤디지털카드는 발급기관의 필요와 요청에 따라 회수 또는 수정될 수 있습니다. 회수된 디지털카드 및 디지털카드에 담긴 정보는 복구할 수 없습니다.\n⑥디지털카드에는 발급기관이 설정한 유효기간이 있으며, 유효기간이 경과하는 경우 제휴사의 정책에 따라 디지털카드의 기능을 사용할 수 없거나 기타 제출 등의 활용에 제한이 있을 수 있습니다. 또한, 디지털카드의 활용처는 제휴사의 사정에 따라 변동될 수 있고, 회사는 디지털카드의 영속성을 보장하지 않으며, 기능 외의 금전적 가치도 인정하지 않습니다.\n⑦회원이 카카오계정을 탈퇴하는 경우 해당 카카오계정에 발급되어 있는 디지털카드는 삭제되고, 동일한 디지털카드의 재발급이 불가능할 수 있습니다.\n⑧회원은 디지털카드 서비스를 이용함에 있어서 아래 각 호의 행위는 하여서는 안 됩니다.\n1.서비스 이용 시 허위 사실을 기재하거나, 타인의 명의 및 정보를 도용하여 회사가 제공하는 서비스 또는 디지털카드를 이용하는 행위\n2.디지털카드 정보를 회원 본인이 아닌 제3자가 사용하도록 대여하는 행위\n3.유효하지 않은 디지털카드를 비정상적 목적으로 사용하는 행위\n4.서비스에서 회사가 게시한 정보의 무단 변경 또는 회사가 정한 정보 이외의 정보(컴퓨터 프로그램 등)등의 송신 또는 게시하는 행위\n5.회사가 정하지 않은 비정상적인 방법으로 서비스를 이용하거나 시스템에 접근하는 행위\n6.회사가 정하지 않은 비정상적인 방법으로 부당하게 디지털카드를 주고 받는 행위(예: 디지털카드의 유상거래, 이용자간 합의되지 않은 전송에 따른 탈취 행위, 정상적으로 안내되지 않은 방법에 의한 거래 행위 등)\n7.기타 관련법령, 회사의 약관 및 운영정책을 위반하여 회사나 제휴사 또는 다른 제3자에게 손해를 끼치거나 손해를 끼칠 것으로 합리적으로 예상되는 경우\n⑨회사는 디지털카드의 활용과 관련하여 회원, 발급기관, 이용기관 간의 관계에서 어떠한 책임도 부담하지 않으며, 회사는 발급기관과 이용기관의 귀책사유로 인하여 회원에게 발생한 손해에 대하여 회사의 귀책사유가 없는 한 책임을 지지 않습니다.\n⑩본 조에서 정하고 있는 내용 외에 디지털카드 서비스와 관련된 상세한 사항은 디지털카드 서비스 운영정책에 따르며, 회사는 서비스 공지사항 및 고객센터 도움말 페이지 등을 통하여 회원에게 안내합니다."},{"doc":"카카오계정 약관","article_no":12,"title":"회원의 의무","text":"①여러분이 카카오계정 서비스를 이용할 때 아래 각 호의 행위는 하여서는 안 됩니다.\n1.이용 신청 또는 변경 시 허위 사실을 기재하거나, 다른 회원의 카카오계정 및 비밀번호를 도용, 부정하게 사용하거나, 다른 사람의 명의를 사용하거나 명의자의 허락 없이 문자메시지(SMS) 인증 등을 수행하는 행위\n2.타인의 명예를 손상시키거나 불이익을 주는 행위\n3.게시판 등에 음란물을 게재하거나 음란사이트를 연결(링크)하는 행위\n4.회사 또는 제3자의 저작권 등 기타 권리를 침해하는 행위\n5.공공질서 및 미풍양속에 위반되는 내용의 정보, 문장, 도형, 음성 등을 타인에게 유포하는 행위\n6.카카오계정 서비스와 관련된 설비의 오동작이나 정보 등의 파괴 및 혼란을 유발시키는 컴퓨터 바이러스 감염 자료를 등록 또는 유포하는 행위\n7.카카오계정 서비스의 운영을 고의로 방해하거나 안정적 운영을 방해할 수 있는 정보 및 수신자의 명시적인 수신거부의사에 반하여 광고성 정보 또는 스팸메일(Spam Mail)을 전송하는 행위\n8.회사의 동의 없이 서비스 또는 이에 포함된 소프트웨어의 일부를 복사, 수정, 배포, 판매, 양도, 대여, 담보제공하거나 타인에게 그 이용을 허락하는 행위와 소프트웨어를 역설계하거나 소스 코드의 추출을 시도하는 등 서비스를 복제, 분해 또는 모방하거나 기타 변형하는 행위\n9.타인으로 가장하는 행위 및 타인과의 관계를 허위로 명시하는 행위\n10.다른 회원의 개인정보를 수집, 저장, 공개하는 행위\n11.자기 또는 타인에게 재산상의 이익을 주거나 타인에게 손해를 가할 목적으로 허위의 정보를 유통시키는 행위\n12.윤락행위를 알선하거나 음행을 매개하는 내용의 정보를 유통시키는 행위\n13.수치심이나 혐오감 또는 공포심을 일으키는 말이나 음향, 글이나 화상 또는 영상을 계속하여 상대방에게 도달하게 하여 상대방의 일상적 생활을 방해하는 행위\n14.관련 법령에 의하여 그 전송 또는 게시가 금지되는 정보(컴퓨터 프로그램 포함)의 전송 또는 게시 행위\n15.회사 또는 관계사의 직원이나 운영자를 가장하거나 사칭하여 또는 타인의 명의를 도용하여 글을 게시하거나 E-mail, 카카오톡 메시지 등을 발송하는 행위\n16.컴퓨터 소프트웨어, 하드웨어, 전기통신 장비의 정상적인 가동을 방해, 파괴할 목적으로 고안된 소프트웨어 바이러스, 기타 다른 컴퓨터 코드, 파일, 프로그램을 포함하고 있는 자료를 게시하거나 E-mail, 카카오톡 메시지 등으로 발송하는 행위\n17.기타 불법한 행위\n②여러분은 서비스의 이용권한, 기타 이용계약상 지위를 타인에게 양도·증여할 수 없으며, 담보로 제공할 수 없습니다.\n③혹시라도 여러분이 관련 법령, 회사의 모든 약관 또는 정책을 준수하지 않는다면, 회사는 여러분의 위반행위 등을 조사할 수 있고, 여러분의 계정・서비스 이용을 잠시 또는 계속하여 중단하거나, 재가입에 제한을 둘 수도 있습니다. 또한 여러분이 서비스와 관련된 설비의 오작동이나 시스템의 파괴 및 혼란을 유발하는 등 서비스 제공에 악영향을 미치거나 안정적 운영을 심각하게 방해한 경우, 회사는 이러한 위험 활동이 확인된 여러분의 계정들에 대하여 이용제한을 할 수 있습니다. 다만, 여러분은 이용제한과 관련하여 조치 결과가 불만족스러울 경우 고객센터를 통해 이의를 제기할 수 있습니다.\n④본 조에서 정한 사항 및 그 밖에 카카오계정 서비스의 이용에 관한 자세한 사항은 카카오 운영정책 등을 참고해 주시기 바랍니다."},{"doc":"카카오계정 약관","article_no":13,"title":"개인정보의 보호","text":"여러분의 개인정보의 안전한 처리는 회사에게 있어 가장 중요한 일 중 하나입니다. 여러분의 개인정보는 서비스의 원활한 제공을 위하여 여러분이 동의한 목적과 범위 내에서만 이용됩니다. 법령에 의하거나 여러분이 별도로 동의하지 아니하는 한 회사가 여러분의 개인정보를 제3자에게 제공하는 일은 결코 없으므로, 안심하셔도 좋습니다. 회사가 여러분의 개인정보를 안전하게 처리하기 위하여 기울이는 노력이나 기타 자세한 사항은 카카오 개인정보처리방침을 참고하여 주십시오."},{"doc":"카카오계정 약관","article_no":14,"title":"회원에 대한 통지 및 공지","text":"회사는 여러분과의 의견 교환을 소중하게 생각합니다. 여러분은 언제든지 고객센터에 방문하여 의견을 개진할 수 있습니다. 서비스 이용자 전체에 대한 공지는 칠(7)일 이상 서비스 공지사항란에 게시함으로써 효력이 발생합니다. 여러분에게 중대한 영향을 미치는 사항의 경우에는 카카오계정에 등록된 이메일 주소로 이메일(이메일주소가 없는 경우 서비스 내 전자쪽지 발송, 서비스 내 알림 메시지를 띄우는 등의 별도의 전자적 수단) 발송 또는 여러분이 등록한 휴대폰번호로 카카오톡 메시지 또는 문자메시지 발송하는 방법 등으로 개별적으로 알려 드리겠습니다."},{"doc":"카카오계정 약관","article_no":15,"title":"이용계약 해지","text":"①여러분이 카카오계정 이용을 더 이상 원치 않는 때에는 언제든지 서비스 내 제공되는 메뉴를 이용하여 이용계약의 해지 신청을 할 수 있으며, 회사는 법령이 정하는 바에 따라 신속히 처리하겠습니다.\n②회사는 여러분이 카카오계정 서비스를 이용하기 위해 카카오계정 로그인 혹은 접속한 기록이 없는 경우 여러분이 등록한 이메일주소, 휴대폰번호로 이메일, 문자메시지 또는 카카오톡 메시지를 보내는 등 기타 유효한 수단으로 통지 후 여러분의 카카오계정 정보를 파기하거나 분리 보관할 수 있으며, 이로 인해 카카오계정 서비스 이용을 위한 필수적인 정보가 부족할 경우 이용계약이 해지될 수도 있습니다. 이와 관련된 보다 자세한 사항은 카카오 운영정책의 서비스 장기 미이용 처리 정책을 참고하시기 바랍니다.\n③이용계약이 해지되면 법령 및 개인정보 처리방침에 따라 여러분의 정보를 보유하는 경우를 제외하고는 여러분의 카카오계정 정보 및 카카오계정으로 이용하였던 개별 서비스 데이터는 삭제됩니다. 다만, 여러분이 개별 서비스 내에서 작성한 게시물 등 모든 데이터의 삭제와 관련한 사항은 개별 서비스의 약관에 따릅니다.\n④이용계약이 해지된 경우라도 여러분은 다시 회사에 대하여 이용계약의 체결을 신청할 수 있습니다."},{"doc":"카카오계정 약관","article_no":16,"title":"손해배상","text":"①회사는 법령상 허용되는 한도 내에서 서비스와 관련하여 본 약관에 명시되지 않은 어떠한 구체적인 사항에 대한 약정이나 보증을 하지 않습니다. 또한, 회사는 CP(Contents Provider)가 제공하거나 회원이 작성하는 등의 방법으로 서비스에 게재된 정보, 자료, 사실의 신뢰도, 정확성 등에 대해서는 보증을 하지 않으며, 회사의 과실 없이 발생된 여러분의 손해에 대하여는 책임을 부담하지 아니합니다.\n②회사는 회사의 과실로 인하여 여러분이 손해를 입게 될 경우 본 약관 및 관련 법령에 따라 여러분의 손해를 배상하겠습니다. 다만 회사는 회사의 과실 없이 발생된 아래와 같은 손해에 대해서는 책임을 부담하지 않습니다. 또한 회사는 법률상 허용되는 한도 내에서 간접 손해, 특별 손해, 결과적 손해, 징계적 손해, 및 징벌적 손해에 대한 책임을 부담하지 않습니다.\n1.천재지변 또는 이에 준하는 불가항력의 상태에서 발생한 손해\n2.여러분의 귀책사유로 서비스 이용에 장애가 발생한 경우\n3.서비스에 접속 또는 이용과정에서 발생하는 개인적인 손해\n4.제3자가 불법적으로 회사의 서버에 접속하거나 서버를 이용함으로써 발생하는 손해\n5.제3자가 회사 서버에 대한 전송 또는 회사 서버로부터의 전송을 방해함으로써 발생하는 손해\n6.제3자가 악성 프로그램을 전송 또는 유포함으로써 발생하는 손해\n7.전송된 데이터의 생략, 누락, 파괴 등으로 발생한 손해, 명예훼손 등 제3자가 서비스를 이용하는 과정에서 발생된 손해\n8.기타 회사의 고의 또는 과실이 없는 사유로 인해 발생한 손해"},{"doc":"카카오계정 약관","article_no":17,"title":"분쟁의 해결","text":"본 약관 또는 서비스는 대한민국법령에 의하여 규정되고 이행됩니다. 서비스 이용과 관련하여 회사와 여러분 간에 분쟁이 발생하면 이의 해결을 위해 성실히 협의할 것입니다. 그럼에도 불구하고 해결되지 않으면 민사소송법상의 관할법원에 소를 제기할 수 있습니다."},{"doc":"카카오 위치정보 이용약관","article_no":1,"title":"목적","text":"본 약관은 주식회사 카카오(이하 \"회사\")가 제공하는 사물위치정보 및 위치기반 서비스(이하, 위치정보 서비스)에 대해 회사와 서비스를 이용하는 이용자간의 권리·의무 및 책임사항, 기타 필요한 사항 규정을 목적으로 합니다."},{"doc":"카카오 위치정보 이용약관","article_no":2,"title":"이용약관의 효력 및 변경","text":"①본 약관은 이용자가 본 약관에 동의하고 회사가 정한 절차에 따라 위치정보 서비스의 이용자로 등록됨으로써 효력이 발생합니다.\n②이용자가 본 약관의 “동의하기” 버튼을 클릭하였을 경우 본 약관의 내용을 모두 읽고 이를 충분히 이해하였으며, 그 적용에 동의한 것으로 봅니다.\n③회사는 위치정보 서비스의  변경사항을 반영하기 위한 목적 등으로 필요한 경우 관련 법령을 위배하지 않는 범위에서 본 약관을 수정할 수 있습니다.\n④약관이 변경되는 경우 회사는 변경사항을 그 적용일자 최소 15일 전에 회사의 홈페이지 또는 서비스 공지사항 등(이하, 홈페이지 등)을 통해 공지합니다. 다만, 개정되는 내용이 이용자 권리의 중대한 변경을 발생시키는 경우 적용일 최소 30일 전에  이메일(이메일주소가 없는 경우 서비스 내 전자쪽지 발송, 서비스 내 알림 메시지를 띄우는 등의 별도의 전자적 수단) 발송 또는 등록한 휴대폰번호로 카카오톡 메시지 또는 문자메시지를 발송하는 방법 등으로 개별적으로 고지합니다.\n⑤회사가 전항에 따라 공지 또는 통지를 하면서 공지 또는 통지일로부터 개정약관 시행일 7일 후까지 거부의사를 표시하지 아니하면 승인한 것으로 본다는 뜻을 명확하게 고지하였음에도 이용자의 의사표시가 없는 경우에는 변경된 약관을 승인한 것으로 봅니다. 이용자가 개정약관에 동의하지 않을 경우 본 약관에 대한 동의를 철회할 수 있습니다."},{"doc":"카카오 위치정보 이용약관","article_no":3,"title":"약관 외 준칙","text":"이 약관에 명시되지 않은 사항에 대해서는 위치 정보의 보호 및 이용 등에 관한 법률, 개인정보보호법, 전기통신사업법, 정보통신망 이용촉진 및 정보보호 등에 관한 법률 등 관계법령 및 회사가 정한 지침 등의 규정에 따릅니다."},{"doc":"카카오 위치정보 이용약관","article_no":4,"title":"서비스의 내용","text":"회사는 위치정보사업자로부터 수집한 이용자의 위치정보 또는 직접 수집한 사물위치정보를 이용하여 아래와 같은 위치정보 서비스를 제공합니다.\n①검색결과 제공 및 콘텐츠 추천 : 이용자의 위치나 경로를 바탕으로 관련 정보나 콘텐츠를 검색하거나 추천해주는 서비스를 제공합니다.\n②생활편의 서비스 제공 : 이용자의 위치에 따른 길찾기, 경로 또는 이동수단 추천, 경로 안내 및 알림 서비스를 제공합니다.\n③위치 기반 콘텐츠 분류(Geo Tagging) : 이용자가 작성한 게시글, 사진, 영상 등에 위치정보를 저장하거나, 위치를 기반으로 콘텐츠를 분류하는 기능을 제공합니다.\n④위치기반 소셜 서비스 제공 : 내 위치를 다른 이용자와 공유하거나 콘텐츠 남기기 등 인터랙션을 포함한 위치 서비스를 제공합니다.\n⑤위치기반 광고 : 이용자의 위치정보를 활용한 광고성 정보 안내, 검색 및 디스플레이 광고소재 제공, 맞춤형 광고를 제공합니다."},{"doc":"카카오 위치정보 이용약관","article_no":5,"title":"서비스 이용요금","text":"회사가 제공하는 위치정보 서비스는 무료입니다.\n단, 무선 서비스 이용 시 발생하는 데이터 통신료는 별도이며, 이용자가 가입한 각 이동통신사의 정책에 따릅니다."},{"doc":"카카오 위치정보 이용약관","article_no":6,"title":"서비스의 변경・제한・중지","text":"①회사는 정책변경 또는 관련법령 변경 등과 같은 제반 사정을 이유로 위치기반서비스를 유지할 수 없는 경우 위치기반서비스의 전부 또는 일부를 변경·제한·중지할 수 있습니다.\n②회사는 아래 각호의 경우에는 이용자의 서비스 이용을 제한하거나 중지시킬 수 있습니다.\n1.이용자가 회사 서비스의 운영을 고의 또는 중과실로 방해하는 경우\n2.서비스용 설비 점검, 보수 또는 공사로 인하여 부득이한 경우\n3.전기통신사업법에 규정된 기간통신사업자가 전기통신 서비스를 중지했을 경우\n4.국가비상사태, 서비스 설비의 장애 또는 서비스 이용의 폭주 등으로 서비스 이용에 지장이 있는 때\n5.기타 중대한 사유로 인하여 회사가 서비스 제공을 지속하는 것이 부적당하다고 인정하는 경우\n③회사가 제1항 및 제2항의 규정에 의하여 서비스 이용을 제한하거나 중지한 때에는 그 사유 및 제한기간 등을 회사 홈페이지 등을 통해 사전 공지하거나 이용자에게 통지합니다."},{"doc":"카카오 위치정보 이용약관","article_no":7,"title":"개인위치정보의 이용 또는 제공","text":"①회사는 개인위치정보를 이용하여 위치기반서비스를 제공하는 경우 본 약관에 고지하고 동의를 받습니다.\n②회사는 이용자의 동의 없이 개인위치정보를 제3자에게 제공하지 않으며, 제3자에게 제공하는 경우에는 제공받는 자 및 제공목적을 사전에 이용자에게 고지하고 동의를 받습니다.\n③제2항에 따라 개인위치정보를 이용자가 지정하는 제3자에게 제공하는 경우 개인위치정보를 수집한 통신단말장치 또는 전자우편주소로 매회 이용자에게 제공받는 자, 제공일시 및 제공목적을 즉시 통지합니다. 단, 아래의 경우 이용자가 미리 특정하여 지정한 통신단말장치 또는 전자우편주소, 온라인게시 등으로 통지합니다.\n1.개인위치정보를 수집한 당해 통신단말장치가 문자, 음성 또는 영상의 수신기능을 갖추지 아니한 경우\n2.이용자의 개인위치정보를 수집한 통신단말장치 외의 통신단말장치 또는 전자우편주소, 온라인게시 등으로 통보할 것을 미리 요청한 경우\n위치정보 제공 현황 자세히 보기"},{"doc":"카카오 위치정보 이용약관","article_no":8,"title":"위치정보 수집·이용·제공사실 확인자료의 보관","text":"회사는 위치정보의 보호 및 이용 등에 관한 법률 제16조 제2항에 근거하여 위치정보 수집·이용·제공사실 확인자료를 위치정보시스템에 자동으로 기록·보존하며, 해당 자료는 6개월간 보관합니다."},{"doc":"카카오 위치정보 이용약관","article_no":9,"title":"개인위치정보의 보유 목적 및 보유기간","text":"회사는 위치기반서비스 제공을 위해 아래와 같이 개인위치정보를 보유합니다.\n①본 약관 제4조 따른 위치기반서비스 이용 및 제공 목적 달성한 때에는 지체없이 개인위치정보를 파기합니다.\n②다만, 이용자가 작성한 게시물 또는  콘텐츠와 함께 위치정보가 저장되는 서비스의 경우  해당 게시물 또는  콘텐츠의 보관기간 동안 개인위치정보가 보관됩니다.\n③그 외 위치기반서비스 제공을 위해  필요한 경우 이용목적 달성을 위해 필요한 최소한의 기간 동안 개인위치정보를 보유할 수 있습니다.\n④위 1, 2, 3항에도 불구하고 다른 법령 또는 위치정보법에 따라 보유해야하는 정당한 사유가 있는 경우에는 그에 따릅니다."},{"doc":"카카오 위치정보 이용약관","article_no":10,"title":"개인위치정보주체의 권리","text":"①이용자는 언제든지 개인위치정보를 이용한 위치기반서비스의 이용 및 제공에 대한 동의 전부 또는 일부를 유보할 수 있습니다.\n②이용자는 언제든지 개인위치정보를 이용한 위치기반서비스의 이용 및 제공에 대한 동의 전부 또는 일부를 철회할 수 있습니다. 이 경우 회사는 지체 없이 철회된 범위의 개인위치정보 및 위치정보 이용·제공사실 확인자료를 파기합니다.\n③이용자는 개인위치정보의 이용·제공의 일시적인 중지를 요구할 수 있습니다. 이 경우 회사는  이를 거절할 수 없으며 이를 충족하는 기술적 수단을 마련합니다\n④이용자는 회사에 대하여 아래 자료에 대한 열람 또는 고지를 요구할 수 있으며, 해당 자료에 오류가 있는 경우에는 정정을 요구할 수 있습니다. 이 경우 회사는 정당한 사유 없이 요구를 거절하지 않습니다.\n1.이용자에 대한 위치정보 이용·제공사실 확인자료\n2.이용자의 개인위치정보가 위치정보의 보호 및 이용 등에 관한 법률 또는 다른 법령의 규정에 의하여 제3자에게 제공된 이유 및 내용\n⑤이용자는 권리행사를 위해 본 약관 제14조의 연락처를 이용하여 회사에 요청할 수 있습니다."},{"doc":"카카오 위치정보 이용약관","article_no":11,"title":"법정대리인의 권리","text":"①회사는 14세 미만의 이용자에 대해서는 개인위치정보를 이용한 위치기반서비스 제공 및 개인위치정보의 제3자 제공에 대한 동의를 이용자 및 이용자의 법정대리인으로부터 받아야 합니다. 이 경우 법정대리인은 본 약관 제10조에 의한 이용자의 권리를 모두 가집니다.\n②회사는 14세 미만의 아동의 개인위치정보 또는 위치정보 이용, 제공사실 확인자료를 이용약관에 명시 또는 고지한 범위를 넘어 이용하거나 제3자에게 제공하고자 하는 경우 이용자와 이용자의 법정대리인의 동의를 받아야 합니다. 단, 아래의 경우는 제외합니다.\n1.위치정보 및 위치기반서비스 제공에 따른 요금정산을 위하여 위치정보 이용, 제공사실 확인자료가 필요한 경우\n2.통계작성, 학술연구 또는 시장조사를 위하여 특정 개인을 알아볼 수 없는 형태로 가공하여 제공하는 경우"},{"doc":"카카오 위치정보 이용약관","article_no":12,"title":"8세 이하의 아동 등의 보호의무자의 권리","text":"①회사는 아래의 경우에 해당하는 자(이하 “8세 이하의 아동 등”)의 위치정보의 보호 및 이용 등에 관한 법률 제26조2항에 해당하는 자(이하 “보호의무자”)가 8세 이하의 아동 등의 생명 또는 신체보호를 위하여 개인위치정보의 이용 또는 제공에 동의하는 경우에는 본인의 동의가 있는 것으로 봅니다.\n1.8세 이하의 아동\n2.피성년후견인\n3.장애인복지법 제2조제2항제2호에 따른 정신적 장애를 가진 사람으로서 장애인고용촉진 및 직업재활법 제2조제2호에 따른 중증장애인에 해당하는 사람(장애인복지법 제32조에 따라 장애인 등록을 한 사람만 해당한다)\n②8세 이하의 아동 등의 생명 또는 신체의 보호를 위하여 개인위치정보의 이용 또는 제공에 동의를 하고자 하는 보호의무자는 서면동의서에 보호의무자임을 증명하는 서면을 첨부하여 회사에 제출하여야 합니다.\n③보호의무자는 8세 이하의 아동 등의 개인위치정보 이용 또는 제공에 동의하는 경우 본 약관 제9조에 의한 이용자의 권리를 모두 가집니다."},{"doc":"카카오 위치정보 이용약관","article_no":13,"title":"손해배상","text":"회사의 위치정보의 보호 및 이용 등에 관한 법률 제15조 및 26조의 규정을 위반한 행위로 인해 손해를 입은 경우 이용자는 회사에 손해배상을 청구할 수 있습니다. 회사는 고의, 과실이 없음을 입증하지 못하는 경우 책임을 면할 수 없습니다."},{"doc":"카카오 위치정보 이용약관","article_no":14,"title":"면책","text":"①회사는 다음 각 호의 경우로 위치기반서비스를 제공할 수 없는 경우 이로 인하여 이용자에게 발생한 손해에 대해서는 회사의 고의 과실이 없는 한 책임을 부담하지 않습니다.\n1.천재지변 또는 이에 준하는 불가항력의 상태가 있는 경우\n2.위치기반서비스 제공을 위하여 회사와 서비스 제휴계약을 체결한 제3자의 고의적인 서비스 방해가 있는 경우\n3.이용자의 귀책사유로 위치기반서비스 이용에 장애가 있는 경우\n4.제1호 내지 제3호를 제외한 기타 회사의 고의·과실이 없는 사유로 인한 경우\n②회사는 위치기반서비스 및 위치기반서비스에 게재된 정보, 자료, 사실의 신뢰도, 정확성 등에 대해서는 보증을 하지 않으며 이로 인해 발생한 이용자의 손해에 대하여는 회사의 고의 과실이 없는 한 책임을 부담하지 아니합니다."},{"doc":"카카오 위치정보 이용약관","article_no":15,"title":"분쟁의 조정 및 기타","text":"①회사는 위치정보와 관련된 분쟁의 해결을 위해 이용자와 성실히 협의합니다.\n②전항의 협의에서 분쟁이 해결되지 않은 경우, 회사와 이용자는 위치정보의 보호 및 이용 등에 관한 법률 제28조의 규정에 의해 방송통신위원회에 재정을 신청하거나, 개인정보보호법 제43조의 규정에 의해 개인정보 분쟁조정위원회에 조정을 신청할 수 있습니다."},{"doc":"카카오 위치정보 이용약관","article_no":16,"title":"사업자 및 위치정보관리책임자 정보","text":"① 회사의 상호, 주소 및 연락처는 아래와 같습니다.\n상호 : 주식회사 카카오\n주소 : 제주특별자치도 제주시 첨단로 242 (영평동)\n대표전화 : 1577-3754 (유료)\n② 회사는 개인위치정보를 적절히 관리·보호하고, 이용자의 불만을 원활히 처리할 수 있도록 실질적인 책임을 질 수 있는 지위에 있는 자를 위치정보관리책임자로 지정해 운영하고 있습니다. 위치정보관리책임자는 위치기반서비스를 제공하거나 관리하는 부서의 부서장으로서 성명과 연락처는 아래와 같습니다.\n성명 : 김연지\n대표전화 : 1577-3754 (유료)\n위치정보 전용문의 게시판 (바로가기)"},{"doc":"카카오 통합서비스약관","article_no":1,"title":"목적 및 정의","text":"주식회사 카카오(이하 ‘회사’)가 제공하는 서비스를 이용해 주셔서 감사합니다. 회사는 여러분이 회사가 제공하는 다양한 인터넷과 모바일 서비스(이하 해당 서비스들을 모두 합하여 “통합서비스” 또는 “서비스”라 함)에 더 가깝고 편리하게 다가갈 수 있도록 ‘카카오 통합서비스약관’(이하 ‘본 약관’)을 마련하였습니다. 여러분은 본 약관에 동의함으로써 통합서비스에 가입하여 통합서비스를 이용할 수 있습니다. 단, 여러분은 회사가 아닌 계열사를 포함한 제3자가 제공하는 서비스 (예: ㈜카카오모빌리티가 제공하는 카카오 T 택시 서비스)에 가입되지는 않으며, 회사가 제공하는 유료서비스의 경우 여러분이 별도의 유료이용약관에 대한 동의한 때에 회사와 여러분 간의 유료서비스 이용계약이 성립합니다. 본 약관은 여러분이 통합서비스를 이용하는 데 필요한 권리, 의무 및 책임사항, 이용조건 및 절차 등 기본적인 사항을 규정하고 있으므로 조금만 시간을 내서 주의 깊게 읽어주시기 바랍니다.\n통합서비스: 회사가 제공하는 1) “카카오” 브랜드를 사용하는 서비스(예:카카오톡) 또는 2) 카카오계정으로 이용하는 서비스(예: 브런치) (단, 서비스 명칭에 ‘카카오’가 사용되더라도 회사가 아닌 카카오 계열사에서 제공하는 서비스 (예: 카카오 T택시 서비스)는 본 약관의 통합서비스에 포함되지 않습니다)\n개별 서비스: 통합서비스를 구성하는 세부 하위 서비스를 의미하며, 예를 들어 각 통합서비스 내의 유료서비스, 카카오톡 서비스 등을 의미함"},{"doc":"카카오 통합서비스약관","article_no":2,"title":"약관의 효력 및 변경","text":"①본 약관의 내용은 통합서비스의 화면에 게시하거나 기타의 방법으로 공지하고, 본 약관에 동의한 여러분 모두에게 그 효력이 발생합니다.\n②회사는 필요한 경우 관련 법령을 위배하지 않는 범위 내에서 본 약관을 변경할 수 있습니다. 본 약관이 변경되는 경우 회사는 변경사항을 시행일자 15일 전부터 여러분에게 서비스 공지사항에서 공지 또는 통지하는 것을 원칙으로 하며, 피치 못하게 여러분에게 불리한 내용으로 변경할 경우에는 그 시행일자 30일 전부터 카카오계정에 등록된 이메일 주소로 이메일(이메일주소가 없는 경우 서비스 내 전자쪽지 발송, 서비스 내 알림 메시지를 띄우는 등의 별도의 전자적 수단) 발송 또는 여러분이 등록한 휴대폰번호로 카카오톡 메시지 또는 문자메시지 발송하는 방법 등으로 개별적으로 알려 드리겠습니다.\n③회사가 전 항에 따라 공지 또는 통지를 하면서 공지 또는 통지일로부터 개정약관 시행일 7일 후까지 거부의사를 표시하지 아니하면 승인한 것으로 본다는 뜻을 명확하게 고지하였음에도 여러분의 의사표시가 없는 경우에는 변경된 약관을 승인한 것으로 봅니다.\n④여러분은 변경된 약관에 대하여 거부의사를 표시함으로써 이용계약의 해지를 선택할 수 있습니다.\n⑤본 약관은 여러분이 본 약관에 동의한 날로부터 본 약관 제13조에 따른 이용계약의 해지 시까지 적용하는 것을 원칙으로 합니다. 단, 본 약관의 일부 조항은 이용계약의 해지 후에도 유효하게 적용될 수 있습니다"},{"doc":"카카오 통합서비스약관","article_no":3,"title":"약관 외 준칙","text":"본 약관에 규정되지 않은 사항에 대해서는 관련 법령 또는 통합서비스를 구성하는 개별 서비스의 이용약관, 운영정책 및 규칙, 카카오 운영정책 및 규칙 등(이하 총칭하여 '세부 지침')의 규정에 따릅니다. 세부지침은 본 약관과 더불어 이용계약의 일부를 구성합니다."},{"doc":"카카오 통합서비스약관","article_no":4,"title":"계약의 성립","text":"①통합서비스에 가입하기 위해서는 카카오계정이 필요합니다. 카카오계정이 없으신 경우 카카오계정을 먼저 생성하시기 바랍니다.\n②통합서비스 이용계약은 여러분이 본 약관의 내용에 동의한 후 회사가 여러분의 카카오계정 정보 등을 확인한 후 승낙함으로써 체결됩니다."},{"doc":"카카오 통합서비스약관","article_no":5,"title":"통합서비스 가입의 제한","text":"①제4조에 따른 가입 신청자에게 회사는 원칙적으로 통합서비스 가입을 승낙합니다. 다만, 회사는 아래 각 호의 경우에는 그 사유가 해소될 때까지 승낙을 유보하거나 승낙하지 않을 수 있습니다. 특히, 여러분이 만 14세 미만인 경우에는 부모님 등 법정대리인의 동의가 있는 경우에만 통합서비스에 가입할 수 있습니다.\n1.여러분이 다른 사람의 명의나 이메일 주소 등 개인정보를 이용하여 통합서비스에 가입하려고 한 경우\n2.통합서비스 제공 설비 용량에 현실적인 여유가 없는 경우\n3.통합서비스 제공을 위한 기술적인 부분에 문제가 있다고 판단되는 경우\n4.기타 회사가 재정적, 기술적으로 필요하다고 인정하는 경우\n5.회사로부터 통합서비스 이용정지 조치 등을 받은 자가 그 조치기간에 통합서비스 이용계약을 임의로 해지하고 재가입을 신청하는 경우\n6.기타 관련 법령에 위배되거나 세부지침 등 회사가 정한 기준에 반하는 경우\n②만약, 여러분이 위 조건에 위반하여 통합서비스에 가입한 것으로 판명된 때에는 회사는 즉시 여러분의 통합서비스 이용을 정지시키거나 카카오계정을 삭제하는 등 적절한 제한을 할 수 있습니다."},{"doc":"카카오 통합서비스약관","article_no":6,"title":"다양한 서비스의 제공","text":"①통합서비스 이용계약이 성립되면, 여러분은 통합서비스를 구성하는 개별 서비스를 여러분이 원하는 때에 자유롭게 이용할 수 있습니다.\n②다만, 통합서비스 내에서도 일부 개별 서비스의 경우 별도의 이용약관에 동의해야 이용이 가능하며 필요한 추가 정보를 기재하거나, 이메일 주소 승인 또는 문자메시지 인증, 인증서 발급 등 회사가 정한 인증 절차를 완료하여야 서비스 이용이 가능합니다.\n③여러분은 통합서비스 가입 후에도 언제든지 통합서비스를 구성하는 개별 서비스  화면 또는 메뉴에서 제공하는 기능을 이용하여 해당 개별 서비스 의 이용을 종료할 수 있으며, 이 경우 관련 법령에서 정하는 바에 따라 일정기간 보관해야 하는 정보를 제외하고는 해당 서비스 이용기록, 여러분이 작성한 게시물 등 모든 데이터는 즉시 삭제 처리됩니다. 다만, 여러분이 작성한 게시물이 제3자에 의하여 스크랩 또는 다른 공유 기능으로 게시되거나, 여러분이 제3자의 게시물에 댓글 등 게시물을 추가하는 등의 경우에는 해당 게시물 및 댓글은 삭제되지 않으므로 반드시 이용 종료 전에 삭제하시기 바라며, 일부 서비스의 특성 및 콘텐츠의 성질 등에 따라 게시물의 삭제가 불가능할 수도 있으니 이 점 유의하여 주시기 바랍니다. 개별 서비스 이용 종료 시점에 향후 일정기간 해당 서비스의 재이용에 제한이 있을 수 있다는 별도 안내가 있는 경우 해당 안내에 따라 해당 서비스의 재이용에 일정한 시간적 제한이 있을 수 있는 점 또한 유의하여 주시기 바랍니다.\n④전항에 따른 개별 서비스의 이용 종료는 해당 개별 서비스의 이용 종료만을 의미하며, 통합서비스를 구성하는 다른 서비스의 이용이 종료되지는 않습니다. 여러분이 통합서비스 전체의 이용을 종료하고 싶은 경우에는 본 약관 제13조에서 정한 바처럼 통합서비스 이용계약을 해지하여야 합니다.\n⑤회사는 여러분에게 SNS, 게시판 서비스, 온라인 콘텐츠 제공 서비스, 위치기반서비스 등 여러분이 인터넷과 모바일로 즐길 수 있는 다양한 서비스를 제공합니다. 여러분은 스마트폰의 어플리케이션 스토어 등에서 서비스를 다운받아 설치하거나 직접 PC에 설치 혹은 웹페이지에 접속하여 서비스를 이용할 수 있습니다. 그런데 회사는 여러분이 원하는 다양한 서비스를 시시각각 제공하기 때문에 서비스의 자세한 내용은 별도로 알려드릴 수밖에 없습니다. 이러한 회사의 사정을 이해하여 주시길 바라며, 회사도 개별적인 서비스 이용방법을 어플리케이션 스토어와 각 서비스의 Q&A 센터, 해당 안내 및 고지사항에서 더 상세하게 안내하고 있으니 언제든지 확인하여 주시기 바랍니다.\n⑥회사는 여러분이 통합서비스를 마음껏 이용할 수 있도록 이에 필요한 소프트웨어의 개인적이고 전 세계적이며 양도불가능하고 비독점적인 무상의 라이선스를 여러분에게 제공합니다. 단, 회사가 여러분에게 회사의 상표 및 로고를 사용할 권리를 부여하는 것은 아니라는 점은 잊지 말아주시기 바랍니다.\n⑦회사가 여러분에게 제공하는 통합서비스에는 인공지능에 기반하여 운용되는 서비스가 포함될 수 있으며, 회사가 인공지능에 의하여 생성된 결과물을 제공할 경우에는 관련 법에 따라 고지 및 표시합니다.\n⑧회사는 더 나은 통합서비스를 위하여 통합서비스에 필요한 소프트웨어의 업데이트 버전을 제공할 수 있습니다. 소프트웨어의 업데이트에는 중요한 기능의 추가 또는 불필요한 기능의 제거 등이 포함되어 있습니다. 여러분들도 통합서비스를 즐겁게 이용할 수 있도록 꾸준히 업데이트를 하여 주시기 바랍니다.\n⑨회사는 스팸성 메일(피싱, 바이러스 유포, 개인정보 탈취 등 각종 불법 및 사행성 스팸을 의미합니다)로부터 이용자를 보호하기 위해 수발신 메일에 대한 스팸 대응 및 보안 조치를 합니다. 더불어 유관기관의 권고가 있거나 이용자 보호를 위하여 필요하다고 판단하는 경우 이에 따른 추가적인 스팸 운영정책 및 기능을 제공합니다.\n⑩회사는 더 나은 통합서비스의 제공을 위하여 여러분에게 통합서비스의 이용과 관련된 각종 고지, 관리 메시지 및 기타 광고를 비롯한 다양한 정보를 통합서비스 내에 표시하거나 여러분의 카카오계정 정보에 등록되어 있는 연락처로 직접 발송할 수 있습니다. 단, 광고성 정보 전송의 경우에는 사전에 수신에 동의한 경우에만 전송합니다.\n⑪통합서비스 이용 중 시스템 오류 등 문제점을 발견하신다면 언제든지 고객센터로 알려주시기 바랍니다.\n⑫여러분이 통합서비스를 이용하는 과정에서 Wi-Fi 무선인터넷을 사용하지 않고, 가입하신 이동통신사의 무선인터넷에 연결하여 이용하는 경우 이동통신사로부터 여러분에게 별도의 데이터 통신요금이 부과될 수 있는 점을 유의하여 주시기 바랍니다. 통합서비스 이용 과정에서 발생하는 데이터 통신요금은 여러분이 여러분의 비용과 책임 하에 이동통신사에 납부하셔야 합니다. 데이터 통신요금에 대한 자세한 안내는 여러분이 가입하신 이동통신사에 문의하시기 바랍니다."},{"doc":"카카오 통합서비스약관","article_no":7,"title":"통합서비스의 변경 및 종료","text":"①회사는 통합서비스를 365일, 24시간 쉬지 않고 제공하기 위하여 최선의 노력을 다합니다. 다만, 아래 각 호의 경우 통합서비스의 전부 또는 일부를 제한하거나 중지할 수 있습니다.\n1.통합서비스용 설비의 유지·보수 등을 위한 정기 또는 임시 점검의 경우\n2.정전, 제반 설비의 장애 또는 이용량의 폭주 등으로 정상적인 통합서비스 이용에 지장이 있는 경우\n3.관계사와의 계약 종료, 정부의 명령/규제, 서비스/회원 정책 변경 등 회사의 제반 사정으로 통합서비스의 전부 또는 일부를 유지할 수 없는 경우\n4.기타 천재지변, 국가비상사태 등 불가항력적 사유가 있는 경우\n②전항에 의한 통합서비스 중단의 경우에는 미리 제17조에서 정한 방법으로 여러분에게 통지 내지 공지하겠습니다. 이 때 원만한 서비스 및 정책 변경 등을 위하여 서비스 이용 시 재로그인 또는 추가적인 동의 절차 등이 필요할 수 있습니다. 다만, 회사로서도 예측할 수 없거나 통제할 수 없는 사유(회사의 과실이 없는 디스크 내지 서버 장애, 시스템 다운 등)로 인해 사전 통지 내지 공지가 불가능한 경우에는 그러하지 아니합니다. 이러한 경우에도 회사는 상황을 파악하는 즉시 최대한 빠른 시일 내에 서비스를 복구하도록 노력하고, 2시간 이상 복구가 지연되는 경우 서비스 공지사항, 카카오 고객센터 공지사항 등에 게시하여 알려 드리겠습니다."},{"doc":"카카오 통합서비스약관","article_no":8,"title":"게시물의 관리","text":"①여러분의 게시물이 정보통신망 이용촉진 및 정보보호 등에 관한 법률(이하 ‘정보통신망법’)및 저작권법 등 관련 법령에 위반되는 내용을 포함하는 경우, 권리자는 회사에 관련 법령이 정한 절차에 따라 해당 게시물의 게시중단 및 삭제 등을 요청할 수 있으며, 회사는 관련 법령에 따라 조치를 취합니다.\n②회사는 권리자의 요청이 없는 경우라도 권리침해가 인정될 만한 사유가 있거나 기타 회사의 정책 및 관련 법령에 위반되는 경우에는 관련 법령에 따라 해당 게시물에 대해 임시조치 등을 취할 수 있습니다.\n③위와 관련된 세부 절차는 정보통신망법 및 저작권법이 규정한 범위 내에서 회사가 정한\n권리침해 신고 절차\n에 따릅니다."},{"doc":"카카오 통합서비스약관","article_no":9,"title":"권리의 귀속 및 저작물의 이용","text":"① 여러분은 사진, 글, 정보, (동)영상, 통합서비스 또는 회사에 대한 의견이나 제안 등 콘텐츠(이하 ‘게시물’)를 통합서비스 내에 게시할 수 있으며, 이러한 게시물에 대한 저작권을 포함한 지적재산권은 당연히 권리자가 계속하여 보유합니다.\n② 여러분은 통합서비스 내에 게시한 게시물에 대한 사용, 저장, 수정, 복제, 공중송신, 전시, 배포 등의 방식으로 이용할 수 있도록 사용을 허락하는 전 세계적인 라이선스를 회사에게 제공하게 됩니다. 본 라이선스에서 여러분이 회사에게 부여하는 권리는 통합서비스를 운영, 개선, 홍보하고 새로운 서비스를 개발하기 위한 범위 내에서 사용되며, 이러한 목적 범위 내에서 회사와 명시적인 업무계약을 체결한 상대방 또는 다른 이용자에 대한 서브라이선스 또한 여기에 포함됩니다. 또한, 통합서비스의 개선 및 연구개발 목적으로 회사 및 회사의 계열사에서 게시물을 사용할 수 있습니다. 일부 개별 서비스에서는 여러분이 제공한 콘텐츠에 접근하거나 이를 삭제하는 방법을 제공할 수 있습니다(다만 일부 서비스의 특성 및 콘텐츠의 성질 등에 따라 게시물의 삭제가 불가능할 수도 있습니다). 또한 일부 서비스에서는 제공된 콘텐츠에 대한 회사의 사용 범위를 제한하는 설정이 있습니다.\n③여러분은 회사에 제공한 콘텐츠에 대해 회사에 라이선스를 부여하기 위해 필요한 권리를 보유해야 합니다. 이러한 권리를 보유하지 않아 발생하는 모든 문제에 대해서는 게시자가 책임을 부담하게 됩니다. 또한, 여러분은 음란하거나 폭력적이거나 기타 공서양속 및 법령에 위반하는 콘텐츠를 공개 또는 게시할 수 없습니다.\n④회사는 여러분의 콘텐츠가 관련 법령에 위반되거나 음란 또는 청소년에게 유해한 게시물, 차별 갈등을 조장하는 게시물, 도배 · 광고 · 홍보 · 스팸성 게시물, 계정을 양도 또는 거래하는 내용의 게시물, 타인을 사칭하는 게시물 등이라고 판단되는 경우 이를 삭제하거나 게시를 거부할 수 있습니다. 다만 회사가 모든 콘텐츠를 검토할 의무가 있는 것은 아닙니다. 누군가 여러분의 권리를 침해하였다면, 고객센터를 통해 게시중단 요청에 대한 도움을 받으실 수 있습니다. 위와 관련된 구체적인 기준 및 이용제한 절차의 내용은\n카카오 운영정책\n에서 확인하실 수 있습니다.\n⑤통합서비스에서는 회사가 보유하지 않은 일부 콘텐츠가 표시될 수 있습니다. 그러한 콘텐츠에 대해서는 콘텐츠를 제공한 주체가 단독으로 모든 책임을 부담하게 됩니다. 여러분이 통합서비스를 이용하더라도 다른 이용자의 콘텐츠에 대하여 어떠한 권리를 가지게 되는 것은 아닙니다. 여러분이 다른 이용자의 콘텐츠를 사용하기 위해서는 콘텐츠 소유자로부터 별도로 허락을 받아야 합니다."},{"doc":"카카오 통합서비스약관","article_no":10,"title":"유료 서비스의 이용","text":"①통합서비스를 구성하는 개별 서비스의 대부분은 무료로 제공하고 있으나, 일부 개별 서비스는 유료로 제공될 수 있습니다. 예를 들면, 카카오톡에서 친구들과 무료로 메시지를 주고 받을 수 있으나 일부 이모티콘 등은 유료로 구매해야 친구들에게 보낼 수 있습니다.\n②여러분이 회사가 제공하는 유료서비스를 이용하는 경우 이용대금을 납부한 후 이용하는 것을 원칙으로 합니다. 회사가 제공하는 유료서비스에 대한 이용요금의 결제 방법은 핸드폰결제, 신용카드결제, 일반전화결제, 계좌이체, 무통장입금, 선불전자지급수단 결제 등이 있으며 각 유료서비스마다 결제 방법의 차이가 있을 수 있습니다. 매월 정기적인 결제가 이루어지는 서비스의 경우 여러분 개인이 해당 서비스의 이용을 중단하고 정기 결제의 취소를 요청하지 않는 한 매월 결제가 이루어집니다.\n③회사는 결제의 이행을 위하여 반드시 필요한 여러분의 개인정보를 추가적으로 요구할 수 있으며, 여러분은 회사가 요구하는 개인정보를 정확하게 제공하여야 합니다.\n④본 조에서 정하지 않은 내용은 개별 서비스에 적용되는 유료서비스 약관(예: 카카오 유료서비스 이용약관 등)에서 정하며, 본 조의 내용과 개별 서비스에 적용되는 유료서비스 약관의 내용이 충돌하는 경우 개별 서비스에 적용되는 유료서비스 약관의 규정에 따릅니다."},{"doc":"카카오 통합서비스약관","article_no":11,"title":"게시판 이용 상거래","text":"①여러분이 서비스를 이용하여 통신판매 또는 통신판매중개를 업으로 하는 경우 전자상거래 등에서의 소비자보호에 관한 법률(이하 ‘전자상거래법’)에 따른 의무사항을 준수하여야 합니다.\n②여러분이 통신판매 또는 통신판매중개를 함에 있어 다른 이용자와 전자상거래 관련 분쟁이 발생하는 경우, 회사는 다른 이용자에게 소비자피해 구제 대행 신청을 할 수 있는 장치를 마련합니다.\n③회사는 전자상거래법에 따라 신원정보를 입력하는 기능 등을 제공하여 여러분의 신원정보를 확인하고, 여러분과 다른 이용자 사이에 분쟁이 발생하여 전자상거래법에 따라 소비자피해 분쟁조정기구, 공정거래위원회, 시·도지사 또는 시장·군수·구청장이 신원정보 제공을 요구하는 경우 이에 협조합니다."},{"doc":"카카오 통합서비스약관","article_no":12,"title":"통합서비스 이용 방법 및 주의점","text":"①여러분은 통합서비스를 자유롭게 이용할 수 있으나, 아래 각 호의 행위는 하여서는 안 됩니다.\n1.이용 신청 또는 변경 시 허위 사실을 기재하거나, 다른 사람의 카카오계정 및 비밀번호를 도용, 부정하게 사용하거나, 다른 사람의 명의를 사용하거나 명의자의 허락 없이 문자메시지(SMS) 인증 등을 수행하는 행위\n2.회사의 서비스 정보를 이용하여 얻은 정보를 회사의 사전 승낙 없이 복제 또는 유통시키거나 상업적으로 이용하는 행위\n3.서비스 내에서 다운로드 또는 스트리밍을 통해 제공받은 음원을 사적 목적으로 이용하는 것 외에, 공공장소 및 영리를 목적으로 하는 영업장, 매장 등에서 재생하는 등의 방법으로 이용하는 행위\n4.타인의 명예를 손상시키거나 불이익을 주는 행위\n5.게시판 등에 음란물을 게재하거나 음란사이트를 연결(링크)하는 행위\n6.회사 또는 제3자의 저작권 등 기타 권리를 침해하는 행위(국내외 제3자의 저작권 등을 침해하는 행위로서 회사가 IP 접속 차단 등 기술적인 조치를 통하여 타인에 대한 권리 침해 방지 조치를 취하였음에도 불구하고 이용자가 회사를 기망하는 수단과 방법 등을 통하여 서비스에 접속 하는 등 제3자의 저작권 등을 침해하는 행위를 포함합니다)\n7.서비스 내에 회사나 제3자 등에 대한 허위의 사실을 게시하는 행위\n8.공공질서 및 미풍양속에 위반되는 내용의 정보, 문장, 도형, 음성 등을 타인에게 유포하는 행위\n9.통합서비스와 관련된 설비의 오동작이나 정보 등의 파괴 및 혼란을 유발시키는 컴퓨터 바이러스 감염 자료를 등록 또는 유포하는 행위\n10.통합서비스의 운영을 방해하거나 안정적 운영을 방해할 수 있는 정보 및 수신자의 명시적인 수신거부의사에 반하여 또는 수신자의 명시적인 동의 없이 광고성 정보 또는 스팸메일(Spam Mail)을 전송하는 행위\n11.회사의 동의 없이 서비스 또는 이에 포함된 소프트웨어의 일부를 복사, 수정, 배포, 판매, 양도, 대여, 담보제공하거나 타인에게 그 이용을 허락하는 행위와 소프트웨어를 역설계하거나 소스 코드의 추출을 시도하는 등 서비스를 복제, 분해 또는 모방하거나 기타 변형하는 행위\n12.타인으로 가장하는 행위 및 타인과의 관계를 허위로 명시하는 행위\n13.다른 이용자의 개인정보를 수집, 저장, 공개하는 행위\n14.자기 또는 타인에게 재산상의 이익을 주거나 타인에게 손해를 가하는 등 피해를 입힐 목적으로 허위의 정보를 유통시키는 행위\n15.재물을 걸고 도박하거나 사행행위를 하는 행위\n16.윤락행위를 알선하거나 음행을 매개하는 내용의 정보를 유통시키는 행위\n17.수치심이나 혐오감 또는 공포심을 일으키는 말이나 음향, 글이나 화상 또는 영상을 계속하여 상대방에게 도달하게 하여 상대방의 일상적 생활을 방해하는 행위\n18.관련 법령에 의하여 그 전송 또는 게시가 금지되는 정보(컴퓨터 프로그램 포함)의 전송 또는 게시 행위\n19.회사 또는 관계사의 직원이나 운영자를 가장하거나 사칭하여 또는 타인의 명의를 도용하여 글을 게시하거나 E-mail, 카카오톡 메시지 등을 발송하는 행위\n20.컴퓨터 소프트웨어, 하드웨어, 전기통신 장비의 정상적인 가동을 방해, 파괴할 가능성이 있는 소프트웨어 바이러스, 기타 다른 컴퓨터 코드, 파일, 프로그램을 포함하고 있는 자료를 게시하거나 E-mail, 카카오톡 메시지 등으로 발송하는 행위\n21.스토킹(stalking), 허위 또는 악의적 신고 남용 등 다른 이용자를 괴롭히는 행위\n22.1개월 이내 통합서비스 가입 및 유료서비스 구매 후 다시 해지하는 행위를 2회 이상 반복하는 등 통합서비스를 부당하게 악용하는 행위\n23.기타 현행 법령, 본 약관 및 운영정책 등 회사가 제공하는 서비스 관련 세부지침을 위반하는 행위\n②여러분은 서비스의 이용 권한, 기타 이용계약상 지위를 타인에게 양도·증여할 수 없으며, 담보로 제공할 수 없습니다.\n③여러분의 자격 혹은 나이에 따라 아래 각 호처럼 통합서비스 이용의 일부가 제한될 수 있습니다.\n1.만19세 미만의 이용자는(단, 만 19세에 도달하는 해의 1월 1일을 맞이한 자는 제외, 이하 본 조에서 동일함) 정보통신망법 및 청소년보호법의 규정에 의하여 청소년유해매체물은 이용할 수 없습니다.\n2.청소년유해매체물을 이용하시기 위해서는 만 19세 이상이어야 하며, 정보통신망법 및 청소년보호법의 규정에 의하여 실명인증을 통해 본인 및 연령 인증을 받으셔야 합니다. 인증을 받지 않으시면, 해당 서비스의 이용이 제한됩니다.\n3.만 19세 미만의 이용자의 서비스에 대하여 법정대리인의 요청 및 만19세 미만 이용자 본인의 동의가 있는 경우 개별 서비스의 전체 또는 일부의 이용이 일정기간 제한됩니다.\n④회사는 음성대화 기능 등을 제공하는 일부 서비스 내에서 이용자간 신고가 있는 경우, 신고된 이용자의 음성정보를 저장 및 보관할 수 있으며 이 정보는 회사만 보유합니다. 회사는 이용자간 분쟁 조정, 민원 처리를 위한 목적에 한하여, 제 3 자는 법령에 따라 권한이 부여된 경우에 한하여 이 정보를 열람할 수 있습니다. 회사는 부정이용 방지 및 관리의 목적에 따라 신고 접수시부터 3년간 해당 정보를 3년간 보관 후 파기합니다. 단, 저 사양 단말기의 경우에는 신고된 음성정보가 저장되지 않을 수 있습니다.\n⑤회사는 수사기관(경찰청 등)이 피싱 범죄에 이용중인 사실을 확인하여 법령 등에 따라 정당한 절차로 긴급차단을 요청하는 경우 여러분의 개별 서비스의 일부 또는 전부의 이용을 잠시 또는 계속하여 중단하는 이용 제한을 할 수 있습니다. 여러분이 이러한 이용 제한과 관련하여 이의가 있는 경우 이용정지를 요청한 수사기관에 이의제기를 할 수 있습니다. 수사기관에서 정당한 사유에 대한 소명이 확인된 경우 회사는 이용 제한을 해제할 수 있습니다.\n⑥혹시라도 여러분이 관련 법령, 회사의 모든 약관 또는 정책을 준수하지 않는다면, 회사는 여러분의 위반행위 등을 조사할 수 있고, 해당 게시물 등을 삭제 또는 임시 삭제하거나 여러분의 계정・통합서비스 전체 또는 통합서비스를 구성하는 일부 개별 서비스의 이용을 잠시 또는 계속하여 중단하거나, 통합서비스 재가입 또는 일부 개별 서비스의 재이용에 제한을 둘 수도 있습니다. 또한 여러분이 통합서비스와 관련된 설비의 오작동이나 시스템의 파괴 및 혼란을 유발하는 등 통합서비스 제공에 악영향을 미치거나 안정적 운영을 심각하게 방해한 경우, 회사는 이러한 위험 활동이 확인된 여러분의 계정들에 대하여 이용제한을 할 수 있습니다. 다만, 여러분은 이용제한과 관련하여 조치 결과가 불만족스러울 경우 고객센터를 통해 이의를 제기할 수 있습니다.\n⑦이용 제한은 위반 활동의 누적 정도에 따라 한시적 제한에서 영구적 제한으로 단계적 제한하는 것을 원칙으로 하지만, 음란한 내용의 게시와 유포 및 사행성 도박 홍보 등 관련 법령에서 금지하는 명백한 불법행위나 타인의 권리침해로서 긴급한 위험 또는 피해 차단이 요구되는 사안에 대해서는 위반 활동 횟수의 누적 정도와 관계 없이 즉시 영구적으로 이용이 제한될 수 있습니다.\n⑧본 조에서 정한 사항 및 그 밖에 통합서비스의 이용에 관한 자세한 사항은\n카카오 운영정책\n등을 참고해 주시기 바랍니다."},{"doc":"카카오 통합서비스약관","article_no":13,"title":"이용계약 해지","text":"①여러분이 카카오계정 탈퇴를 하는 경우 통합서비스 이용계약도 자동으로 해지됩니다.\n②통합서비스 이용계약 해지를 원하는 경우 여러분은 언제든지 서비스 내 제공되는 메뉴를 이용하여 해지 신청을 할 수 있으며,회사는 법령이 정하는 바에 따라 신속히 처리하겠습니다.\n③통합서비스 이용계약이 해지되면 관련 법령 및 카카오 개인정보 처리방침에 따라 여러분의 일정 정보를 보유하는 경우를 제외하고는 여러분의 정보나 여러분이 작성한 게시물 등 모든 데이터는 삭제됩니다. 다만, 여러분이 작성한 게시물이 제3자에 의하여 스크랩 또는 다른 공유 기능으로 게시되거나, 여러분이 제3자의 게시물에 댓글 등 게시물을 추가하는 등의 경우에는 해당 게시물 및 댓글은 삭제되지 않으므로 반드시 해지 신청 전에 삭제하시기 바랍니다.\n④전항에 따라 여러분이 삭제하지 않은 게시물은 다른 이용자의 정상적 서비스 이용을 위하여 필요한 범위 내에서 통합서비스 내에 삭제되지 않고 남아 있게 됩니다.\n⑤유료서비스 이용계약의 해지는 여러분의 유료서비스 이용계약 해지 신청 및 회사의 승낙에 의해 성립하게 되고, 환불할 금액이 있는 경우 환불도 이루어 지게 됩니다. 다만 각 개별 서비스의 유료서비스에서 본 약관과 다른 계약해지 방법 및 효과를 규정하고 있는 경우 각 유료서비스 약관 및 관련 세부지침에서 정한 바에 따릅니다.\n⑥통합서비스를 구성하는 일부 개별 서비스의 경우 일정기간 동안 해당 개별 서비스를 이용하지 않을 경우 여러분의 정보를 파기하거나 분리 보관할 수 있으며, 또는 해당 개별 서비스 기능의 일부 또는 전부를 이용할 수 없도록 제한할 수 있습니다. 자세한 사항은 개별 서비스의 세부지침에서 확인하실 수 있습니다.\n⑦통합서비스 이용계약이 해지된 경우라도 여러분은 다시 회사에 대하여 이용계약의 체결을 신청할 수 있습니다. 다만, 여러분이 관련 법령, 본 약관 및 세부지침을 준수하지 않아 서비스의 이용이 중단된 상태에서 이용계약을 해지한 후 다시 이용계약 체결을 신청하는 경우에는 통합서비스 가입에 일정기간 시간적 제한이 있을 수 있습니다. 또한 통합서비스를 구성하는 일부 개별 서비스의 경우 다시 통합서비스 이용계약을 체결한 후에도 해당 개별 서비스를 바로 이용하는 것에는 제6조 제3항에서 정한 바와 같이 일정한 시간적 제한 등이 따를 수 있습니다."},{"doc":"카카오 통합서비스약관","article_no":14,"title":"개인정보의 보호","text":"여러분의 개인정보의 안전한 처리는 회사에게 있어 가장 중요한 일 중 하나입니다. 여러분의 개인정보는 통합서비스의 원활한 제공을 위하여 여러분이 동의한 목적과 범위 내에서만 이용됩니다. 관련 법령에 의하거나 여러분이 별도로 동의하지 아니하는 한 회사가 여러분의 개인정보를 제3자에게 제공하는 일은 결코 없으므로, 안심하셔도 좋습니다. 회사가 여러분의 개인정보를 안전하게 처리하기 위하여 기울이는 노력이나 기타 자세한 사항은\n카카오 개인정보처리방침\n등을 참고해 주시기 바랍니다."},{"doc":"카카오 통합서비스약관","article_no":15,"title":"손해배상 등","text":"①회사는 관련 법령상 허용되는 한도 내에서 통합서비스와 관련하여 본 약관에 명시되지 않은 어떠한 구체적인 사항에 대한 약정이나 보증을 하지 않습니다. 또한, 회사는 CP(Contents Provider)가 제공하거나 여러분이 작성하는 등의 방법으로 통합서비스에 게재된 정보, 자료, 사실의 신뢰도, 정확성 등에 대해서는 보증을 하지 않으며, 회사의 과실 없이 발생된 여러분의 손해에 대하여는 책임을 부담하지 아니합니다.\n②회사는 회사의 과실로 인하여 여러분이 손해를 입게 될 경우 본 약관 및 관련 법령에 따라 여러분의 손해를 배상하겠습니다. 다만 회사는 회사의 과실 없이 발생된 아래와 같은 손해에 대해서는 책임을 부담하지 않습니다. 또한 회사는 법률상 허용되는 한도 내에서 간접 손해, 특별 손해, 결과적 손해, 징계적 손해, 및 징벌적 손해에 대한 책임을 부담하지 않습니다.\n1.천재지변 또는 이에 준하는 불가항력의 상태에서 발생한 손해\n2.여러분의 귀책사유로 통합서비스 이용에 장애가 발생한 경우\n3.통합서비스에 접속 또는 이용과정에서 발생하는 개인적인 손해\n4.제3자가 불법적으로 회사의 서버에 접속하거나 서버를 이용함으로써 발생하는 손해\n5.제3자가 회사 서버에 대한 전송 또는 회사 서버로부터의 전송을 방해함으로써 발생하는 손해\n6.제3자가 악성 프로그램을 전송 또는 유포함으로써 발생하는 손해\n7.전송된 데이터의 생략, 누락, 파괴 등으로 발생한 손해, 명예훼손 등 제3자가 서비스를 이용하는 과정에서 발생된 손해\n8.기타 회사의 고의 또는 과실이 없는 사유로 인해 발생한 손해\n③회사는 회사의 고의 또는 과실이 없는 한 여러분이 통합서비스를 이용하여 기대하는 수익을 상실한 것에 대하여 책임을 지지 않으며 그 밖에 통합서비스를 통하여 얻은 자료로 인한 손해 등에 대하여도 책임을 지지 않습니다.\n④회사는 회사의 과실이 없는 한 여러분 상호간 또는 여러분과 제3자 상호간에 통합서비스를 매개로 발생한 분쟁에 대해서는 개입할 의무가 없으며 이로 인한 손해를 배상할 책임도 없습니다."},{"doc":"카카오 통합서비스약관","article_no":16,"title":"청소년보호","text":"통합서비스는 기본적으로 모든 연령대가 자유롭게 이용할 수 있는 공간으로서 유해 정보로부터 청소년을 보호하고 청소년의 안전한 인터넷 사용을 돕기 위해 정보통신망법에서 정한 청소년보호정책을 별도로 시행하고 있으며, 구체적인 내용은 통합서비스를 구성하는 개별 서비스 초기 화면 등에서 확인할 수 있습니다."},{"doc":"카카오 통합서비스약관","article_no":17,"title":"통지 및 공지","text":"회사는 여러분과의 의견 교환을 소중하게 생각합니다. 여러분은 언제든지\n카카오 고객센터\n에 방문하여 의견을 개진할 수 있습니다. 서비스 이용자 전체에 대한 공지는 칠(7)일 이상\n서비스 공지사항\n란에 게시함으로써 효력이 발생합니다. 여러분에게 중대한 영향을 미치는 사항의 경우에는 카카오계정에 등록된 이메일 주소로 이메일(이메일주소가 없는 경우 서비스 내 전자쪽지 발송, 서비스 내 알림 메시지를 띄우는 등의 별도의 전자적 수단) 발송 또는 여러분이 등록한 휴대폰번호로 카카오톡 메시지 또는 문자메시지 발송하는 방법 등으로 개별적으로 알려 드리겠습니다."},{"doc":"카카오 통합서비스약관","article_no":18,"title":"분쟁의 해결","text":"본 약관 또는 통합서비스는 대한민국법령에 의하여 규정되고 이행됩니다. 통합서비스 이용과 관련하여 회사와 여러분 간에 분쟁이 발생하면 이의 해결을 위해 성실히 협의할 것입니다. 그럼에도 불구하고 해결되지 않으면 민사소송법의 관할법원에 소를 제기할 수 있습니다."},{"doc":"카카오 통합 약관","article_no":1,"title":"목적","text":"㈜카카오(이하 ‘회사’)가 제공하는 서비스를 이용해 주셔서 감사합니다. 회사는 여러분이 회사가 제공하는 다양한 인터넷과 모바일 서비스에 더 가깝게 다가갈 수 있도록 카카오 서비스 및 Daum 서비스(이하 통칭하여 ‘서비스’)에 통합 적용될 수 있는 카카오 통합 약관(이하 ‘본 약관’)을 마련하였습니다. 본 약관은 여러분이 서비스를 이용하는 데 필요한 권리, 의무 및 책임사항, 이용조건 및 절차 등 기본적인 사항을 규정하고 있으므로 조금만 시간을 내서 주의 깊게 읽어주시기 바랍니다. • '카카오 서비스'라 함은 회사가 제공하는 1) “카카오” 브랜드를 사용하는 서비스(예:카카오톡) 또는 2) 카카오계정으로 이용하는 서비스(예: 브런치)를 의미하며, \"Daum\" 브랜드를 사용하는 서비스는 포함되지 않습니다. • ‘Daum 서비스’라 함은 회사가 제공하는 “Daum” 브랜드를 사용하는 서비스를 말합니다."},{"doc":"카카오 통합 약관","article_no":2,"title":"약관의 명시, 효력 및 변경","text":"1. 본 약관의 내용은 회사가 제공하는 개별 서비스 또는 서비스 초기 화면에 게시하거나 기타의 방법으로 공지하고, 본 약관에 동의한 여러분 모두에게 그 효력이 발생합니다. 2. 회사는 필요한 경우 관련법령을 위배하지 않는 범위 내에서 본 약관을 변경할 수 있습니다. 본 약관이 변경되는 경우 회사는 변경사항을 시행일자 15일 전부터 여러분에게 Daum 공지사항 또는 카카오 서비스 공지사항에서 공지 또는 통지하는 것을 원칙으로 하며, 피치 못하게 여러분에게 불리한 내용으로 변경할 경우에는 그 시행일자 30일 전부터 카카오계정 또는 Daum 아이디로 사용하는 이메일 주소로 이메일을 발송하거나, 여러분이 등록한 휴대폰번호로 카카오톡 메시지 또는 문자메시지를 보내거나, 서비스 내 전자쪽지 발송, 알림 메시지를 띄우는 등 합리적으로 가능한 방법으로 변경사항을 공지 또는 통지하겠습니다. 3. 회사가 전 항에 따라 공지 또는 통지를 하면서 공지 또는 통지일로부터 개정약관 시행일 7일 후까지 거부의사를 표시하지 아니하면 승인한 것으로 본다는 뜻을 명확하게 고지하였음에도 여러분의 의사표시가 없는 경우에는 변경된 약관을 승인한 것으로 봅니다. 여러분이 개정약관에 동의하지 않을 경우 여러분은 제14조 제1항에 따라 이용계약을 해지할 수 있습니다."},{"doc":"카카오 통합 약관","article_no":3,"title":"약관 외 준칙","text":"본 약관에 규정되지 않은 사항에 대해서는 관련법령 또는 회사가 정한 서비스의 개별 이용약관, 운영정책 및 규칙 등(이하 ‘세부지침’)의 규정에 따릅니다. 또한 본 약관과 세부지침의 내용이 충돌할 경우 세부지침에 따릅니다."},{"doc":"카카오 통합 약관","article_no":4,"title":"카카오계정 또는 Daum 아이디 생성 및 연결","text":"1. 카카오계정이란 여러분이 카카오 서비스를 사용하기 위하여 필요한 로그인 계정을 의미합니다. 카카오계정은 여러분이 약관에 동의하고 카카오계정 생성을 위해 필요한 일정 정보를 입력하시면, 카카오가 입력된 일정 정보를 인증한 후 가입을 승낙하는 절차로 생성됩니다. 2. Daum 아이디란 여러분이 Daum 서비스에서 본인을 식별하기 위해 미리 등록한 문자, 특수문자, 숫자 등의 조합으로, 여러분이 Daum 서비스약관 또는 본 약관에 동의하고 회원등록에 필요한 필수사항을 입력한 후 회원등록을 완료하면 회사가 승낙하는 절차로 생성됩니다. 다만, 여러분이 카카오계정으로 Daum 서비스를 이용하는 경우 자동으로 추천된 Daum 아이디가 생성될 수 있습니다. 3. 카카오 서비스를 이용하기 위하여 반드시 카카오계정이 필요한 것은 아니나, 어떤 카카오 서비스는 카카오계정이 반드시 필요합니다. Daum 서비스약관에 동의한 Daum 아이디로는 Daum 서비스만 이용할 수 있습니다. 여러분이 Daum 서비스 중 카카오 서비스와 연결되는 기능을 이용하기 위해서는 카카오계정으로 Daum서비스를 이용하거나, 카카오계정과 Daum 아이디와의 연결이 필요합니다. 카카오계정으로 Daum 서비스를 이용하거나, 카카오계정에 기존에 등록한 Daum 아이디를 연결하면 카카오 서비스와 Daum 서비스에 설정한 정보, 서비스 이용기록 등을 카카오 서비스와 Daum 서비스에서 모두 이용할 수 있습니다. 회사는 서비스 회원 정책의 변경 등의 사유가 발생하였을 때, 회원에게 안내 후 계정 연결 서비스를 종료할 수 있습니다. 4. 여러분이 카카오계정으로 Daum 서비스를 이용하거나, 카카오계정과 Daum 아이디를 연결하면서 본 약관에 동의하면 그 이후부터는 본 약관만을 적용받게 되고, 카카오 서비스 약관 및 Daum 서비스 약관은 더 이상 적용되지 않습니다. 다만, 계정 연결 서비스를 종료할 경우에는 약관 변경에 따라 카카오 통합서비스 약관 또는 카카오 서비스 약관 및 Daum 서비스 약관이 적용될 수 있습니다."},{"doc":"카카오 통합 약관","article_no":5,"title":"카카오계정 또는 Daum 아이디 생성 거절 및 유보","text":"1. 회사는 아래와 같은 경우에는 여러분의 카카오계정 및/또는 Daum 아이디의 생성을 승낙하지 않을 수 있습니다. 특히, 여러분이 14세 미만인 경우에는 부모님 등 법정대리인의 동의가 있는 경우에만 카카오계정 및/또는 Daum 아이디를 생성할 수 있습니다. • 회사가 본 약관에 의해 여러분의 카카오계정 또는 Daum 아이디를 삭제하였던 경우 • 여러분이 다른 사람의 명의나 이메일 주소 등 개인정보를 이용하여 카카오계정 또는 Daum 아이디를 생성하려 한 경우 • 카카오계정 또는 Daum 아이디 생성 시 필요한 정보를 입력하지 않거나 허위의 정보를 입력한 경우 • 기타 관련법령에 위배되거나 세부지침 등 회사가 정한 기준에 반하는 경우 2. 만약, 여러분이 위 조건에 위반하여 카카오계정 및/또는 Daum 아이디를 생성한 것으로 판명된 때에는 회사는 즉시 여러분의 서비스 이용을 중단하거나 카카오계정 및 Daum 아이디를 삭제하는 등 적절한 제한을 할 수 있습니다. 3. 회사는 아래와 같은 경우에는 여러분의 카카오계정 및/또는 Daum 아이디 생성을 유보할 수 있습니다. • 제공 서비스 설비용량에 현실적인 여유가 없는 경우 • 서비스 제공을 위한 기술적인 부분에 문제가 있다고 판단되는 경우 • 기타 회사가 재정적, 기술적으로 필요하다고 인정하는 경우"},{"doc":"카카오 통합 약관","article_no":6,"title":"카카오계정 또는 Daum 아이디 등의 관리","text":"1. 카카오계정 및 Daum 아이디는 여러분 본인만 이용할 수 있으며, 다른 사람이 여러분의 카카오계정 및 Daum 아이디를 이용하도록 허락할 수 없습니다. 그리고 여러분은 다른 사람이 여러분의 카카오계정 및 Daum 아이디를 무단으로 사용할 수 없도록 직접 비밀번호를 관리하여야 합니다. 회사는 다른 사람이 여러분의 카카오계정 및/또는 Daum 아이디를 무단으로 사용하는 것을 막기 위하여 비밀번호 입력 및 추가적인 본인 확인 절차를 거치도록 할 수 있습니다. 만약 무단 사용이 발견된다면, 고객센터를 통하여 회사에게 알려주시기 바라며, 회사는 무단 사용을 막기 위한 방법을 여러분에게 안내하도록 하겠습니다. 2. 여러분은 서비스 내 설정 화면을 통하여 여러분의 정보를 열람하고 수정할 수 있습니다. 다만, 서비스의 제공 및 관리를 위해 필요한 카카오계정, Daum 아이디, 전화번호, 단말기 식별번호, 기타 본인확인정보 등 일부 정보는 수정이 불가능할 수 있으며, 수정하는 경우에는 추가적인 본인 확인 절차가 필요할 수 있습니다. 여러분이 서비스 이용 신청 시 알려주신 내용에 변동이 있을 때, 직접 서비스에서 수정하거나 이메일, 고객센터를 통하여 회사에 알려 주시기 바랍니다. 3. 여러분이 서비스 내 정보를 수정하지 않아 발생하는 손해에 대하여 회사는 책임을 부담하지 아니합니다."},{"doc":"카카오 통합 약관","article_no":7,"title":"다양한 서비스 제공 및 변경 등","text":"1. 회사는 SNS, 게시판 서비스, 온라인 콘텐츠 제공 서비스, 위치기반서비스 등 여러분이 인터넷과 모바일로 즐길 수 있는 다양한 서비스를 제공합니다. 여러분은 스마트폰의 어플리케이션 스토어 등에서 서비스를 다운받아 설치하거나 직접 PC에 설치 혹은 웹페이지에 접속하여 서비스를 이용할 수 있습니다. 그런데 회사는 여러분이 원하는 다양한 서비스를 시시각각 제공하기 때문에 서비스의 자세한 내용은 별도로 알려드릴 수밖에 없습니다. 이러한 회사의 사정을 이해하여 주시길 바라며, 회사도 개별적인 서비스 이용방법을 어플리케이션 스토어와 각 서비스의 Q&A 센터, 해당 안내 및 고지사항에서 더 상세하게 안내하고 있으니 언제든지 확인하여 주시기 바랍니다. 2. 회사는 여러분이 서비스를 마음껏 이용할 수 있도록 이에 필요한 소프트웨어의 개인적이고 전 세계적이며 양도불가능하고 비독점적인 무상의 라이선스를 여러분에게 제공합니다. 단, 회사가 여러분에게 회사의 상표 및 로고를 사용할 권리를 부여하는 것은 아니라는 점은 잊지 말아주시기 바랍니다. 3. 회사는 더 나은 서비스를 위하여 서비스에 필요한 소프트웨어의 업데이트 버전을 제공할 수 있습니다. 소프트웨어의 업데이트에는 중요한 기능의 추가 또는 불필요한 기능의 제거 등이 포함되어 있습니다. 여러분들도 서비스를 즐겁게 이용할 수 있도록 꾸준히 업데이트를 하여 주시기 바랍니다. 4. 회사는 더 나은 서비스의 제공을 위하여 여러분에게 서비스의 이용과 관련된 각종 고지, 관리 메시지 및 기타 광고를 비롯한 다양한 정보를 서비스에 표시하거나 여러분의 메일 계정으로 직접 발송할 수 있습니다. 5. 서비스 이용 중 시스템 오류 등 문제점을 발견하신다면 언제든지 카카오 고객 센터로 알려주시기 바랍니다. 6. 여러분이 서비스를 이용하는 과정에서 Wi-Fi 무선인터넷을 사용하지 않고, 가입하신 이동통신사의 무선인터넷에 연결하여 이용하는 경우 이동통신사로부터 여러분에게 별도의 데이터 통신요금이 부과되는 점을 유의하여 주시기 바랍니다. 서비스 이용 과정에서 발생하는 데이터 통신요금은 여러분이 여러분의 비용과 책임 하에 이동통신사에 납부하셔야 합니다. 데이터 통신요금에 대한 자세한 안내는 여러분이 가입하신 이동통신사에 문의하시기 바랍니다."},{"doc":"카카오 통합 약관","article_no":8,"title":"서비스 이용 방법 및 주의점","text":"1. 여러분은 서비스를 자유롭게 이용할 수 있으나, 아래와 같이 서비스를 잘못된 방법으로 이용할 수 없다는 점을 잊지 말아주셨으면 합니다. • 여러분은 잘못된 방법으로 서비스의 제공을 방해하거나 회사가 안내하는 방법 이외의 다른 방법을 사용하여 서비스에 접근할 수 없습니다. • 다른 서비스 이용자의 정보를 무단으로 수집, 이용하거나 다른 사람들에게 제공하는 행위도, 수신자의 명시적 수신거부 의사에 반하여 또는 수신자의 명시적인 동의 없이 광고성 정보를 전송하거나 서비스를 영리 목적으로 이용하는 것도, 음란 정보나 저작권 침해, 회사나 제3자 등에 대한 허위의 사실을 게시하는 정보 등 공서양속 및 법령에 위반되는 내용의 정보 등을 발송하거나 게시하는 행위도 금지됩니다. • 회사의 동의 없이 서비스 또는 이에 포함된 소프트웨어의 일부를 복사, 수정, 배포, 판매, 양도, 대여, 담보제공하거나 타인에게 그 이용을 허락하는 행위와 소프트웨어를 역설계하거나 소스 코드의 추출을 시도하는 등 서비스를 복제, 분해 또는 모방하거나 기타 변형하는 행위도 금지됩니다. 2. 여러분은 서비스의 이용권한, 기타 이용 계약상 지위를 타인에게 양도·증여할 수 없으며, 담보로 제공할 수 없습니다. 3. 카카오는 음성대화 기능 등을 제공하는 일부 서비스 내에서 이용자간 신고가 있는 경우, 신고된 이용자의 음성정보를 저장 및 보관할 수 있으며 이 정보는 회사만 보유합니다. 카카오는 이용자간 분쟁 조정, 민원 처리를 위한 목적에 한하여, 제 3 자는 법령에 따라 권한이 부여된 경우에 한하여 이 정보를 열람할 수 있습니다. 카카오는 부정이용 방지 및 관리의 목적에 따라 신고 접수시부터 3년간 해당 정보를 3년간 보관 후 파기합니다. 단, 저 사양 단말기의 경우에는 신고된 음성정보가 저장되지 않을 수 있습니다. 4. 혹시라도 여러분이 관련 법령, 회사의 모든 약관 또는 정책을 준수하지 않는다면, 회사는 여러분의 위반행위 등을 조사할 수 있고, 해당 게시물 등을 삭제 또는 임시 삭제하거나 여러분의 서비스 이용을 잠시 또는 계속하여 중단하거나, 재가입에 제한을 둘 수도 있습니다. 또한 여러분이 서비스와 관련된 설비의 오작동이나 시스템의 파괴 및 혼란을 유발하는 등 서비스 제공에 악영향을 미치거나 안정적 운영을 심각하게 방해한 경우, 회사는 이러한 위험 활동이 확인된 여러분의 계정들에 대하여 이용제한을 할 수 있습니다. 다만, 여러분은 이용제한과 관련하여 조치 결과가 불만족스러울 경우 고객센터를 통해 이의를 제기할 수 있습니다. 5. 회사는 법령에서 정하는 기간 동안 여러분이 서비스를 이용하기 위해 로그인 혹은 접속한 기록이 없는경우 여러분이 등록한 이메일주소, 휴대폰번호로 이메일, 문자메시지 또는 카카오톡 메시지를 보내는 등 기타 유효한 수단으로 통지 후 여러분의 정보를 파기하거나 분리 보관할 수 있으며, 이로 인해 서비스 이용 을 위한 필수적인 정보가 부족할 경우 이용계약이 해지될 수도 있습니다. 6. 회사는 스팸성 메일(피싱, 바이러스 유포, 개인정보 탈취 등 각종 불법 및 사행성 스팸을 의미합니다)로부터 이용자를 보호하기 위해 수발신 메일에 대한 스팸 대응 및 보안 조치를 합니다. 더불어 유관기관의 권고가 있거나 이용자 보호를 위하여 필요하다고 판단하는 경우 이에 따른 추가적인 스팸 운영정책 및 기능을 제공합니다. 7. 본 조에서 정한 사항 및 그 밖에 서비스의 이용에 관한 자세한 사항은 서비스 운영정책 등 을 참고해 주시기 바랍니다."},{"doc":"카카오 통합 약관","article_no":9,"title":"게시물의 관리","text":"1. 여러분의 게시물이 정보통신망 이용촉진 및 정보보호 등에 관한 법률(이하 ‘정보통신망법’)및 저작권법등 관련법에 위반되는 내용을 포함하는 경우, 권리자는 회사에 관련법이 정한 절차에 따라 해당 게시물의 게시중단 및 삭제 등을 요청할 수 있으며, 회사는 관련법에 따라 조치를 취합니다. 2. 회사는 권리자의 요청이 없는 경우라도 권리침해가 인정될 만한 사유가 있거나 기타 회사의 정책 및 관련법에 위반되는 경우에는 관련법에 따라 해당 게시물에 대해 임시조치 등을 취할 수 있습니다. 3. 위와 관련된 세부절차는 정보통신망법 및 저작권법이 규정한 범위 내에서 회사가 정한 ‘권리침해 신고(카카오 서비스, Daum 서비스)’절차에 따릅니다."},{"doc":"카카오 통합 약관","article_no":10,"title":"권리의 귀속 및 저작물의 이용","text":"1. 여러분은 사진, 글, 정보, (동)영상, 카카오 서비스, Daum 서비스 또는 회사에 대한 의견이나 제안 등 콘텐츠(이하 ‘게시물’)를 서비스에 게시할 수 있으며, 이러한 게시물에 대한 저작권을 포함한 지적재산권은 당연히 권리자가 계속하여 보유합니다. 2. 여러분은 카카오 서비스 또는 Daum 서비스에 게시한 게시물에 대한 사용, 저장, 수정, 복제, 공중송신, 전시, 배포 등의 방식으로 이용할 수 있도록 사용을 허락하는 전 세계적이고 영구적인 라이선스를 회사에게 제공하게 됩니다. 본 라이선스에서 여러분이 회사에게 부여하는 권리는 서비스를 운영, 개선, 홍보하고 새로운 서비스를 개발하기 위한 범위 내에서 사용됩니다. 이러한 목적 범위 내에서 회사와 명시적인 업 무계약을 체결한 상대방 또는 다른 이용자에 대한 서브라이선스 또한 여기에 포함됩니다. 또한, 서비스의 개선 및 연구개발 목적으로 회사 및 회사의 계열사에서 게시물을 사용할 수 있습니다. 본 라이선스는 여러분이 서비스의 사용을 중단하거나 카카오계정 및/또는 Daum 아이디를 탈퇴한 후에도 존속하게 됩니다. 일부 서비스에서는 여러분이 제공한 콘텐츠에 접근하거나 이를 삭제하는 방법을 제공할 수 있습니다(다만 일부 서비스의 특성 및 콘텐츠의 성질 등에 따라 게시물의 삭제가 불가능할 수도 있습니다). 또한 일부 서비스에서는 제공된 콘텐츠에 대한 회사의 사용 범위를 제한하는 설정이 있습니다. 3. 여러분은 회사에 제공한 콘텐츠에 대해 회사에 라이선스를 부여하기 위해 필요한 권리를 보유해야 합니다. 이러한 권리를 보유하지 않아 발생하는 모든 문제에 대해서는 게시자가 책임을 부담하게 됩니다. 또한, 여러분은 음란하거나 폭력적이거나 기타 공서양속 및 법령에 위반하는 콘텐츠를 공개 또는 게시할 수 없습니다. 4. 회사는 여러분의 콘텐츠가 법령에 위반되거나 음란 또는 청소년에게 유해한 게시물, 차별 갈등을 조장하는 게시물, 도배·광고·홍보·스팸성 게시물, 계정을 양도 또는 거래하는 내용의 게시물, 타인을 사칭하는 게시물 등 이라고 판단되는 경우 이를 삭제하거나 게시를 거부할 수 있습니다. 다만 회사가 모든 콘텐츠를 검토할 의무가 있는 것은 아닙니다. 누군가 여러분의 권리를 침해하였다면, 고객센터를 통해 게시중단요청에 대한 도움을 받으실 수 있습니다. 위와 관련된 구체적인 기준 및 이용제한 절차의 내용은 카카오 운영정책에서 확인하실 수 있습니다. 5. 서비스에서는 회사가 보유하지 않은 일부 콘텐츠가 표시될 수 있습니다. 그러한 콘텐츠에 대해서는 콘텐츠를 제공한 주체가 단독으로 모든 책임을 부담하게 됩니다. 여러분이 서비스를 이용하더라도 다른 이용자의 콘텐츠에 대하여 어떠한 권리를 가지게 되는 것은 아닙니다. 여러분이 다른 이용자의 콘텐츠를 사용하기 위해서는 콘텐츠 소유자로부터 별도로 허락을 받아야 합니다."},{"doc":"카카오 통합 약관","article_no":11,"title":"유료 서비스의 이용","text":"1. 회사는 무료로 서비스를 제공하고 있으나, 일부 서비스의 경우 유료로 제공할 수 있습니다. 예를 들면, 카카오톡에서 친구들과 무료로 메시지를 주고 받을 수 있으나 일부 이모티콘 등은 유료로 구매해야 친구들에게 보낼 수 있으며, Daum 메일은 무료로 이용할 수 있으나 프리미엄 메일은 유료로 이용할 수 있습니다. 2. 여러분이 회사가 제공하는 유료서비스를 이용하는 경우 이용대금을 납부한 후 이용하는 것을 원칙으로 합니다. 회사가 제공하는 유료서비스에 대한 이용요금의 결제 방법은 핸드폰결제, 신용카드결제, 일반전화결제, 계좌이체, 무통장입금, 선불전자지급수단 결제 등이 있으며 각 유료서비스마다 결제 방법의 차이가 있을 수 있습니다. 매월 정기적인 결제가 이루어지는 서비스의 경우 여러분 개인이 해당 서비스의 이용을 중단하고 정기 결제의 취소를 요청하지 않는 한 매월 결제가 이루어집니다. 3. 회사는 결제의 이행을 위하여 반드시 필요한 여러분의 개인정보를 추가적으로 요구할 수 있으며, 여러분은 회사가 요구하는 개인정보를 정확하게 제공하여야 합니다. 4. 여러분 개인의 귀책사유로 이용요금을 환불하는 경우 일반적인 방법은 아래와 같습니다. • 회사가 제공하는 유료서비스가 결제 후 1회의 이용만으로 서비스의 이용이나 구매가 완료되는 서비스인 경우 해당 서비스를 이용한 후에는 환불이 불가능합니다. 단, 1회의 구매 완료 후 그 사용기한이 무제한인 아바타, 배경음악, 스킨 등의 서비스는 구매 완료일로부터 1년 이내에만 환불이 가능하며 환불금액은 구입금액*(365-사용일수/365)로 합니다. • 회사가 제공하는 유료서비스가 결제 후 1개월(결제 기준) 이하로 지속되는 서비스인 경우 해지일로부터 이용일수에 해당하는 금액을 제외한 나머지 금액을 환불합니다. 본 항의 규정은 일(1)개월 단위로 매월 결제되는 서비스의 경우에도 적용됩니다. • 회사가 제공하는 유료서비스가 결제 후 1개월(결제 기준)을 초과하여 지속되는 서비스인 경우 해지일로부터 이용일수에 해당하는 금액과 총 남은 이용일수의 10%를 제외한 금액을 환불합니다. 단, 유료 서비스 이용 개시일로부터 7일 이내에 해지를 요구하는 경우 이용일수에 해당하는 금액만을 제외하고 환불합니다. 5. 상기의 규정에도 불구하고 아래 각 호의 경우에는 여러분 개인이 결제한 전액을 환불합니다. 단, 1회의 구매 완료 후 그 사용기한이 무제한인 아바타, 배경음악, 스킨 등의 서비스는 구매 완료일로부터 1년 이내일 경우에만 환불합니다. • 여러분이 결제를 완료한 후 서비스를 이용한 내역이 없는 경우 • 서비스 장애 또는 회사가 제시한 최소한의 기술사양을 충족하였음에도 불구하고 회사의 귀책사유로 서비스를 이용하지 못한 경우 • 여러분이 구매한 서비스가 제공되지 않은 경우 • 제공되는 서비스가 표시·광고 등과 상이하거나 현저한 차이가 있는 경우 • 제공되는 서비스의 결함으로 서비스의 정상적인 이용이 현저히 불가능한 경우 6. 여러분은 이용요금에 대하여 이의를 제기할 수 있습니다. 단, 이용요금에 관한 이의는 그 사유 발생을 안 날로부터 1월, 그 사유가 발생한 날로부터 3월 이내에 제기하여야 합니다. 7. 회사는 과오금이 발생한 경우 또는 전액 환불의 경우 이용대금의 결제와 동일한 방법으로 환불하여야 합니다. 다만, 동일한 방법으로 환불이 불가능하거나 서비스의 중도해지로 인한 부분 환불 등의 경우에는 회사가 정하는 별도의 방법으로 환불합니다. 회사는 환불 의무가 발생한 날로부터 3영업일 이내에 환불을 진행하며, 환불이 지연되는 경우 지연이자율은 연리 11%로 합니다. 단, 환불에 여러분의 협조가 필요한 경우에 여러분의 귀책사유로 인한 환불 지연에 대해서는 지연이자를 지급하지 않습니다. 환불에 소요되는 비용은 여러분의 귀책사유로 인한 환불의 경우에는 여러분이, 회사의 귀책사유로 인한 환불의 경우에는 회사가 각각 부담합니다. 8. 본 약관의 유료서비스 규정과 각 각 개별 유료서비스 약관의 내용이 충돌하는 경우 각 개별약관의 규정에 따릅니다."},{"doc":"카카오 통합 약관","article_no":12,"title":"게시판 이용 상거래","text":"1. 여러분이 서비스를 이용하여 통신판매 또는 통신판매중개를 업으로 하는 경우 전자상거래 등에서의 소비자보호에 관한 법률(이하 ‘전자상거래법’)에 따른 의무사항을 준수하여야 합니다. 2. 여러분이 통신판매 또는 통신판매중개를 함에 있어 다른 이용자와 전자상거래 관련 분쟁이 발생하는 경우, 회사는 다른 이용자에게 소비자피해 구제 대행 신청을 할 수 있는 장치를 마련합니다. 3. 회사는 전자상거래법에 따라 신원정보를 입력하는 기능 등을 제공하여 여러분의 신원정보를 확인하고, 여러분과 다른 이용자 사이에 분쟁이 발생하여 전자상거래법에 따라 소비자피해 분쟁조정기구, 공정거래 위원회, 시도지사 또는 시장 군수 구청장이 신원정보 제공을 요구하는 경우 이에 협조합니다."},{"doc":"카카오 통합 약관","article_no":13,"title":"서비스의 이용, 변경 및 종료","text":"1. 회사는 서비스를 365일, 24시간 쉬지 않고 제공하기 위하여 최선의 노력을 다합니다. 다만, 장비의 유지·보수를 위한 정기 또는 임시 점검 또는 다른 상당한 이유로 서비스의 제공이 일시 중단될 수 있으며, 이때에는 미리 서비스 제공화면에 공지하겠습니다. 만약, 회사로서도 예측할 수 없는 이유로 서비스가 중단된 때에는 회사가 상황을 파악하는 즉시 최대한 빠른 시일 내에 서비스를 복구하도록 노력하고, 2시간 이상 복구가 지연되는 경우 Daum 공지사항 또는 카카오 서비스 공지사항, 카카오 고객센터 공지사항 등에 게시하여 알려 드리겠습니다. 2. 회사의 서비스 제공을 위해 계약한 CP와의 계약 종료 및 변경, 서비스/회원 정책의 변경, 신규서비스 개시 등의 사유로 서비스의 내용이 변경되거나, 서비스가 종료될 수도 있습니다. 서비스 변경 사항 또는 종료는 개별 서비스의 화면 또는 공지사항 란에 게시하여 여러분들께 알려드리겠습니다. 여러분께 중대한 영향을 미치는 서비스 변경 사항이나 종료는 전자메일(전자메일이 없는 경우 서비스 내 알림 등 별도의 전자적 수단) 또는 전화번호로 문자메세지를 발송하는 방법 등으로 개별적으로 알려드리겠습니다. 이 때 원만한 서비스 및 정책 변경 등을 위하여 서비스 이용 시 재로그인 또는 추가적인 동의 절차 등이 필요할 수 있습니다."},{"doc":"카카오 통합 약관","article_no":14,"title":"이용계약 해지","text":"1. 여러분이 서비스의 이용을 더 이상 원치 않는 때에는 언제든지 서비스 내 제공되는 메뉴를 이용하여 서비스 이용계약의 해지 신청을 할 수 있으며, 회사는 법령이 정하는 바에 따라 신속히 처리하겠습니다. 2. 이용계약이 해지되면 법령 및 개인정보 처리방침에 따라 여러분의 정보를 보유하는 경우를 제외하고는 여러분의 정보나 여러분이 작성한 게시물 등 모든 데이터는 삭제됩니다. 다만, 여러분이 작성한 게시물이 제3자에 의하여 스크랩 또는 다른 공유 기능으로 게시되거나, 여러분이 제3자의 게시물에 댓글 등 게시물을 추가하는 등의 경우에는 해당 게시물 및 댓글이 삭제되지 않으므로 반드시 해지신청 전에 삭제하신 후 탈퇴하시기 바랍니다. 3. 또한, 여러분은 다양한 서비스 중에서 일부 서비스만을 선택적으로 해지하실 수 있으며, 이 경우에는 해지된 서비스에 대한 데이터만 삭제되며, 다른 서비스 이용을 위한 카카오계정 및 Daum 아이디는 삭제되지 않고 남아 있게 됩니다. 4. 유료서비스 이용계약의 해지는 여러분의 서비스 해지 신청 및 회사의 승낙에 의해 성립하게 되고, 환불할 금액이 있는 경우 환불도 이루어 지게 됩니다. 다만 각 개별 유료서비스에서 본 약관과 다른 계약해지 방법 및 효과를 규정하고 있는 경우 각 개별약관의 규정에 따릅니다. 5. 이용계약이 해지된 경우라도 여러분은 다시 회사에 대하여 이용계약의 체결을 신청할 수 있습니다. 다만, 일부 서비스의 경우 다시 이용계약을 체결함에 있어 시간적 제한 등이 따를 수 있으며 이에 대한 구체적인 내용은 세부지침에서 확인하실 수 있습니다."},{"doc":"카카오 통합 약관","article_no":15,"title":"개인정보의 보호","text":"여러분의 개인정보의 안전한 처리는 회사에게 있어 가장 중요한 일 중 하나입니다. 여러분의 개인정보는 서비스의 원활한 제공을 위하여 여러분이 동의한 목적과 범위 내에서만 이용됩니다. 법령에 의하거나 여러분이 별도로 동의하지 아니하는 한 회사가 여러분의 개인정보를 제3자에게 제공하는 일은 결코 없으므로, 안심하셔도 좋습니다. 회사가 여러분의 개인정보를 안전하게 처리하기 위하여 기울이는 노력이나 기타 자세한 사항은 Daum 개인정보처리방침과 카카오 개인정보처리방침을 참고하여 주십시오."},{"doc":"카카오 통합 약관","article_no":16,"title":"위치기반서비스 제공","text":"1. 회사는 여러분의 실생활에 더욱 보탬이 되는 유용한 서비스를 제공하기 위하여 서비스에 위치기반서비스를 포함시킬 수 있습니다. 2. 회사의 위치기반서비스는 여러분의 단말기기의 위치정보를 수집하는 위치정보사업자로부터 위치정보를 전달받아 제공하는 무료서비스이며, 구체적으로는 아래와 같습니다. • 여러분의 현재 위치 또는 특정 위치를 다른 이용자와 공유하거나 그와 관련된 게시물을 작성할 수 있도록 하는 서비스(장소공유서비스) • 여러분의 현재 위치를 이용한 생활 정보나 광고성 정보를 제공하는 서비스(정보제공서비스) • 여러분이 보유하는 사진 등 콘텐츠에 기록되거나 콘텐츠와 결합된 위치정보를 활용하여 다른 이용자와 콘텐츠를 공유하도록 도와주는 서비스(콘텐츠공유서비스) 3. 여러분이 14세 미만 이용자로서 개인위치정보를 활용한 위치기반서비스를 이용하기 위해서는 회사는 여 러분의 개인위치정보를 이용 또는 제공하게 되며, 이 경우 부모님 등 법정대리인의 동의가 먼저 있어야 합니다. 만약 법정대리인의 동의 없이 위치기반서비스가 이용된 것으로 판명된 때에는 회사는 즉시 여러분의 위치기반서비스 이용을 중단하는 등 적절한 제한을 할 수 있습니다. 4. 여러분(14세 미만 이용자의 법정대리인 포함)은 서비스와 관련된 개인위치정보의 이용, 제공 목적, 제공받는 자의 범위 및 위치기반서비스의 일부에 대하여 동의를 유보하거나, 이용·제공에 대한 동의의 전부 또는 일부 철회할 수 있으며, 일시적인 중지를 요구할 수 있습니다. 회사는 위치정보의 보호 및 이용 등에 관한 법률의 규정에 따라 개인위치정보 및 위치정보 이용·제공사실 확인자료를 6개월 이상 보관하며, 여러분이 동의의 전부 또는 일부를 철회한 때에는 회사는 철회한 부분에 해당하는 개인위치정보 및 위치정보 이용·제공사실 확인자료를 지체 없이 파기하겠습니다. 5. 여러분(14세 미만 이용자의 법정대리인 포함)은 회사에 대하여 여러분에 대한 위치정보 이용·제공사실 확인자료나, 여러분의 개인위치정보가 법령에 의하여 제3자에게 제공되었을 때에는 그 이유 및 내용의 열람 또는 고지를 요구할 수 있고, 오류가 있는 때에는 정정을 요구할 수 있습니다. 만약, 회사가 여러분의 개인위치정보를 여러분이 지정하는 제3자에게 직접 제공하는 때에는 법령에 따라 개인위치정보를 수집한 스마트폰 등으로 여러분에게 개인위치정보를 제공받는 자, 제공 일시 및 제공 목적을 즉시 통보하겠습니다. 6. 회사는 8세 이하의 아동 등(금치산자, 중증 정신장애인 포함)의 보호의무자가 개인위치정보의 이용 또는 제공에 서면으로 동의하는 경우에는 해당 본인의 동의가 있는 것으로 보며, 이 경우 보호의무자는 개인위치정보주체의 권리를 모두 행사할 수 있습니다. 7. 만약 회사가 제공하는 위치기반서비스와 관련하여 여러분의 권리를 침해당했거나 권리행사가 필요한 경우 고객센터를 통해 도움을 받으실 수 있으며, 여러분과 회사 간의 위치정보와 관련한 분쟁에 대하여 협의가 어려운 때에는 여러분은 위치정보의 보호 및 이용 등에 관한 법률 제 28조 제2항 및 개인정보보호법 제43조의 규정에 따라 개인정보 분쟁조정위원회에 조정을 신청할 수 있습니다."},{"doc":"카카오 통합 약관","article_no":17,"title":"인증서비스","text":"1. 본 조에서 사용하는 용어의 정의는 다음과 같습니다. • 인증서비스 : 회사가 제공하는 전자서명과 인증서를 활용한 일체의 서비스를 말합니다. • 전자서명: 서명자의 신원을 확인하고 서명자가 해당 전자문서에 서명하였다는 사실을 나타내는데 이용하 기 위하여 전자문서에 첨부되거나 논리적으로 결합된 전자적 형태의 정보를 말합니다. • 인증서: 인증서라 함은 회사가 인증서비스를 통하여 발급하는 전자서명생성정보가 회원에게 유일하게 속한다는 사실 등을 확인하고 이를 증명하는 전자적 정보를 말합니다. • 전자서명생성정보: 전자서명을 생성하기 위하여 이용하는 전자적 정보를 말합니다. • 이용기관: 인증회원의 전자서명 및 인증서를 바탕으로 한 거래 등을 위하여 인증서비스를 이용하려는 제3자를 말합니다. • 인증회원 : 회사로부터 전자서명생성정보를 인증 받은 회원을 말합니다. 2. 회사는 전자서명생성정보 및 인증서를 발급하고, 전자서명과 인증서를 활용한 각종 서비스를 아래 각 호와 같이 인증회원에게 제공합니다. 이 때 회사는 필요한 경우 인증서비스의 유형 및 종류를 추가하거나, 부가서비스를 별도로 제공할 수 있습니다. • 전자서명생성정보 및 인증서 발급 • 전자서명 및 인증서를 활용한 각종 서비스 • 이용기관 로그인 및 신원확인을 위한 간편인증 • 기타 전자서명인증업무 운영준칙에서 정하는 서비스 3. 회원은 회사가 정하는 방법에 따라 정확한 정보만을 제공하여 인증서비스에 가입하여야 하며, 인증서를 발급받음과 동시에 인증회원으로 전환됩니다. 인증서는 명의자 당 1개의 카카오계정에서 1대의 기기에만 발급됩니다. 만일 인증회원이 다른 기기 또는 다른 카카오계정에서 인증서를 재발급하는 경우 기존에 발급받은 인증서는 자동 폐지됩니다. 4. 인증회원은 회사가 정한 방법에 따라 인증서비스를 이용하여야 합니다. 또한 인증회원은 자신의 전자서명생성정보와 인증서 및 이와 관련된 모든 정보를 안전하게 관리하고 인증서비스 이용 기간 중 회사에 제공한 정보 및 인증서에 포함된 정보가 정확하고 완전하게 유지되도록 하여야 합니다. 인증회원은 자신의 전자서명생성정보와 인증서 및 이에 관련된 정보를 타인에게 양도, 증여, 판매, 사용 허락할 수 없으며, 분실, 훼손, 도난 또는 유출되거나 그러할 위험이 있다고 인지한 경우 즉시 그 사실을 회사에 통지하여야 합니다. 5. 회사는 다음 각 호의 경우 인증서의 신청 및 발급을 제한하거나 발급된 인증서를 인증회원의 동의 없이 폐지할 수 있습니다. • 피성년후견인 또는 피한정후견인이 법정대리인의 동의 없이 가입한 경우 • 타인 명의의 신청 및 정보 도용 등 신청 내용이 허위의 사실이라 판단되는 경우 • 회사가 제시하는 인증 절차를 완료하지 못하거나, 회사가 정하지 않은 비정상적인 방법으로 시스템에 접근하여 인증서비스에 가입하는 경우 • 회사로부터 이용 정지를 당하거나, 법령 또는 본 약관을 위반하는 등의 이유로 서비스 이용 계약이 해지된 회원이 재이용신청을 하는 경우 • 기타 회원의 귀책사유로 발급이 곤란한 경우 또는 회사가 정한 이용신청 요건이 충족되지 않은 경우 6. 인증회원은 인증서비스를 자유롭게 이용할 수 있으나, 아래 각 호의 행위는 하여서는 안 됩니다. • 회사가 정하지 않은 비정상적인 방법으로 시스템에 접근하거나 인증서비스를 이용하는 행위 • 부정한 방법으로 인증서를 발급받거나 행사하는 등 인증서비스를 불법적 또는 부당한 용도로 사용하는 행위 7. 회사는 다음 각 호의 경우 인증회원에게 발급한 인증서 이용의 일부 또는 전부를 제한할 수 있으며, 인증회원의 동의 없이 인증서를 즉시 폐지할 수 있습니다. • 인증서의 유효기간이 경과한 경우 • 인증회원이 인증서의 비밀번호를 연속하여 제한 횟수 이상 잘못 입력한 경우 • 인증회원의 카카오계정에 등록된 카카오톡 전화번호가 변경된 경우 • 인증회원의 사망, 구속 등으로 신원확인이나 전자거래가 불가능한 경우 • 인증서비스 가입 시 본인확인기관에서 전달받은 연계정보(CI)가 국적, 성별 등의 변경으로 더이상 유효하지 않음이 확인된 경우 • 회사가 인증서비스와 관련된 보안절차나 인증회원의 전자서명생성정보 유출과 같은 보안상의 이유로 기 발급된 인증서의 이용제한이 필요한 경우 • 전시, 사변, 천재지변 또는 이에 준하는 비상사태가 발생하거나 발생할 우려가 있는 경우 • 회사 고객센터 등을 통해서 인증서의 분실신고가 접수된 경우 • 인증회원의 인증서가 부정하게 사용된 사실을 회사가 인지한 경우 등 인증회원이 본 약관 및 운영정책을 포함한 회사의 서비스 이용 정책을 위반하거나 위반할 우려가 있다고 회사가 판단하는 경우 • 기타 인증서비스의 안전성과 신뢰성을 저해할 우려가 있는 경우 8. 회사는 인증서를 사용하는 인증회원과 이용기관 상호간 거래에 대하여 어떠한 책임도 부담하지 않으며, 회사는 인증회원과 이용기관의 귀책사유로 인하여 발생한 손해에 대하여 회사의 귀책사유가 없는 경우 책임을 부담하지 않습니다. 9. 본 조에서 정하고 있는 내용 외에 인증서비스와 관련된 상세한 사항은 전자서명법 등을 포함한 관련법령 및 회사가 별도로 정한 전자서명인증업무준칙에 따르며, 회사는 인증서비스 공지사항 및 고객센터 도움말 페이지 등을 통하여 회원에게 안내합니다."},{"doc":"카카오 통합 약관","article_no":18,"title":"손해배상 등","text":"1. 회사는 법령상 허용되는 한도 내에서 서비스와 관련하여 본 약관에 명시되지 않은 어떠한 구체적인 사항에 대한 약정이나 보증을 하지 않습니다. 또한, 회사는 CP(Contents Provider)가 제공하거나 회원이 작성하는 등의 방법으로 서비스에 게재된 정보, 자료, 사실의 신뢰도, 정확성 등에 대해서는 보증을 하지 않으며, 회사의 과실 없이 발생된 여러분의 손해에 대하여는 책임을 부담하지 아니합니다. 2. 회사는 회사의 과실로 인하여 여러분이 손해를 입게 될 경우 본 약관 및 법령에 따라 여러분의 손해를 배상하겠습니다. 다만 회사는 회사의 과실 없이 발생된 아래와 같은 손해에 대해서는 책임을 부담하지 않습니다. 또한 회사는 법률상 허용되는 한도 내에서 간접 손해, 특별 손해, 결과적 손해, 징계적 손해, 및 징 벌적 손해에 대한 책임을 부담하지 않습니다. • 천재지변 또는 이에 준하는 불가항력의 상태에서 발생한 손해 • 여러분의 귀책사유로 서비스 이용에 장애가 발생한 경우 • 서비스에 접속 또는 이용과정에서 발생하는 개인적인 손해 • 제3자가 불법적으로 회사의 서버에 접속하거나 서버를 이용함으로써 발생하는 손해 • 제3자가 회사 서버에 대한 전송 또는 회사 서버로부터의 전송을 방해함으로써 발생하는 손해 • 제3자가 악성 프로그램을 전송 또는 유포함으로써 발생하는 손해 • 전송된 데이터의 생략, 누락, 파괴 등으로 발생한 손해, 명예훼손 등 제3자가 서비스를 이용하는 과정에서 발생된 손해 • 기타 회사의 고의 또는 과실이 없는 사유로 인해 발생한 손해"},{"doc":"카카오 통합 약관","article_no":19,"title":"청소년보호","text":"모든 연령대가 자유롭게 이용할 수 있는 공간으로써 유해 정보로부터 청소년을 보호하고 청소년의 안전한 인터넷 사용을 돕기 위해 정보통신망법에서 정한 청소년보호정책을 별도로 시행하고 있으며, 구체적인 내용은 서비스 초기 화면 등에서 확인할 수 있습니다."},{"doc":"카카오 통합 약관","article_no":20,"title":"통지 및 공지","text":"회사는 여러분과의 의견 교환을 소중하게 생각합니다. 여러분은 언제든지 고객센터에 방문하여 의견을 개진할 수 있습니다. 회사는 카카오계정 또는 Daum 아이디로 사용하는 이메일 주소로 이메일을 발송하거나, 여러분이 등록한 휴대폰번호로 카카오톡 메시지 또는 문자메시지를 보내거나, 서비스 내 전자쪽지 발송, 알림 메시지를 띄우는 등 합리적으로 가능한 방법으로 여러분에게 공지 또는 통지하며, 서비스 이용자 전체에 대한 공지는 칠(7)일 이상 서비스 공지사항 란에 게시함으로써 효력이 발생합니다."},{"doc":"카카오 통합 약관","article_no":21,"title":"분쟁의 해결","text":"본 약관 또는 서비스는 대한민국법령에 의하여 규정되고 이행됩니다. 서비스 이용과 관련하여 회사와 여러분 간에 분쟁이 발생하면 이의 해결을 위해 성실히 협의할 것입니다. 그럼에도 불구하고 해결되지 않으면 민사소송법의 관할법원에 소를 제기할 수 있습니다."}]'''


def _load_articles():
    """노트북에 내장된 72개 약관 원문을 직접 로드한다.

    외부 URL에 의존하지 않도록 해 fresh Colab 세션에서도
    동일한 원문으로 안정적으로 실행되게 한다.
    """
    _articles = _json.loads(_ARTICLES_JSON_FALLBACK)
    print(f"[약관] 내장 원문 {len(_articles)}개 조항 로드")
    return _articles


ARTICLES = _load_articles()
# 문서명을 반드시 포함한다 — 안 넣으면 질문이 "카카오계정 약관에서는..."처럼 문서를
# 명시해도 그 신호가 인덱스에 없어 무시된다. 실측: "카카오계정 약관에 규정 안 된 사항은
# 어떻게 되나요?"가 문서명 없이는 6위로 밀리고, 엉뚱한 문서 본문에 우연히 섞인 "카카오"
# 글자(예: "카카오 운영정책"이라는 무관한 문구)가 낚여서 1위를 차지했다. 문서명을 넣자
# 즉시 1위로 정정됐고, 공개 10문항 회귀도 없었다(MRR 1.0 유지).
_CORPUS_TEXT = [f"{a['doc']} {a['title']} {a['text']}" for a in ARTICLES]


# -------------------------------------------------------------------------------------
# 1-2. BM25 인덱스 — 형태소 분석기 없이 문자 bigram으로 토크나이즈한다.
#      약관 질문은 "제16조 제2항", "2시간", "8세 이하"처럼 조사가 붙은 고유 토큰이 많아
#      어절 단위 토크나이즈보다 문자 bigram이 정확 일치 회수율이 더 안정적이다.
# -------------------------------------------------------------------------------------
def _bigram_tokens(text):
    t = _re.sub(r"\s+", "", _unicodedata.normalize("NFC", text))
    if len(t) < 2:
        return [t] if t else []
    return [t[i:i + 2] for i in range(len(t) - 1)]


_BM25 = BM25Okapi([_bigram_tokens(t) for t in _CORPUS_TEXT])


# -------------------------------------------------------------------------------------
# 1-2b. 항(①②③, 1.2.3.) 단위 서브청크 인덱스 — 조 전체 스코어링의 길이 편향을 보정한다.
#      실측(오프라인 하네스): "이용자의 개인정보는 어떤 목적으로 이용되나요?" 같은 질문에서
#      조 전체 BM25는 3000자 넘는 무관한 긴 조항을 1~2위로 잘못 뽑았다(짧고 관련 있는
#      조항보다 우연히 겹치는 글자가 많아서). 리랭커가 이를 바로잡아 주지만, 리랭커 다운로드가
#      실패하는 폴백 경로에서는 이 편향이 그대로 남는다.
#      항 단위로 쪼개 "그 조에서 가장 잘 맞는 항의 점수"를 대표 점수로 쓰면 길이 편향이
#      줄어들지만, 반대로 짧고 밀도 높은 항이 문맥과 무관하게 과대평가되는 새 문제가 생긴다
#      (예: "탈퇴" 질문에서 "계정 생성 거절" 조항의 항 하나가 계정 관련 단어만으로 오탐).
#      두 방식을 각각 썼을 때 자체 평가셋(공개 10문항 + 패러프레이즈 4문항, 리랭커 없이)에서
#      MRR이 조 전체 0.952 / 항 최고점 0.893으로 항 단위가 오히려 더 나쁘게 나왔다.
#      두 순위를 RRF(Reciprocal Rank Fusion)로 섞으면 0.964로 상승 — 두 방식의 실패
#      유형이 서로 다르기 때문에(정반대 편향) 섞을 때 서로를 보완한다. 그래서 항 단위 점수를
#      "단독"이 아니라 조 전체 점수와 "결합"해서만 쓴다.
def _split_subchunks(text):
    # (?<!\d): 순수 lookahead 분할은 re.split이 제로폭 매치 뒤 커서를 1글자만 전진시켜서,
    # "10." 같은 두 자리 이상 번호에서 "1"과 "0."으로 다시 쪼개지는 버그가 있었다
    # (실측: 계정약관 제12조·통합서비스약관 제12조에서 10~17번 항목이 전부 깨짐,
    # 파편 21개 발견). 숫자 앞이 또 숫자면 매치하지 않게 해 다자리 번호를 통째로 지킨다.
    parts = _re.split(r"(?=[①②③④⑤⑥⑦⑧⑨⑩])|(?=(?<!\d)\d+\.[^\d])", text)
    parts = [p.strip() for p in parts if p.strip()]
    return parts if parts else [text]


def _is_exhaustive_question(question):
    """목록 전체/여러 항목을 빠짐없이 요구하는 질문인지 가볍게 판별한다.

    특정 공개 문항의 의미를 분류하려는 목적이 아니라,
    질문 표면에 명시된 '모두/전부/각각/N가지' 같은 구조만 확인한다.
    """
    markers = (
        "모두",
        "전부",
        "각각",
        "나열",
        "목록",
        "몇 가지",
    )
    if any(marker in question for marker in markers):
        return True

    return _re.search(r"\d+\s*(?:개|가지|명|종류|항목)", question) is not None


def _rank_article_subchunks(question, article):
    """한 조항 내부의 항/목록 조각을 질문과의 BM25 관련도 순으로 반환한다."""
    chunks = _split_subchunks(article["text"])

    if len(chunks) <= 1:
        return chunks

    bm25 = BM25Okapi([_bigram_tokens(chunk) for chunk in chunks])
    scores = bm25.get_scores(_bigram_tokens(question))

    ranked_indices = sorted(
        range(len(chunks)),
        key=lambda i: scores[i],
        reverse=True,
    )
    return [chunks[i] for i in ranked_indices]


def _best_subchunk_for_question(question, article):
    """리랭커가 긴 조항의 앞부분만 보지 않도록 가장 관련 높은 내부 조각을 사용한다."""
    ranked = _rank_article_subchunks(question, article)
    return ranked[0] if ranked else article["text"]


# -------------------------------------------------------------------------------------
# V2. 질문을 의미 '유형'으로 하드코딩하지 않고 1~3개의 질문 단위로 가볍게 분해한다.
#
# 핵심 원칙:
#   - actor/order/effect 같은 의미 라벨을 붙이지 않는다.
#   - 질문에 실제로 존재하는 절/문장 경계만 이용한다.
#   - 분해가 실패해도 항상 원 질문 전체를 별도로 scoring하므로 기존 성능을 보존한다.
# -------------------------------------------------------------------------------------
_QUESTION_CUE_RE = _re.compile(
    r"(?:무엇|어떤|어떻게|누가|누구|언제|어디|몇|왜|얼마|어느|"
    r"여부|인가|나요|까요|습니까|되는지|하는지|인지|있는지|없는지)"
)


def _normalize_question_part(text):
    text = _re.sub(r"\s+", " ", text).strip()
    return text.strip(" ,;:?.!？。")


def _decompose_question(question, max_parts=3):
    """질문을 최대 3개의 의미 단위로 보수적으로 분리한다.

    의미 유형을 추론하지 않고, 실제 질문의 문장/접속 구조만 사용한다.
    분해된 각 조각은 evidence coverage용 보조 query이며,
    원 질문 전체는 항상 별도로 사용한다.
    """
    q = _normalize_question_part(question)
    if not q:
        return [question]

    # 명확한 접속어/구두점 경계를 우선 사용한다.
    marked = _re.sub(
        r"\s+(그리고|또한|그렇다면|그러면)\s+",
        " ||| ",
        q,
    )

    # 한국어 복합 질문에서 자주 쓰이는 '...하며, ...' 형태를 분리한다.
    # 의미를 해석하지 않고 문법적 연결부만 경계로 사용한다.
    marked = _re.sub(
        r"(하며|이며|되고|되며|이고|하고)\s*,\s*",
        r"\1 ||| ",
        marked,
    )

    # 세미콜론과 연속된 물음표 문장도 안전한 경계다.
    marked = _re.sub(r"[;；]+", " ||| ", marked)
    marked = _re.sub(r"[?？]+\s+(?=\S)", " ||| ", marked)

    raw_parts = [
        _normalize_question_part(p)
        for p in marked.split("|||")
    ]
    raw_parts = [p for p in raw_parts if len(p) >= 6]

    # 쉼표만으로 연결된 복합 질문은 양쪽 모두 질문 표지가 있을 때만 추가 분리한다.
    if len(raw_parts) == 1 and "," in q:
        comma_parts = [
            _normalize_question_part(p)
            for p in q.split(",")
        ]
        comma_parts = [p for p in comma_parts if len(p) >= 6]
        cue_parts = [p for p in comma_parts if _QUESTION_CUE_RE.search(p)]
        if len(cue_parts) >= 2:
            raw_parts = comma_parts

    # 실제 질문성 표현이 있는 조각을 우선한다.
    cue_parts = [p for p in raw_parts if _QUESTION_CUE_RE.search(p)]
    parts = cue_parts if len(cue_parts) >= 2 else raw_parts

    # 지나친 분해는 오히려 노이즈가 되므로 최대 3개로 제한한다.
    if len(parts) > max_parts:
        parts = parts[: max_parts - 1] + [
            _normalize_question_part(" ".join(parts[max_parts - 1:]))
        ]

    # 중복 제거. 하나뿐이면 굳이 분해하지 않는다.
    deduped = []
    seen = set()
    for part in parts:
        key = _re.sub(r"\s+", "", part)
        if key and key not in seen:
            seen.add(key)
            deduped.append(part)

    return deduped if len(deduped) >= 2 else [q]


# -------------------------------------------------------------------------------------
# V2/V6. 생성용 evidence selection — 질문 coverage + 원문 key fact 보존
#
# 기존: 원 질문 전체와 가장 비슷한 문장만 상대점수로 선택
# 개선: 원 질문 전체 + 분해된 질문 단위 각각이 최소 한 번은 근거 선택에 반영되도록 한다.
#       특정 질문 유형이나 정답 표현을 하드코딩하지 않는다.
# -------------------------------------------------------------------------------------
def _requested_count(question):
    """질문에 명시된 요구 개수(N개/N가지/N명/N종류/N항목)가 있으면 반환한다."""
    match = _re.search(
        r"(\d+)\s*(?:개|가지|명|종류|항목)",
        question,
    )
    return int(match.group(1)) if match else None


def _select_list_evidence(question, article):
    """명시적 N개 목록 질문에서 관련성이 확인되는 연속 번호 목록만 우선 전달한다.

    조항 안에 번호 목록이 여러 개 있을 수 있으므로 '첫 N개'를 무조건 쓰지 않는다.
    1,2,3...으로 이어지는 각 목록 블록을 찾고, 바로 앞 문맥까지 포함해 질문과의
    문자-bigram coverage가 가장 높은 블록을 선택한다. 관련성이 약하면 일반 coverage
    selection으로 넘긴다.
    """
    count = _requested_count(question)
    if count is None or not _is_exhaustive_question(question):
        return None

    chunks = _split_subchunks(article["text"])
    runs = []
    current = []
    current_start = None
    prev_num = None

    for idx, chunk in enumerate(chunks):
        match = _re.match(r"^(\d+)\.", chunk)
        if not match:
            if current:
                runs.append((current_start, current))
                current = []
                current_start = None
                prev_num = None
            continue

        num = int(match.group(1))
        if not current or num != prev_num + 1:
            if current:
                runs.append((current_start, current))
            current = [chunk]
            current_start = idx
        else:
            current.append(chunk)
        prev_num = num

    if current:
        runs.append((current_start, current))

    candidates = []
    q_tokens = set(_bigram_tokens(question))
    if not q_tokens:
        return None

    for start_idx, items in runs:
        if len(items) < count:
            continue

        header = ""
        if start_idx is not None and start_idx > 0:
            prev_chunk = chunks[start_idx - 1]
            if not _re.match(r"^\d+\.", prev_chunk):
                header = prev_chunk

        chosen = items[:count]
        block_text = " ".join(([header] if header else []) + chosen)
        block_tokens = set(_bigram_tokens(block_text))
        score = len(q_tokens & block_tokens) / max(1, len(q_tokens))
        candidates.append((score, chosen))

    if not candidates:
        return None

    best_score, best_items = max(candidates, key=lambda x: x[0])

    # 질문과 목록 블록의 표면 연관성이 너무 낮으면 잘못된 목록일 수 있으므로 강제하지 않는다.
    if best_score < 0.10:
        return None

    return "\n".join(best_items)


def _make_evidence_units_with_groups(article, max_short_len=350):
    """생성용 최소 근거 단위와 상위 subchunk(항/호) 정보를 함께 만든다.

    검색은 문장 수준까지 세밀하게 하되, 최종 evidence를 만들 때는
    선택된 문장이 속한 같은 항/호 문맥을 다시 복원할 수 있도록 group 정보를 보존한다.
    """
    chunks = _split_subchunks(article["text"])
    units = []

    for group_idx, chunk in enumerate(chunks):
        chunk = chunk.strip()
        if not chunk:
            continue

        if len(chunk) <= max_short_len:
            units.append({
                "text": chunk,
                "group": group_idx,
                "parent": chunk,
            })
            continue

        sentences = _re.split(
            r"(?<=[.!?])\s+|\n+",
            chunk,
        )
        sentences = [
            sentence.strip()
            for sentence in sentences
            if sentence.strip()
        ]

        if not sentences:
            sentences = [chunk]

        for sentence in sentences:
            units.append({
                "text": sentence,
                "group": group_idx,
                "parent": chunk,
            })

    if not units:
        units = [{
            "text": article["text"],
            "group": 0,
            "parent": article["text"],
        }]

    return units, chunks


def _select_coverage_evidence(
    question,
    article,
    max_units=6,
    per_part=2,
    near_ratio=0.70,
    whole_rel_ratio=0.55,
    parent_expand_chars=900,
    adjacent_parent_ratio=0.60,
    adjacent_min_coverage=0.12,
    max_parent_groups=4,
):
    """질문 coverage를 확보하면서 같은 규정의 핵심 문맥을 최대한 보존한다.

    기존 방식은 질문과 가장 비슷한 문장만 남겨 정답 의미는 맞더라도
    같은 항/호 안의 조건·효력·후속 사실이나 원문 핵심 표현이 잘릴 수 있었다.

    개선 원칙:
    1) 질문을 최대 3개 단위로 가볍게 분해하고 각 부분의 상위 evidence를 찾는다.
    2) 선택된 문장이 속한 상위 항/호(subchunk)가 너무 길지 않으면 그 항/호 전체를 보존한다.
       따라서 한 규정 안의 연결된 조건·효력·예외·후속 문장을 함께 Qwen이 볼 수 있다.
    3) 바로 인접한 항/호도 질문과 충분히 관련 있으면 제한적으로 함께 보존한다.
    4) 아주 짧은 조항은 과도하게 자르지 않고 조항 전체를 그대로 전달한다.
    5) 원 질문 전체 기준 scoring도 함께 사용해 질문 분해 오류를 보정한다.

    특정 공개 문항의 정답이나 의미 유형은 사용하지 않는다.
    """
    units, parent_chunks = _make_evidence_units_with_groups(article)

    if len(units) <= 1:
        return article["text"]

    # 이미 충분히 짧은 조항은 문장 단위 pruning으로 핵심 수식어를 잃는 이득보다
    # 정보 손실 위험이 더 크므로 원문 전체를 보존한다.
    if len(units) <= 4:
        return article["text"]

    unit_texts = [unit["text"] for unit in units]
    tokenized_units = [_bigram_tokens(text) for text in unit_texts]
    bm25 = BM25Okapi(tokenized_units)

    parts = _decompose_question(question)
    selected_units = set()

    # 각 질문 단위가 최소 한 근거에 연결되도록 한다.
    for part in parts:
        scores = bm25.get_scores(_bigram_tokens(part))
        ranked = sorted(
            range(len(units)),
            key=lambda i: scores[i],
            reverse=True,
        )
        if not ranked:
            continue

        top_idx = ranked[0]
        top_score = float(scores[top_idx])
        if top_score > 0:
            selected_units.add(top_idx)

        # 한 질문 부분이 같은 규정의 두 문장에 걸쳐 있을 수 있으므로
        # 2위가 충분히 가까울 때만 함께 보존한다.
        if per_part >= 2 and len(ranked) >= 2 and top_score > 0:
            second_idx = ranked[1]
            second_score = float(scores[second_idx])
            if second_score >= top_score * near_ratio:
                selected_units.add(second_idx)

        if len(selected_units) >= max_units:
            break

    # 질문 분해가 잘못되었더라도 원 질문 전체 기준으로 강한 근거를 추가한다.
    whole_scores = bm25.get_scores(_bigram_tokens(question))
    whole_ranked = sorted(
        range(len(units)),
        key=lambda i: whole_scores[i],
        reverse=True,
    )

    if whole_ranked:
        whole_top = float(whole_scores[whole_ranked[0]])
        if whole_top > 0:
            cutoff = whole_top * whole_rel_ratio
            for idx in whole_ranked:
                if len(selected_units) >= max_units:
                    break
                if float(whole_scores[idx]) < cutoff:
                    break
                selected_units.add(idx)

    if not selected_units and whole_ranked:
        selected_units.add(whole_ranked[0])

    # -------------------------------------------------------------------------
    # KeyFact 보존 핵심: 선택된 문장 자체만 보내지 않고 같은 상위 항/호 문맥을 복원한다.
    # -------------------------------------------------------------------------
    selected_groups = {
        units[idx]["group"]
        for idx in selected_units
    }

    # 질문과 각 parent subchunk의 표면 coverage를 계산한다.
    q_tokens = set(_bigram_tokens(question))
    parent_coverage = []
    for chunk in parent_chunks:
        if not q_tokens:
            parent_coverage.append(0.0)
            continue
        c_tokens = set(_bigram_tokens(chunk))
        parent_coverage.append(
            len(q_tokens & c_tokens) / max(1, len(q_tokens))
        )

    # 선택된 규정 바로 앞/뒤의 항도 같은 질문과 충분히 연결되면 보존한다.
    # 예: 직접 절차를 정한 항 다음에 그 절차의 효력/권리를 정한 항이 붙는 경우.
    candidate_adjacent = []
    for group_idx in sorted(selected_groups):
        base_score = parent_coverage[group_idx] if group_idx < len(parent_coverage) else 0.0
        for neighbor in (group_idx - 1, group_idx + 1):
            if neighbor < 0 or neighbor >= len(parent_chunks):
                continue
            if neighbor in selected_groups:
                continue

            neighbor_score = parent_coverage[neighbor]
            required = max(
                adjacent_min_coverage,
                base_score * adjacent_parent_ratio,
            )
            if neighbor_score >= required:
                candidate_adjacent.append((neighbor_score, neighbor))

    for _, group_idx in sorted(candidate_adjacent, reverse=True):
        if len(selected_groups) >= max_parent_groups:
            break
        selected_groups.add(group_idx)

    # 최종 evidence는 원문 순서를 유지한다.
    # 일반적인 항/호는 전체 문맥을 보존하고, 아주 긴 항만 선택 문장 주변으로 제한한다.
    output_blocks = []
    for group_idx in sorted(selected_groups):
        parent = parent_chunks[group_idx].strip()
        if not parent:
            continue

        if len(parent) <= parent_expand_chars:
            output_blocks.append(parent)
            continue

        group_unit_indices = [
            i for i, unit in enumerate(units)
            if unit["group"] == group_idx
        ]
        chosen_in_group = sorted(
            i for i in selected_units
            if units[i]["group"] == group_idx
        )

        # 긴 항에서는 선택 문장과 바로 인접한 문장만 보존해 prompt 폭증을 막는다.
        expanded = set(chosen_in_group)
        group_set = set(group_unit_indices)
        for idx in chosen_in_group:
            if idx - 1 in group_set:
                expanded.add(idx - 1)
            if idx + 1 in group_set:
                expanded.add(idx + 1)

        if expanded:
            output_blocks.append(
                "\n".join(units[i]["text"] for i in sorted(expanded))
            )
        else:
            output_blocks.append(parent[:parent_expand_chars])

    if not output_blocks:
        # 예외 상황에서도 최소 하나의 원문 근거를 보장한다.
        if selected_units:
            output_blocks = [
                units[i]["text"]
                for i in sorted(selected_units)
            ]
        else:
            output_blocks = [article["text"]]

    return "\n".join(output_blocks)


_SUBCHUNKS = []  # [(article_index, subchunk_text), ...]
for _ai, _a in enumerate(ARTICLES):
    for _sc in _split_subchunks(_a["text"]):
        _SUBCHUNKS.append((_ai, f"{_a['doc']} {_a['title']} {_sc}"))

_BM25_SUB = BM25Okapi([_bigram_tokens(t) for _, t in _SUBCHUNKS])


def _bm25_rrf_scores(question, k=60):
    """조 전체 BM25 순위 + 항 최고점 BM25 순위를 RRF로 결합한 article별 점수를 반환한다."""
    whole_scores = _BM25.get_scores(_bigram_tokens(question))
    whole_rank = sorted(range(len(ARTICLES)), key=lambda i: whole_scores[i], reverse=True)
    whole_pos = {i: r for r, i in enumerate(whole_rank, 1)}

    sub_scores = _BM25_SUB.get_scores(_bigram_tokens(question))
    best_sub = {}
    for (ai, _), s in zip(_SUBCHUNKS, sub_scores):
        if ai not in best_sub or s > best_sub[ai]:
            best_sub[ai] = s
    sub_rank = sorted(best_sub, key=lambda ai: best_sub[ai], reverse=True)
    sub_pos = {i: r for r, i in enumerate(sub_rank, 1)}

    fused = {
        i: 1.0 / (k + whole_pos[i]) + 1.0 / (k + sub_pos.get(i, len(ARTICLES) + k))
        for i in whole_pos
    }
    return fused


# -------------------------------------------------------------------------------------
# 1-3. 리랭커 — 전체 72개 조항이 소규모 코퍼스이므로 BM25 상위 후보 전체를 cross-encoder로
#      재정렬한다. 원격 API가 아니라 로컬 GPU에서 실행되므로 금지 규정에 해당하지 않는다.
#      다운로드가 실패해도 전체 파이프라인이 죽지 않도록 BM25 단독 폴백을 둔다.
# -------------------------------------------------------------------------------------
_HAS_RERANKER = False
try:
    print("[로딩] 리랭커(BAAI/bge-reranker-v2-m3) ...")
    _RERANKER = CrossEncoder(
        "BAAI/bge-reranker-v2-m3",
        device=_DEVICE,
        max_length=512,
    )
    if _DEVICE == "cuda":
        _RERANKER.model.half()
    _HAS_RERANKER = True
except Exception as _exc:
    print(f"[경고] 리랭커 로딩 실패({type(_exc).__name__}: {_exc}) — BM25 단독으로 폴백합니다.")


def _retrieve(
    question,
    min_k=1,
    max_k=4,
    bm25_pool=30,
    rerank_margin=0.01,
    rrf_k=10,
):
    """RRF(조 전체+항 최고점) 후보를 리랭커로 재정렬하고 필요한 근거만 반환한다.

    검색 담당자 버전의 핵심 구조는 유지한다.
    - 문서명 + 제목 + 원문을 BM25 corpus에 사용
    - 조 전체 BM25 + 항 단위 최고점 BM25를 RRF로 결합
    - bge-reranker-v2-m3로 재정렬

    추가 보정:
    1) 질문에 공식 문서명이 명시되면 해당 문서(여러 개면 그 문서들의 합집합) 안에서만
       최종 후보를 선정해 유사한 다른 약관이 섞이는 것을 줄인다.
    2) CrossEncoder.predict()가 반환한 0~1 점수를 그대로 사용한다. 이미 정규화된 점수에
       sigmoid를 다시 적용하면 점수 차이가 압축되어 약한 후보가 과도하게 살아남는다.

    [리랭커 2채널 RRF 결합 — 조 전체 vs 질문과 가장 가까운 항 하나]
    둘 중 하나만 쓰면(조전체만 또는 서브청크만) 자체 평가셋(공개10+함정10) 종합
    MRR이 각각 0.9750으로 동점이었지만 실패 문항이 정반대였다(조전체 단독은 T01
    실패, 서브청크 단독은 T03 실패). BM25 때(조전체+항최고점 RRF)와 같은 이유로 두
    순위를 RRF 결합했더니 20문항 전부 1위, MRR 1.0000으로 개선됐다(실측 확인).

    [컷오프]
    RRF 결합 점수는 순위 역수의 합이라 확률이 아니라서 기존 dominance_gap(0.20)이나
    rel_gap_ratio는 이 스케일에 안 맞는다(결합 점수 최댓값 자체가 0.2 미만이라
    dominance_gap이 사실상 발동을 못 함). P02(정답3+무관1)·P05(정답1+위험1)·
    P08(정답1+위험2)로 이 새 스케일을 다시 실측한 결과 위험 후보들의 1위 대비
    점수차가 0.015 이상이었고 margin=0.01이 이들을 정확히 걸러냈다. P02의 3번째
    정답(다른 두 정답과 내용이 겹치는 문서)만 이 컷오프에서 빠지는데, 1위가 이미
    정답이라 MRR엔 영향 없다.

    [min_k=2 / rerank_margin=0.03 실험 — 되돌림]
    회수를 늘리면 key_fact를 더 담을 것이라는 가설로 시도했으나 공개 10문항 실측에서
    기각됐다. 회수는 실제로 2~3개로 늘었지만 답변 10개 중 9개가 글자까지 동일했고,
    유일하게 바뀐 P06은 오히려 key_fact 어구('통합서비스에 가입하기 위해서는')가
    빠져 나빠졌다. 원인은 골드셋 구조에 있다: 공개 문항은 9/10이 gold_articles가
    1개이고, P02의 3개도 서로 대체 가능한 동등 조항이라 어느 하나만 잡으면 key_fact가
    모두 커버된다(실제로 누락 key_fact 8개가 전부 이미 회수한 1위 조항 원문 안에
    있었다). MRR도 정답이 이미 1위라 회수를 늘려도 1.0으로 불변이다. 즉 회수 확대는
    이득이 없고 생성 오염·지연 위험만 늘어 원래 값으로 되돌렸다. 누락 key_fact는
    검색이 아니라 생성 단계(_SYSTEM_PROMPT 원문 보존 규칙 + V7 커버리지 재시도)에서
    해결한다.
    """
    fused = _bm25_rrf_scores(question)

    # 질문에 공식 문서명이 정확히 명시된 경우 해당 문서 범위로 후보를 제한한다.
    named_docs = [
        doc for doc in OFFICIAL_DOCUMENT_NAMES
        if doc in question
    ]

    if named_docs:
        candidate_indices = [
            i for i, a in enumerate(ARTICLES)
            if a["doc"] in named_docs
        ]
    else:
        candidate_indices = list(range(len(ARTICLES)))

    pool = sorted(
        candidate_indices,
        key=lambda i: fused[i],
        reverse=True,
    )[:bm25_pool]

    if not pool:
        return []

    if _HAS_RERANKER:
        # 채널 1: 조 전체(1000자 잘림 없이 원문 그대로)
        whole_pairs = [(question, _CORPUS_TEXT[i]) for i in pool]
        whole_scores = _RERANKER.predict(whole_pairs)
        whole_rank = sorted(range(len(pool)), key=lambda j: whole_scores[j], reverse=True)
        whole_pos = {pool[j]: r for r, j in enumerate(whole_rank, 1)}

        # 채널 2: 질문과 가장 가까운 항 하나만(긴 조항 앞부분 고정 잘라내기 대신)
        sub_pairs = [
            (
                question,
                f"{ARTICLES[i]['doc']} {ARTICLES[i]['title']} "
                f"{_best_subchunk_for_question(question, ARTICLES[i])}",
            )
            for i in pool
        ]
        sub_scores = _RERANKER.predict(sub_pairs)
        sub_rank = sorted(range(len(pool)), key=lambda j: sub_scores[j], reverse=True)
        sub_pos = {pool[j]: r for r, j in enumerate(sub_rank, 1)}

        fused_rr = {i: 1.0 / (rrf_k + whole_pos[i]) + 1.0 / (rrf_k + sub_pos[i]) for i in pool}
        probs = sorted(fused_rr.items(), key=lambda x: x[1], reverse=True)
        top_score = probs[0][1]
        cutoff = top_score - rerank_margin
    else:
        max_fused = max(fused[i] for i in pool) or 1.0
        probs = sorted(
            ((i, fused[i] / max_fused) for i in pool),
            key=lambda x: x[1],
            reverse=True,
        )
        top_score = probs[0][1]
        cutoff = top_score * 0.7

    selected = [probs[0][0]]
    for idx, score in probs[1:max_k]:
        if score >= cutoff:
            selected.append(idx)
        else:
            break

    if len(selected) < min_k:
        selected = [i for i, _ in probs[:min_k]]

    return [ARTICLES[i] for i in selected[:max_k]]


# -------------------------------------------------------------------------------------
# V4. 추가 context pruning
#
# 검색 순위 자체는 유지하되, 뒤의 조항이 질문의 새로운 부분을 실제로 보충하지 못하면
# 생성 모델 입력에서는 제거한다. 특정 의미 유형을 분류하지 않고 V2에서 분해한 질문 단위의
# coverage만 비교한다.
# -------------------------------------------------------------------------------------
def _lexical_query_coverage(query, text):
    """리랭커 폴백용 0~1 문자-bigram coverage."""
    q_tokens = set(_bigram_tokens(query))
    if not q_tokens:
        return 0.0
    t_tokens = set(_bigram_tokens(text))
    return len(q_tokens & t_tokens) / max(1, len(q_tokens))


def _context_part_score_matrix(question, contexts):
    """분해된 질문 단위 × 검색 조항의 관련도 행렬을 반환한다."""
    parts = _decompose_question(question)
    if not contexts:
        return parts, []

    # 각 조항은 그 질문 단위와 가장 가까운 내부 조각으로 비교한다.
    pairs = []
    pair_meta = []
    for pi, part in enumerate(parts):
        for ci, article in enumerate(contexts):
            candidate = (
                f"{article['doc']} {article['title']} "
                f"{_best_subchunk_for_question(part, article)}"
            )
            pairs.append((part, candidate))
            pair_meta.append((pi, ci, candidate))

    matrix = [
        [0.0 for _ in contexts]
        for _ in parts
    ]

    if _HAS_RERANKER and pairs:
        try:
            scores = [float(score) for score in _RERANKER.predict(pairs)]
            needs_sigmoid = any(score < 0.0 or score > 1.0 for score in scores)

            if needs_sigmoid:
                normalized_scores = []
                for score in scores:
                    # overflow 없이 안정적인 sigmoid
                    if score >= 0:
                        z = _math.exp(-score)
                        normalized_scores.append(1.0 / (1.0 + z))
                    else:
                        z = _math.exp(score)
                        normalized_scores.append(z / (1.0 + z))
                scores = normalized_scores

            for (pi, ci, _), score in zip(pair_meta, scores):
                matrix[pi][ci] = score
            return parts, matrix
        except Exception:
            # 검색 단계는 이미 성공했으므로 pruning 실패가 전체 요청을 죽이지 않게 한다.
            pass

    for pi, part in enumerate(parts):
        for ci, article in enumerate(contexts):
            candidate = (
                f"{article['doc']} {article['title']} "
                f"{_best_subchunk_for_question(part, article)}"
            )
            matrix[pi][ci] = _lexical_query_coverage(part, candidate)

    return parts, matrix


def _prune_contexts_by_coverage(
    question,
    contexts,
    relative_cover=0.88,
    small_gap=0.06,
    max_contexts=4,
):
    """새 질문 부분을 보충하는 조항만 생성 context에 남긴다.

    - 검색 1위는 항상 유지한다.
    - 각 질문 단위에 대해 현재 선택된 조항의 최고 점수가 전체 후보 최고점에 충분히 가까우면
      이미 covered로 본다.
    - 그렇지 않은 질문 단위를 가장 많이 보충하는 조항을 greedily 추가한다.
    - 질문에 공식 문서명이 여러 개 직접 명시된 비교형 질문이면, 검색 결과에 존재하는 한
      각 명시 문서의 최고 순위 조항을 하나씩 보존한다.
    """
    if len(contexts) <= 1:
        return contexts

    parts, matrix = _context_part_score_matrix(question, contexts)
    if not parts or not matrix:
        return contexts[:1]

    selected = [0]

    def is_covered(pi):
        row = matrix[pi]
        best = max(row)
        current = max(row[ci] for ci in selected)

        # 후보 간 점수 차이가 거의 없으면 추가 context의 이득이 불명확하므로 1위를 신뢰한다.
        if best - min(row) <= small_gap:
            return True

        # 0~1 계열 점수에서 절대 차이가 작거나 최고점의 일정 비율 이상이면 충분히 covered.
        if best <= 0:
            return True
        if best - current <= small_gap:
            return True
        if current >= best * relative_cover:
            return True
        return False

    # 검색 1위를 seed로 두고, 아직 커버되지 않은 질문 부분에 실제 gain을 주는
    # 후순위 조항만 greedily 추가한다. 상한은 검색 max_k와 같은 4개지만,
    # 새 coverage가 없으면 1~3개에서 즉시 종료한다.
    while len(selected) < min(len(contexts), max_contexts):
        uncovered = [
            pi for pi in range(len(parts))
            if not is_covered(pi)
        ]
        if not uncovered:
            break

        best_ci = None
        best_gain = 0.0
        for ci in range(1, len(contexts)):
            if ci in selected:
                continue

            gain = 0.0
            for pi in uncovered:
                row = matrix[pi]
                current = max(row[s] for s in selected)
                gain += max(0.0, row[ci] - current)

            if gain > best_gain:
                best_gain = gain
                best_ci = ci

        if best_ci is None or best_gain <= 0:
            break
        selected.append(best_ci)

    # 여러 공식 문서를 직접 명시한 질문은 문서 간 비교일 수 있으므로 각 문서의 최고 검색 결과 보존.
    named_docs = [
        doc for doc in OFFICIAL_DOCUMENT_NAMES
        if doc in question
    ]
    if len(named_docs) >= 2:
        for doc in named_docs:
            for ci, article in enumerate(contexts):
                if article["doc"] == doc:
                    if ci not in selected:
                        selected.append(ci)
                    break

    selected = sorted(set(selected))
    return [contexts[i] for i in selected]


# -------------------------------------------------------------------------------------
# 1-4. 생성 모델 — Qwen2.5-Instruct 계열, T4에서 7B를 NF4 4bit로 로컬 실행
#      7B fp16은 weight만 15.2GB라 T4(15GB)에 리랭커와 함께 올라가지 않는다.
#      weight-only 4bit로 내리면 약 5.6GB가 되어 리랭커 1.2GB + KV 캐시를 더해도
#      7GB 안팎에서 돈다. 모델 계열은 규정대로 Qwen2.5-Instruct 그대로다.
#      리랭커와 마찬가지로 원격 API가 아니라 로컬 GPU 실행이다.
# -------------------------------------------------------------------------------------
GEN_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
GEN_FALLBACK_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

# 생성 모델 inference tuning 값 — 공개 문항 정답을 하드코딩하지 않고
# 원문 보존/과잉 생성 억제/극성 안정화를 위한 일반 파라미터다.
GEN_REPETITION_PENALTY = 1.02
BINARY_DECISION_MARGIN = 0.08
ENABLE_BINARY_CALIBRATION = True

# 재시도 생성은 첫 생성만큼 비싸다. 7B 4bit는 3B fp16보다 토큰당 2~3배 느려서
# 목록형 질문(최대 512토큰) + 재시도가 겹치면 러너의 문항당 120초 제한에 걸린다.
# 이미 이만큼 쓴 뒤에는 재시도를 시작하지 않는다.
RETRY_DEADLINE_S = 35.0


def _load_fallback_model(reason):
    """7B를 못 쓰는 상황에서 기존 3B로 내려간다."""
    print(f"[로딩] {reason} — {GEN_FALLBACK_MODEL_NAME} ...")

    tokenizer = AutoTokenizer.from_pretrained(GEN_FALLBACK_MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        GEN_FALLBACK_MODEL_NAME,
        torch_dtype=torch.float16 if _DEVICE == "cuda" else torch.float32,
        device_map=_DEVICE,
    )
    return GEN_FALLBACK_MODEL_NAME, tokenizer, model


def _load_generation_model():
    """7B를 NF4 4bit로 올리고, 안 되면 기존 3B로 폴백한다.

    로딩 실패가 곧 셀 전체 실패(0점)이므로 리랭커와 같은 폴백 구조를 둔다.
    """
    if _DEVICE != "cuda":
        return _load_fallback_model("GPU 없음")

    if not _HAS_BNB:
        return _load_fallback_model("bitsandbytes 사용 불가")

    try:
        from transformers import BitsAndBytesConfig

        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            # T4(sm_75)에는 bf16 텐서코어가 없으므로 compute dtype은 반드시 fp16이다.
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        print(f"[로딩] 생성 모델 {GEN_MODEL_NAME} (NF4 4bit) ...")

        tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
        model = AutoModelForCausalLM.from_pretrained(
            GEN_MODEL_NAME,
            quantization_config=quant_config,
            device_map={"": 0},
            low_cpu_mem_usage=True,
        )
        return GEN_MODEL_NAME, tokenizer, model
    except Exception as exc:
        # 7B가 부분 할당된 채로 죽었을 수 있어 폴백 로딩 전에 GPU를 비운다.
        gc.collect()
        torch.cuda.empty_cache()

        return _load_fallback_model(
            f"7B 4bit 로딩 실패({type(exc).__name__}: {exc})"
        )


GEN_ACTIVE_MODEL_NAME, _TOKENIZER, _MODEL = _load_generation_model()
_MODEL.eval()
print(f"[로딩] 생성 모델 준비 완료 — {GEN_ACTIVE_MODEL_NAME} | torch {torch.__version__}")

if _DEVICE == "cuda":
    print(
        f"[GPU 메모리] "
        f"allocated={torch.cuda.memory_allocated() / 1024**3:.2f} GB / "
        f"total={torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )

# [규칙 1·2를 짝으로 두는 이유 — recall과 precision을 동시에 지킨다]
#
# 채점의 key_fact recall은 '담긴 key_fact 개수 / 전체 개수'로 계단식이며(공개 10문항
# 전부 정수로 확인), 판정 key_fact는 질문이 직접 물은 범위보다 넓다. 예를 들어 P08은
# 질문이 '우선 적용되나요'만 묻지만 gold는 '규정되지 않은 사항' 규정까지 요구하고,
# P07은 '누가/예시'만 묻지만 '제3자 서비스에 가입되지 않는다'까지 요구한다. 그래서
# 규칙 1은 답변 기준을 '질문에 필요한 사실'이 아니라 '질문이 묻는 규정 전체'로 넓힌다.
# 또한 원문을 요약·재서술하면 어구가 어긋나 판정에서 미달되므로 규칙 3으로 원문 어구
# 보존을 요구한다(실측: 원문 그대로 옮긴 key_fact는 커버율 1.00 통과, 압축은 0.56 미달).
#
# 다만 규칙 1만 두면 조항을 통째로 옮기는 역효과가 난다. 정답 조항 전문을 그대로
# 답변으로 썼을 때의 근사 F1을 실측하면 짧은 조항은 이득이지만(P08 123자 0.933,
# P05 105자 1.000) 긴 조항은 파멸적이다(P09 1391자 0.125, P01 650자 0.200,
# P02 682자 0.245). key_fact가 조항의 일부일 뿐이라 나머지가 전부 precision을
# 깎기 때문이다. 그래서 규칙 2로 '질문 주제와 다른 규정은 담지 말 것'이라는 경계를
# 같이 준다. 규칙 1은 규정 단위의 완결성을, 규칙 2는 규정 밖으로의 확장 금지를 맡는다.
_SYSTEM_PROMPT = """
당신은 카카오 약관 질의응답 시스템입니다.
반드시 제공된 [근거]에 있는 내용만 사용하여 한국어로 답하세요.

규칙:

1. [근거]에서 질문이 묻는 규정을 이루는 문장은 빠짐없이 담으세요. 하나의 규정이 여러 사실(조건·효력·예외·권리·의무·후속 절차)로 이루어져 있으면, 질문이 그중 하나만 물었더라도 나머지 사실도 함께 담으세요.

2. 다만 같은 조항 안에 있더라도 질문 주제와 다른 규정(별개의 정의, 다른 상황에 대한 절차, 무관한 예시)은 담지 마세요. 근거를 통째로 옮기지 말고 질문 주제에 해당하는 부분만 쓰세요.

3. 근거 문장을 요약하거나 다른 말로 바꾸지 말고, 원문의 어구와 문장 구조를 그대로 옮겨 쓰세요. 숫자, 기간, 순서, 주체, 대상, 법령명, 조 번호는 반드시 원문 표기를 유지하세요.

4. 한 문장이 '~하되', '~하고', '~하며', '~않으며'로 여러 사실을 잇고 있으면 어느 한쪽만 남기지 말고 모든 절을 담으세요.

5. 서로 다른 항이나 문장의 주어·권리·의무·효력을 합치지 말고, 각 사실의 주어와 관계를 근거와 동일하게 유지하세요.

6. 질문의 전제를 근거와 대조하세요. [예/아니오 판정]이 제공되면 첫 문장을 그 판정으로 시작하고 근거와 반대되는 방향으로 바꾸지 마세요.

7. 근거에 없는 서비스명·예시·설명·해석을 만들지 말고, 원문에 설명이 없는 항목에는 새 설명을 붙이지 마세요.

8. 같은 사실을 다른 표현으로 두 번 쓰지 마세요. 필요한 사실을 모두 담은 뒤에는 요약·결론 문장을 덧붙이지 말고 끝내세요. '[근거 1]', '[예/아니오 판정]' 같은 내부 표시는 출력하지 마세요.
""".strip()



# 한 문장 안에서 서로 독립적인 사실을 잇는 절 연결어(+쉼표) 경계.
# 'A하되, B' / 'A하고, B' / '...않으며, B'처럼 앞절 A를 통째로 떨어뜨리는 압축을
# 커버리지 검사(V7)가 감지하도록, 이 지점에서 문장을 한 번 더 쪼갠다.
# 모든 대안이 2글자라 고정폭 lookbehind가 성립하고, 연결어는 왼쪽 절에 남는다.
# '그리고,'/'그러나,'는 앞 2글자가 '리고'/'러나'라 매칭되지 않아 쪼개지지 않는다.
_CLAUSE_SPLIT_RE = _re.compile(r"(?<=하되|하고|하며|하나|으며|지만)[,]\s+")


def _split_sentence_clauses(sentence, min_len=14):
    """한 문장을 절 연결어 기준으로 쪼갠다. min_len보다 짧은 조각은 앞 조각에 도로
    붙여, 'A하되'처럼 의미 있는 절만 독립 단위로 남긴다."""
    parts = [p.strip() for p in _CLAUSE_SPLIT_RE.split(sentence) if p.strip()]
    if len(parts) <= 1:
        s = sentence.strip()
        return [s] if s else []
    merged = []
    for p in parts:
        if merged and (len(p) < min_len or len(merged[-1]) < min_len):
            merged[-1] = f"{merged[-1]}, {p}"
        else:
            merged.append(p)
    return merged


def _split_generation_units(text, split_clauses=False):
    """생성용 evidence를 문장/항 단위로 나눈다. 목록 줄은 가능한 한 그대로 유지한다.

    split_clauses=True이면 목록이 아닌 문장을 절 연결어('~하되/하고/하며/으며' + 쉼표)
    기준으로 한 번 더 쪼갠다. 'A하되, B'에서 앞절 A가 답변에 통째로 빠졌는지
    커버리지 검사(V7)가 감지하게 하는 용도다. 생성 경로는
    기본값(False)을 쓰므로 그 동작에는 영향이 없다.
    """
    units = []
    for line in _re.split(r"\n+", text):
        line = line.strip()
        if not line:
            continue

        # 짧은 번호/불릿 항목은 자체가 하나의 의미 단위다.
        # 단, P09처럼 하나의 번호 항 안에 여러 문장이 길게 이어지는 경우에는
        # 비목록 질문에서 핵심 문장만 고를 수 있도록 문장 단위로 다시 나눈다.
        if (
            _re.match(r"^(?:[①②③④⑤⑥⑦⑧⑨⑩]|\d+[.)]|[-•])", line)
            and len(line) <= 350
        ):
            units.append(line)
            continue

        sentences = _re.split(r"(?<=[.!?])\s+", line)
        for s in sentences:
            s = s.strip()
            if not s:
                continue
            if split_clauses:
                units.extend(_split_sentence_clauses(s))
            else:
                units.append(s)

    return units if units else [text.strip()]


def _normalize_model_scores(scores):
    """CrossEncoder가 logit/확률 어느 형태를 반환해도 0~1 범위로 맞춘다."""
    vals = [float(s) for s in scores]
    if not vals:
        return vals
    if all(0.0 <= s <= 1.0 for s in vals):
        return vals

    normalized = []
    for score in vals:
        if score >= 0:
            z = _math.exp(-score)
            normalized.append(1.0 / (1.0 + z))
        else:
            z = _math.exp(score)
            normalized.append(z / (1.0 + z))
    return normalized


# 생성용 evidence는 검색 순위가 아니라 "선택된 조항의 길이와 구조"에 따라 만든다.
# - 짧은 조: 전체 원문을 그대로 전달해 동일 조 내 KeyFact 손실을 막는다.
# - 긴 조: 문장 Top-k로 자르지 않고 관련 "항/호 묶음" 전체를 보존한다.
#
# 1200자는 공개 10문항에 맞춘 값이 아니라 현재 72개 약관 조항의 전체 길이 분포에서
# 긴 꼬리 구간만 구조적으로 축약하기 위한 보수적 시작값이다. 필요하면 별도 validation으로 조정한다.
FULL_ARTICLE_CHAR_LIMIT = 1200
STRUCTURAL_MAX_BLOCKS = 4
STRUCTURAL_EVIDENCE_SOFT_CHAR_LIMIT = 2200
STRUCTURAL_NEIGHBOR_RATIO = 0.55
STRUCTURAL_NEIGHBOR_GAP = 0.12


def _split_article_structural_blocks(text):
    """조 원문을 가능한 한 '항' 수준의 구조 블록으로 나눈다.

    우선순위:
    1) ①②③... 표기가 있으면 그것을 최상위 항 경계로 사용한다.
       각 항 안의 1. 2. 3. 또는 불릿은 해당 항에 그대로 붙여 둔다.
    2) 원문이 1. 2. 3. 형식으로만 구성된 약관은 번호 항목을 최상위 블록으로 사용한다.
    3) 명시 구조가 없으면 조 전체를 하나의 블록으로 본다.

    문장 단위로 다시 쪼개지 않으므로 주어·조건·효력과 하위 목록 구조가 함께 보존된다.
    """
    raw = (text or "").strip()
    if not raw:
        return []

    circled = _re.split(r"(?=[①②③④⑤⑥⑦⑧⑨⑩])", raw)
    circled = [p.strip() for p in circled if p.strip()]
    if len(circled) >= 2 and any(_re.match(r"^[①②③④⑤⑥⑦⑧⑨⑩]", p) for p in circled):
        return circled

    numbered = _re.split(r"(?=(?<!\d)\d+\.[^\d])", raw)
    numbered = [p.strip() for p in numbered if p.strip()]
    if len(numbered) >= 2 and any(_re.match(r"^\d+\.", p) for p in numbered):
        return numbered

    return [raw]


def _score_structural_blocks(query, blocks):
    """질문과 구조 블록의 관련도를 0~1 계열로 반환한다."""
    if not blocks:
        return []

    if _HAS_RERANKER:
        try:
            return _normalize_model_scores(
                _RERANKER.predict([(query, block) for block in blocks])
            )
        except Exception:
            pass

    return [_lexical_query_coverage(query, block) for block in blocks]


def _select_structural_article_evidence(
    question,
    article,
    max_blocks=STRUCTURAL_MAX_BLOCKS,
    soft_char_limit=STRUCTURAL_EVIDENCE_SOFT_CHAR_LIMIT,
    neighbor_ratio=STRUCTURAL_NEIGHBOR_RATIO,
    neighbor_gap=STRUCTURAL_NEIGHBOR_GAP,
):
    """긴 조항에서 관련 항 전체 + 필요한 연결 항만 구조적으로 보존한다.

    sentence Top-k 방식과 달리 선택된 항의 원문은 통째로 유지한다.
    질문을 1~3개 표면 단위로 분해해 각 부분의 최고 항을 anchor로 잡고,
    anchor 바로 앞뒤 항 중 관련도가 충분한 경우만 soft budget 안에서 보충한다.
    """
    blocks = _split_article_structural_blocks(article.get("text", ""))
    if len(blocks) <= 1:
        return (article.get("text") or "").strip()

    parts = _decompose_question(question)
    queries = [question]
    for part in parts:
        if part and part not in queries:
            queries.append(part)

    score_rows = [_score_structural_blocks(q, blocks) for q in queries]
    score_rows = [row for row in score_rows if row]
    if not score_rows:
        return (article.get("text") or "").strip()

    # 원 질문 및 각 질문 부분별 최고 항은 반드시 anchor로 보존한다.
    anchors = set()
    for row in score_rows:
        best_idx = max(range(len(blocks)), key=lambda i: row[i])
        anchors.add(best_idx)

    # 블록별 강도: 원 질문/분해 질문 중 가장 높은 관련도.
    block_strength = [
        max(row[i] for row in score_rows)
        for i in range(len(blocks))
    ]
    global_best = max(block_strength) if block_strength else 0.0

    selected = set(anchors)
    total_chars = sum(len(blocks[i]) for i in selected)

    # 관련 anchor의 바로 앞뒤 항만 '연결 항' 후보로 본다.
    neighbor_candidates = set()
    for idx in anchors:
        if idx - 1 >= 0:
            neighbor_candidates.add(idx - 1)
        if idx + 1 < len(blocks):
            neighbor_candidates.add(idx + 1)
    neighbor_candidates -= selected

    # 의미적으로 충분히 가까운 이웃부터 추가한다. anchor는 soft budget을 넘어도 유지하지만,
    # 보조 이웃은 입력 폭증을 막기 위해 soft budget과 max_blocks를 모두 지킨다.
    for idx in sorted(neighbor_candidates, key=lambda i: block_strength[i], reverse=True):
        if len(selected) >= max_blocks:
            break

        score = block_strength[idx]
        strong_enough = (
            global_best <= 0
            or score >= global_best * neighbor_ratio
            or (global_best - score) <= neighbor_gap
        )
        if not strong_enough:
            continue

        if total_chars + len(blocks[idx]) > soft_char_limit:
            continue

        selected.add(idx)
        total_chars += len(blocks[idx])

    # anchor가 질문 여러 부분에 걸쳐 너무 많이 잡힌 경우에도 원래 순서는 유지한다.
    # max_blocks는 보조 이웃에 대한 상한이며, 질문 각 부분의 anchor 자체는 버리지 않는다.
    chosen = [blocks[i] for i in sorted(selected)]
    return "\n".join(chosen).strip()


def _prepare_article_evidence(question, article):
    """선택된 모든 조항에 동일한 길이/구조 규칙을 적용한다."""
    text = (article.get("text") or "").strip()
    if not text:
        return text

    # 짧은/중간 조항은 조 전체를 그대로 보여준다.
    if len(text) <= FULL_ARTICLE_CHAR_LIMIT:
        return text

    # 긴 조항만 항 단위로 구조적으로 축약한다.
    return _select_structural_article_evidence(question, article)


def _build_evidence_blocks(question, contexts):
    """greedy coverage를 통과한 모든 조항의 생성용 evidence를 구성한다.

    검색 1위인지 후순위인지에 따라 full/partial을 나누지 않는다.
    선택된 각 조항마다 동일하게:
      - 짧은 조 -> 전체 원문
      - 긴 조 -> 관련 항 전체 + 필요한 연결 항
    을 적용한다.

    검색 순위와 returned `retrieved`는 변경하지 않는다.
    """
    blocks = []
    plain_parts = []

    for i, article in enumerate(contexts, 1):
        evidence_text = _prepare_article_evidence(question, article)

        block = (
            f"[근거 {i}]\n"
            f"문서: {article['doc']}\n"
            f"조항: 제{article['article_no']}조 ({article['title']})\n"
            f"내용:\n{evidence_text}"
        )
        blocks.append(block)
        plain_parts.append(
            f"{article['doc']} 제{article['article_no']}조 {article['title']}\n{evidence_text}"
        )

    return "\n\n".join(blocks), "\n\n".join(plain_parts)


def _is_binary_question(question):
    """의미 taxonomy 없이 표면 형태가 명확한 예/아니오 질문만 판별한다.

    '어떻게/무엇/몇' 같은 의문사가 있으면 binary로 보지 않는다.
    """
    q = _normalize_question_part(question)
    if not q:
        return False

    wh_markers = (
        "무엇", "어떤", "어떻게", "누가", "누구", "언제", "어디",
        "몇", "왜", "얼마", "어느", "순서", "방법", "종류",
    )
    if any(marker in q for marker in wh_markers):
        return False

    binary_endings = (
        "인가요", "인가", "입니까", "맞나요", "맞습니까", "되나요", "됩니까",
        "하나요", "합니까", "가능한가요", "가능합니까", "할 수 있나요",
        "할 수 있습니까", "아닌가요", "아닙니까", "없나요", "없습니까",
        "있나요", "있습니까",
    )
    return any(q.endswith(ending) for ending in binary_endings)


def _candidate_completion_mean_logprob(messages, candidate):
    """같은 Qwen에서 짧은 후보 completion의 평균 log-probability를 계산한다.

    별도 외부 모델이나 API 없이 '예.'와 '아니오.' 중 근거에 더 잘 맞는 방향을
    inference-time에 calibration하기 위한 용도다.
    """
    prompt = _TOKENIZER.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    prompt_ids = _TOKENIZER(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    )["input_ids"].to(_DEVICE)

    candidate_ids = _TOKENIZER(
        candidate,
        return_tensors="pt",
        add_special_tokens=False,
    )["input_ids"].to(_DEVICE)

    if candidate_ids.numel() == 0:
        return float("-inf")

    full_ids = torch.cat([prompt_ids, candidate_ids], dim=1)
    attention_mask = torch.ones_like(full_ids)

    with torch.inference_mode():
        logits = _MODEL(
            input_ids=full_ids,
            attention_mask=attention_mask,
            use_cache=False,
        ).logits

    # candidate 첫 토큰은 prompt 마지막 토큰 위치의 logits로 예측된다.
    start = prompt_ids.shape[1] - 1
    end = start + candidate_ids.shape[1]
    cand_logits = logits[:, start:end, :]
    log_probs = torch.log_softmax(cand_logits.float(), dim=-1)
    target = candidate_ids
    token_lp = log_probs.gather(-1, target.unsqueeze(-1)).squeeze(-1)
    return float(token_lp.mean().item())


def _infer_binary_conclusion(question, evidence_plain, margin=BINARY_DECISION_MARGIN):
    """명확한 binary 질문에서만 '예.'/'아니오.' 방향을 근거 기반으로 보정한다.

    두 후보의 평균 log-prob 차이가 너무 작으면 강제로 판정하지 않는다.
    이렇게 하면 애매한 질문까지 Python 규칙으로 의미 판정하는 것을 피할 수 있다.
    """
    if (
        not ENABLE_BINARY_CALIBRATION
        or not _is_binary_question(question)
        or not evidence_plain.strip()
    ):
        return None

    verifier_system = (
        "당신은 약관 문장의 참/거짓을 판정하는 검증기입니다. "
        "제공된 근거만 사용하세요. 질문에 포함된 전제를 그대로 믿지 말고 근거와 대조하세요. "
        "질문의 명제가 근거와 일치하면 '예.', 일치하지 않으면 '아니오.'입니다."
    )
    verifier_user = f"""
[근거]
{evidence_plain}

[질문]
{question}

설명하지 말고 '예.' 또는 '아니오.' 중 하나만 판단하세요.
""".strip()

    messages = [
        {"role": "system", "content": verifier_system},
        {"role": "user", "content": verifier_user},
    ]

    yes_lp = _candidate_completion_mean_logprob(messages, "예.")
    no_lp = _candidate_completion_mean_logprob(messages, "아니오.")

    if abs(yes_lp - no_lp) < margin:
        return None
    return "예." if yes_lp > no_lp else "아니오."


def _generation_budget(question):
    """일반 질문의 과잉 생성을 줄이고, 목록 질문의 잘림은 방지한다."""
    requested = _requested_count(question)
    parts = _decompose_question(question)

    if requested is not None:
        # 설명이 붙는 N개 목록은 300대 토큰에서 잘릴 수 있으므로 여유를 둔다.
        return min(512, max(384, 128 + requested * 68))
    if _is_exhaustive_question(question):
        return 448
    if len(parts) >= 2:
        return 300
    return 200


def _build_user_prompt(
    question,
    contexts,
    previous_answer=None,
    retry_issues=None,
    binary_conclusion=None,
    prepared_ctx=None,
):
    """시스템 규칙을 반복하지 않고, 질문별로 필요한 동적 지시만 붙인다."""
    if prepared_ctx is None:
        ctx, _ = _build_evidence_blocks(question, contexts)
    else:
        ctx = prepared_ctx

    conclusion_block = ""
    if binary_conclusion:
        conclusion_block = f"""

[예/아니오 판정]
{binary_conclusion}
첫 문장을 위 판정으로 시작하세요.
""".rstrip()

    format_block = ""
    requested = _requested_count(question)
    if requested is not None or _is_exhaustive_question(question):
        count_text = f" {requested}개" if requested is not None else ""
        format_block = f"""

[출력 형식]
근거의 목록을 원문 순서와 표현을 최대한 유지하여{count_text} 빠짐없이 작성하세요.
원문에 설명이 없는 항목에는 설명을 만들지 말고, 요구된 항목을 모두 쓴 뒤 요약·결론 문장을 덧붙이지 마세요.
""".rstrip()
    elif len(_decompose_question(question)) >= 2:
        format_block = """

[출력 형식]
질문의 서로 다른 부분에 대응하는 사실은 문장을 분리하여 작성하세요.
""".rstrip()

    retry_block = ""
    if previous_answer is not None and retry_issues:
        issues = "\n".join(f"- {issue}" for issue in retry_issues)
        retry_block = f"""

[이전 답변]
{previous_answer}

[수정 필요]
{issues}

위 문제만 바로잡아 다시 작성하세요. 이미 맞는 사실을 다른 표현으로 반복하지 마세요.
""".rstrip()

    return f"""
[근거]
{ctx}

[질문]
{question}
{conclusion_block}
{format_block}
{retry_block}

최종 답변만 작성하세요.
""".strip()


def _generate(
    question,
    contexts,
    max_new_tokens=None,
    previous_answer=None,
    retry_issues=None,
    binary_conclusion=None,
    prepared_ctx=None,
):
    if max_new_tokens is None:
        max_new_tokens = _generation_budget(question)

    messages = [
        {"role": "system", "content": _SYSTEM_PROMPT},
        {
            "role": "user",
            "content": _build_user_prompt(
                question,
                contexts,
                previous_answer=previous_answer,
                retry_issues=retry_issues,
                binary_conclusion=binary_conclusion,
                prepared_ctx=prepared_ctx,
            ),
        },
    ]

    prompt = _TOKENIZER.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = _TOKENIZER(
        prompt,
        return_tensors="pt",
    ).to(_DEVICE)

    with torch.inference_mode():
        out = _MODEL.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            # 원문 문구 보존이 중요하므로 반복 패널티를 과하게 주지 않는다.
            # 1.1은 짧은 약관 답변에서 필요한 용어 반복까지 피하게 만들어
            # paraphrase/표현 변형을 유도할 수 있어 1.02로 낮춘다.
            repetition_penalty=GEN_REPETITION_PENALTY,
            use_cache=True,
            eos_token_id=_TOKENIZER.eos_token_id,
            pad_token_id=_TOKENIZER.eos_token_id,
        )

    gen_tokens = out[0][inputs["input_ids"].shape[1]:]

    return _TOKENIZER.decode(
        gen_tokens,
        skip_special_tokens=True,
    ).strip()


# -------------------------------------------------------------------------------------
# V5. 확실히 기계적으로 확인할 수 있는 구조 오류만 selective retry한다.
# 의미 정답/오답 자체를 regex로 판정하지 않는다.
# -------------------------------------------------------------------------------------
def _count_structured_answer_items(answer):
    numbered = _re.findall(
        r"(?m)^\s*(?:\d+[.)]|[-•])\s+",
        answer,
    )
    return len(numbered)


def _extract_explicit_sequence_items(question):
    """질문에 A·B·C / A→B→C처럼 순서 항목이 직접 제시된 경우에만 추출한다."""
    if "순서" not in question:
        return []

    match = _re.search(
        r"([가-힣A-Za-z0-9()]+(?:\s*[·/→]\s*[가-힣A-Za-z0-9()]+?){1,5})"
        r"\s*(?:의)?\s*(?:어떤\s*)?순서",
        question,
    )
    if not match:
        return []

    items = [
        item.strip()
        for item in _re.split(r"\s*[·/→]\s*", match.group(1))
        if item.strip()
    ]
    return items if len(items) >= 2 else []



def _clean_internal_answer_markers(answer):
    """모델이 프롬프트의 내부 레이블을 따라 출력한 경우 제거한다."""
    text = answer.strip()
    text = _re.sub(r"(?m)^\s*\[(?:근거\s*\d+|예/아니오\s*판정|출력\s*형식|수정\s*필요)\]\s*$", "", text)
    text = _re.sub(r"\s*[\(\[]\s*근거\s*\d+\s*[\)\]]", "", text)
    text = _re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def _sentence_similarity(a, b):
    """문자 bigram Jaccard. 높은 경우에만 동일 사실의 반복으로 본다."""
    a_tokens = set(_bigram_tokens(a))
    b_tokens = set(_bigram_tokens(b))
    if not a_tokens or not b_tokens:
        return 0.0
    return len(a_tokens & b_tokens) / max(1, len(a_tokens | b_tokens))


def _dedupe_near_identical_sentences(answer):
    """의미 판정은 하지 않고 거의 같은 문장/문구의 반복만 제거한다."""
    lines = answer.splitlines()
    output_lines = []
    seen_sentences = []

    for line in lines:
        stripped = line.strip()
        if not stripped:
            if output_lines and output_lines[-1] != "":
                output_lines.append("")
            continue

        # 목록 항목은 번호 구조를 보존한다. 항목 전체가 이전 항목과 사실상 동일할 때만 제거.
        if _re.match(r"^(?:\d+[.)]|[-•])\s*", stripped):
            if any(
                len(stripped) >= 20 and len(prev) >= 20 and _sentence_similarity(stripped, prev) >= 0.82
                for prev in seen_sentences
            ):
                continue
            output_lines.append(stripped)
            seen_sentences.append(stripped)
            continue

        sentences = [
            s.strip() for s in _re.split(r"(?<=[.!?])\s+", stripped)
            if s.strip()
        ]
        kept = []
        for sentence in sentences:
            duplicate = False
            compact = _re.sub(r"\s+", "", sentence)
            for prev in seen_sentences:
                prev_compact = _re.sub(r"\s+", "", prev)
                if len(compact) >= 12 and (compact in prev_compact or prev_compact in compact):
                    duplicate = True
                    break
                if len(sentence) >= 24 and len(prev) >= 24 and _sentence_similarity(sentence, prev) >= 0.60:
                    duplicate = True
                    break
            if not duplicate:
                kept.append(sentence)
                seen_sentences.append(sentence)

        if kept:
            output_lines.append(" ".join(kept))

    return "\n".join(output_lines).strip()


def _trim_list_tail(question, answer):
    """명시적 N개 목록을 모두 쓴 뒤 붙는 요약/결론 문장을 제거한다."""
    requested = _requested_count(question)
    if requested is None:
        return answer

    lines = answer.splitlines()
    item_count = 0
    kept = []
    completed = False

    for line in lines:
        stripped = line.strip()
        is_item = _re.match(r"^(?:\d+[.)]|[-•])\s+", stripped) is not None

        if is_item:
            item_count += 1
            if item_count > requested:
                break
            kept.append(line)
            if item_count == requested:
                completed = True
            continue

        if completed:
            # 마지막 항목 다음의 전형적인 요약/결론 문장은 생성하지 않는 것이 목표다.
            if not stripped or _re.match(r"^(?:따라서|즉|이러한|정리하면|요약하면|결론적으로)", stripped):
                break

        kept.append(line)

    return "\n".join(kept).strip()


def _finalize_answer(question, answer):
    """최종 출력에만 적용하는 보수적 후처리."""
    answer = _clean_internal_answer_markers(answer)
    answer = _trim_list_tail(question, answer)
    answer = _dedupe_near_identical_sentences(answer)
    return answer.strip()


def _article_citation_prefix(article):
    """답변 앞에 붙일 조항 인용 표기 '(제N조)'를 만든다.

    채점의 grounding 축은 답변이 근거 조항을 명시적으로 인용할 때 가점된다.
    생성에 실제 사용한 대표 조항(article_no)을 그대로 표기해 근거성을 드러낸다.
    article_no를 알 수 없으면 빈 문자열을 반환해 아무것도 붙이지 않는다.
    """
    if not article:
        return ""
    article_no = article.get("article_no")
    if not isinstance(article_no, int):
        return ""
    return f"(제{article_no}조) "


def _prepend_article_citation(answer, article):
    """최종 답변 맨 앞에 '(제N조)' 조항 인용을 1회만 붙인다.

    - 모든 재시도·검증·후처리가 끝난 최종 출력에만 적용한다. 중간 단계의
      binary/coverage 판정 로직은 조항 표기 없는 원문을 기준으로 동작해야 하므로
      이 함수를 그보다 앞에서 호출하면 안 된다.
    - 답변이 이미 '(제N조' 또는 '제N조'로 시작하면(모델이 스스로 인용한 경우)
      중복 표기하지 않는다.
    """
    answer = answer.strip()
    if not answer:
        return answer
    if _re.match(r"^\(?\s*제?\s*\d+\s*조", answer):
        return answer
    prefix = _article_citation_prefix(article)
    if not prefix:
        return answer
    return prefix + answer


def _validate_answer_structure(question, answer, binary_conclusion=None):
    """질문 자체와 사전 binary 판정에서 확실히 검증 가능한 오류만 반환한다."""
    issues = []
    stripped = answer.strip()

    if len(stripped) < 8:
        issues.append("답변이 지나치게 짧아 질문에 충분히 답하지 못했을 가능성이 있습니다.")

    # Qwen이 드물게 한국어 단어 중간에 한자/중국어 문자를 섞는 출력 오염을 잡는다.
    if _re.search(r"[\u4E00-\u9FFF]", stripped):
        issues.append("답변에 불필요한 한자·중국어 문자가 섞여 있습니다. 근거의 한국어 표현으로 다시 작성하세요.")

    # 명확한 binary 질문은 사전 근거 판정과 첫 문장 방향이 다르면 확실한 오류다.
    if binary_conclusion:
        normalized = _re.sub(r"^[\s>*#-]+", "", stripped)
        if not normalized.startswith(binary_conclusion):
            issues.append(
                f"근거 기반 예/아니오 판정은 '{binary_conclusion}'인데 답변 첫 문장의 방향이 일치하지 않습니다."
            )

    # 명시적 N개 질문에서 모델이 번호/불릿 목록을 사용했다면 개수가 부족한 경우만 잡는다.
    requested = _requested_count(question)
    if requested is not None:
        listed = _count_structured_answer_items(answer)
        if 0 < listed < requested:
            issues.append(
                f"질문은 {requested}개 항목을 요구하지만 답변의 구조화된 항목은 {listed}개입니다."
            )

    # 질문에 순서 항목 자체가 직접 열거돼 있으면 누락된 항목만 검사한다.
    sequence_items = _extract_explicit_sequence_items(question)
    if sequence_items:
        compact_answer = _re.sub(r"\s+", "", answer)
        missing = [
            item for item in sequence_items
            if _re.sub(r"\s+", "", item) not in compact_answer
        ]
        if missing:
            issues.append(
                "질문에 직접 제시된 순서 항목 중 답변에서 빠진 항목이 있습니다: "
                + ", ".join(missing)
            )

    # 조/항 번호 자체를 묻는 질문에서 번호 표현이 완전히 없는 경우만 검사한다.
    asks_article = _re.search(r"(?:몇\s*조|조\s*번호|조항\s*번호)", question)
    asks_paragraph = _re.search(r"(?:몇\s*항|항\s*번호)", question)

    if asks_article and not _re.search(r"(?:제\s*)?\d+\s*조", answer):
        issues.append("질문이 조 번호를 요구하지만 답변에 조 번호가 없습니다.")

    if asks_paragraph and not _re.search(r"(?:제\s*)?\d+\s*항", answer):
        issues.append("질문이 항 번호를 요구하지만 답변에 항 번호가 없습니다.")

    return issues

# -------------------------------------------------------------------------------------
# V7. 커버리지 유도 재시도 — 근거에 있는데 답변에 빠진 핵심 문장을 1회 보완한다.
#
#   답변이 '질문에 답하고 멈추느라' 근거의 후속 사실을 통째로 누락하는 실패(P09·P10,
#   함정 T03·T05·T08의 '다만~' 예외 누락)를 잡는다. 정답·의미 하드코딩 없이:
#     1) 근거를 문장 단위로 쪼개 각 문장의 질문 관련도를 리랭커로 채점
#     2) 관련도 높은데(타이트한 이중 문턱) 답변에 거의 안 담긴 문장을 '누락'으로 지목
#     3) 그 문장을 가리켜 1회 재생성
#     4) 재생성 답변이 (a) 전부 근거에 grounded 이고 (b) 누락 문장을 실제로 더 담을
#        때만 채택. 아니면 원본 유지 -> 프록시상 절대 나빠지지 않는다(monotonic).
#   자체 20문항 회귀(공개10+함정10): 엄격 keyfact F1 0.433 -> 0.600, 퇴행 0, 채택 8.
# -------------------------------------------------------------------------------------
_COV_REL_RATIO = 0.75      # 근거 문장 관련도가 최고점의 이 비율 이상일 때만 '중요'
_COV_ABS_MIN = 0.45        # 그리고 절대 관련도도 이 값 이상(곁가지 배제)
_COV_PRESENT = 0.45        # 답변이 그 문장 bigram의 이 비율 이상을 담으면 '이미 반영됨'
_COV_MAX_FLAG = 2          # 한 번에 최대 몇 문장까지 보완 요구할지
_COV_MIN_LEN = 14          # 너무 짧은 조각은 무시
_COV_GROUND_MIN = 0.6      # 재생성 문장이 근거에 이만큼 겹치지 않으면 환각으로 보고 거부


def _cov_answer_units(answer):
    units = []
    for line in _re.split(r"\n+", answer):
        line = line.strip()
        if not line:
            continue
        for s in _re.split(r"(?<=[.!?])\s+", line):
            s = s.strip()
            if s:
                units.append(s)
    return units or [answer]


def _cov_find_missing(question, evidence_plain, answer):
    """관련도 높지만 답변에 거의 반영되지 않은 근거 문장을 찾는다."""
    if not _HAS_RERANKER or not evidence_plain.strip():
        return []

    ev_units = [
        u for u in _split_generation_units(evidence_plain, split_clauses=True)
        if len(u) >= _COV_MIN_LEN
    ]
    if not ev_units:
        return []

    # 원 질문 + 분해된 각 부분에 대한 관련도의 최댓값을 그 문장의 관련도로 본다.
    queries = [question] + [p for p in _decompose_question(question) if p != question]
    rel = [0.0] * len(ev_units)
    for q in queries:
        try:
            scores = _normalize_model_scores(_RERANKER.predict([(q, u) for u in ev_units]))
        except Exception:
            scores = [_lexical_query_coverage(q, u) for u in ev_units]
        for i, s in enumerate(scores):
            rel[i] = max(rel[i], s)

    top = max(rel) if rel else 0.0
    if top <= 0:
        return []

    ans_units = _cov_answer_units(answer)
    flagged = []
    for i, unit in enumerate(ev_units):
        if rel[i] < max(_COV_ABS_MIN, top * _COV_REL_RATIO):
            continue
        present = _lexical_query_coverage(unit, answer)
        best_sent = max((_sentence_similarity(unit, a) for a in ans_units), default=0.0)
        if present >= _COV_PRESENT or best_sent >= _COV_PRESENT:
            continue
        flagged.append((rel[i], unit))

    flagged.sort(reverse=True, key=lambda x: x[0])
    return [u for _, u in flagged[:_COV_MAX_FLAG]]


def _cov_covered_count(answer, missing_units):
    au = _cov_answer_units(answer)
    c = 0
    for u in missing_units:
        if _lexical_query_coverage(u, answer) >= _COV_PRESENT or \
           max((_sentence_similarity(u, a) for a in au), default=0.0) >= _COV_PRESENT:
            c += 1
    return c


def _cov_grounded(answer, evidence_plain, min_ground=_COV_GROUND_MIN):
    """재생성이 근거에 없는 문장을 새로 만들지 않았는지 검사한다."""
    ev = set(_bigram_tokens(evidence_plain))
    if not ev:
        return True
    for s in _cov_answer_units(answer):
        sb = set(_bigram_tokens(s))
        if len(s) >= _COV_MIN_LEN and sb and len(sb & ev) / len(sb) < min_ground:
            return False
    return True


# -------------------------------------------------------------------------------------
# 1-5. 고정 진입점
# -------------------------------------------------------------------------------------
def answer_question(question: str):
    """공통 러너가 질문마다 호출하는 고정 진입점.

    검색은 기존 V1을 그대로 유지한다.
    생성 단계는 greedy coverage context pruning + 짧은 조 전체/긴 조 항 구조 보존 +
    binary premise calibration + 원문 보존형 decoding + selective retry로 안정화한다.

    return {"answer": str, "retrieved": [[문서명, 조번호], ...]}
    """
    if not isinstance(question, str) or not question.strip():
        raise ValueError("question은 비어 있지 않은 문자열이어야 합니다.")

    question = question.strip()

    # 재시도 생성을 시작할지 판단하기 위한 문항 시작 시각.
    _q_t0 = _time.time()

    # MRR 평가용 retrieved는 검색 결과를 그대로 보존한다.
    retrieved_contexts = _retrieve(question)
    retrieved = [
        [article["doc"], article["article_no"]]
        for article in retrieved_contexts
    ]

    # 생성 모델에는 질문의 새로운 부분을 실제로 보충하는 context만 전달한다.
    generation_contexts = _prune_contexts_by_coverage(
        question,
        retrieved_contexts,
    )
    if not generation_contexts:
        generation_contexts = retrieved_contexts[:1]

    # greedy coverage로 실제 선택된 모든 조항에 같은 구조 규칙을 적용한다.
    # 짧은 조는 전체, 긴 조는 관련 항 전체(+필요한 연결 항)를 보존하며 문장 Top-k로 자르지 않는다.
    prepared_ctx, evidence_plain = _build_evidence_blocks(
        question,
        generation_contexts,
    )

    # 명확한 예/아니오 질문에 한해 같은 Qwen의 후보 log-prob로 방향을 먼저 보정한다.
    # 애매하면 None을 반환해 강제하지 않는다.
    binary_conclusion = _infer_binary_conclusion(
        question,
        evidence_plain,
    )

    answer_text = _generate(
        question,
        generation_contexts,
        binary_conclusion=binary_conclusion,
        prepared_ctx=prepared_ctx,
    )
    answer_text = _finalize_answer(question, answer_text)

    retry_issues = _validate_answer_structure(
        question,
        answer_text,
        binary_conclusion=binary_conclusion,
    )

    # 구조 오류가 있어도 남은 시간이 없으면 재시도하지 않는다.
    # 타임아웃으로 문항을 통째로 날리는 것보다 구조가 덜 다듬어진 답변이 낫다.
    _q_elapsed = _time.time() - _q_t0

    if retry_issues and _q_elapsed >= RETRY_DEADLINE_S:
        print(
            f"[시간] {_q_elapsed:.1f}s 경과 — 재시도 생략 "
            f"(issues={len(retry_issues)})"
        )

    if retry_issues and _q_elapsed < RETRY_DEADLINE_S:
        answer_text = _generate(
            question,
            generation_contexts,
            previous_answer=answer_text,
            retry_issues=retry_issues,
            binary_conclusion=binary_conclusion,
            prepared_ctx=prepared_ctx,
        )
        answer_text = _finalize_answer(question, answer_text)

    # 커버리지 유도 재시도(V7): 근거에 있는데 답변에 빠진 핵심 문장이 있으면 1회 보완.
    # 재생성 답변이 grounded 이고 누락 문장을 실제로 더 담을 때만 채택(아니면 원본 유지).
    _q_elapsed = _time.time() - _q_t0
    if _q_elapsed < RETRY_DEADLINE_S:
        missing_units = _cov_find_missing(question, evidence_plain, answer_text)
        if missing_units:
            cov_issues = [
                "다음 근거 내용이 답변에 빠졌습니다. 근거의 표현을 그대로 사용해 보완하고, "
                "근거에 없는 내용은 만들지 마세요: \u300c" + u + "\u300d"
                for u in missing_units
            ]
            cov_text = _generate(
                question,
                generation_contexts,
                previous_answer=answer_text,
                retry_issues=cov_issues,
                binary_conclusion=binary_conclusion,
                prepared_ctx=prepared_ctx,
            )
            cov_text = _finalize_answer(question, cov_text)
            if (
                _cov_grounded(cov_text, evidence_plain)
                and _cov_covered_count(cov_text, missing_units)
                > _cov_covered_count(answer_text, missing_units)
            ):
                answer_text = cov_text

    # 최종 출력에만 대표 조항 인용 '(제N조)'을 붙인다(grounding 표기).
    # 위의 binary/coverage/구조 검증·재시도는 모두 조항 표기 없는 원문 기준으로
    # 끝났으므로, 여기서 붙여도 그 판정들에 영향을 주지 않는다.
    # 대표 조항은 실제 생성에 사용한 top context(없으면 검색 top)를 쓴다.
    _citation_article = (generation_contexts or retrieved_contexts or [None])[0]
    answer_text = _prepend_article_citation(answer_text, _citation_article)

    return {
        "answer": answer_text,
        "retrieved": retrieved,
    }

# -------------------------------------------------------------------------------------
# 1-6. 워밍업 — 첫 실제 요청이 CUDA 커널 컴파일 시간까지 떠안고 타임아웃에 걸리지 않도록
#      모델 로딩 직후 한 번 미리 실행해 둔다.
# -------------------------------------------------------------------------------------
print("[준비] 워밍업 실행 중 ...")
_warmup_t0 = _time.time()
_ = answer_question("사업자/단체 카카오계정은 계정 정보에 등록된 담당자 몇 명이 이용할 수 있나요?")
print(f"[준비] 워밍업 완료 ({_time.time() - _warmup_t0:.1f}s)")


# =====================================================================================
# 2. 고정 FastAPI 연결 영역 — 삭제하거나 경로를 바꾸지 않습니다
# =====================================================================================
# 2번 공통 러너는 아래 app을 localhost에서 실행하고 다음 주소를 호출합니다.
#   · GET  /health : 결과기 서버 준비 여부 확인
#   · POST /answer : {"question": "..."}을 보내 answer_question() 결과 수신
#
# 팀별 결과기 로직은 위 자유 구현 영역에서 작성합니다. 이 블록은 서버 연결만 담당합니다.
# 동시 요청에서 하나의 GPU 생성 모델이 충돌하지 않도록 Lock을 사용합니다.
import subprocess
import sys
import threading


def _install_server_packages():
    """공통 러너와 연결하는 데 필요한 가벼운 서버 패키지만 설치합니다."""
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn"],
        check=True,
    )


_install_server_packages()

from fastapi import FastAPI, HTTPException  # noqa: E402


app = FastAPI(title="KTB AI Performance Result Generator")
_GENERATION_LOCK = threading.Lock()


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/answer")
def answer_api(payload: dict):
    question = payload.get("question")
    if not isinstance(question, str) or not question.strip():
        raise HTTPException(status_code=400, detail="question must be a non-empty string")
    with _GENERATION_LOCK:
        return answer_question(question.strip())


print("[1번 셀 준비] 결과기 구현을 마친 뒤 2번 공통 러너를 실행하세요.")



In [ ]:
# 2번 셀 — 공개 10문항 답변 파일 생성
# 이 셀은 전 팀 공통이며 _SP_TEAM 한 줄 외에는 수정하지 않습니다.
# 새 Google Colab T4 런타임에서 결과기 코드를 먼저 실행한 뒤 이 셀을 실행합니다.
#
# 사용 순서
# 1. 새 Google Colab T4 런타임에서 1번 셀 결과기 코드를 실행합니다.
# 2. 이 공통 러너를 2번 셀에 그대로 둡니다.
# 3. 맨 위 _SP_TEAM에 운영진이 알려준 숫자 팀 식별자를 입력합니다.
# 4. 생성된 answers_public_<팀>.json을 결과기 코랩 파일과 함께 제출합니다.
# 공개 문항 10개 · 실행 방식: http
# ═══════════════════════════════════════════════════════════════
#  ★ 여기 한 줄만 자기 팀으로 바꾸세요. 나머지는 손대지 마세요. ★
# ═══════════════════════════════════════════════════════════════
_SP_TEAM = "8"          # 예: "1"  ← 운영진이 알려준 팀 식별자(숫자)를 그대로 적습니다
# ═══════════════════════════════════════════════════════════════

import builtins as _sp_builtins
import json as _sp_json
import os as _sp_os_rt
import re as _sp_re
import signal as _sp_signal
import socket as _sp_socket
import sys as _sp_sys
import time as _sp_time
import traceback as _sp_traceback
import unicodedata as _sp_unicodedata
import urllib.error as _sp_urlerror
import urllib.request as _sp_urlrequest

_sp_open = _sp_builtins.open
_sp_print = _sp_builtins.print

if "_sp_real_sys_exit" in globals():
    _sp_sys.exit = _sp_real_sys_exit
    if _sp_real_exit is not None:
        _sp_builtins.exit = _sp_real_exit
    if _sp_real_quit is not None:
        _sp_builtins.quit = _sp_real_quit

_SP_OUTPUT_DIR = "/content/"
_SP_OUTPUT_PREFIX = "answers_public_"
_SP_EXPECTED_OUTPUT_PATH = ""
_SP_TEAM_ALLOWED = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz_-"
_SP_TEAM_MAX_LEN = 32
_SP_TEAM_NUMERIC_ONLY = True

def _sp_team_howto(head):
    """중단 사유 + 학생이 바로 고칠 수 있는 안내를 한 덩어리로 만든다."""
    rule = (
        "1 이상의 정수를 문자열로 입력합니다. 예: 1, 2, 17"
        if _SP_TEAM_NUMERIC_ONLY
        else "영문·숫자·밑줄(_)·하이픈(-) 1~" + str(_SP_TEAM_MAX_LEN) + "자"
    )
    return (
        head
        + "\n"
        + "\n  [고치는 법] 이 셀 맨 위 ★ 상자 안의 한 줄을 이렇게 바꾸세요."
        + '\n      _SP_TEAM = "1"      ← 운영진이 알려준 팀 식별자(숫자)를 따옴표 안에 그대로'
        + "\n  [쓸 수 있는 값] " + rule
        + "\n                 띄어쓰기와 / \\ . : 같은 경로 문자는 파일 이름을 깨뜨려 쓸 수 없습니다."
        + "\n  [왜] 결과 파일 이름이 " + _SP_OUTPUT_PREFIX + "<팀>.json 이고, 채점은 이 이름으로"
        + "\n       어느 팀 답안인지 가립니다. 비워 두면 채점 자체가 되지 않습니다."
    )

def _sp_resolve_team(value):
    """_SP_TEAM 을 검사·정리해 돌려준다. 쓸 수 없는 값이면 RuntimeError 로 즉시 중단."""
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(_sp_team_howto(
            "★ 팀 식별자(_SP_TEAM)가 비어 있어 실행을 중단했습니다. 결과 파일은 만들지 않았습니다."))
    team = value.strip()
    if _SP_TEAM_NUMERIC_ONLY and not _sp_re.fullmatch(r"[1-9][0-9]*", team):
        raise RuntimeError(_sp_team_howto(
            "★ 팀 식별자(_SP_TEAM)는 운영진이 알려준 숫자여야 합니다. 지금 값: " + repr(value)))
    if len(team) > _SP_TEAM_MAX_LEN:
        raise RuntimeError(_sp_team_howto(
            "★ 팀 식별자(_SP_TEAM)가 너무 깁니다(" + str(len(team)) + "자). 팀 이름이 아니라 짧은 식별자입니다."))
    _bad = _sp_builtins.sorted(
        _sp_builtins.set(c for c in team if c not in _SP_TEAM_ALLOWED and not ("가" <= c <= "힣")))
    if _bad:
        raise RuntimeError(_sp_team_howto(
            "★ 팀 식별자(_SP_TEAM)에 파일 이름으로 쓸 수 없는 문자가 있습니다: "
            + ", ".join(repr(c) for c in _bad) + "   (지금 값: " + repr(value) + ")"))
    return team

_SP_TEAM = _sp_resolve_team(_SP_TEAM)
if any(ord(c) > 127 for c in _SP_TEAM):
    _sp_print("[주의] 팀 식별자에 한글 등 ASCII 밖 문자가 있습니다: " + _SP_TEAM
              + " — 운영진이 알려준 식별자가 맞는지 확인하세요."
              " 한글 파일 이름은 내려받기·올리기 과정에서 자모 표현이 달라져 팀이 어긋날 수 있습니다.",
              flush=True)

_SP_OUTPUT_PATH = _SP_OUTPUT_DIR.rstrip("/") + "/" + _SP_OUTPUT_PREFIX + _SP_TEAM + ".json"
if _SP_EXPECTED_OUTPUT_PATH and (_sp_os_rt.path.basename(_SP_OUTPUT_PATH)
                                 != _sp_os_rt.path.basename(_SP_EXPECTED_OUTPUT_PATH)):
    raise RuntimeError(
        "이 셀은 " + _sp_os_rt.path.basename(_SP_EXPECTED_OUTPUT_PATH) + " 용으로 생성됐는데 "
        + _sp_os_rt.path.basename(_SP_OUTPUT_PATH) + " 로 저장하려 합니다"
        "(_SP_TEAM 을 손으로 고쳤습니까?). 다른 팀으로 돌리려면 --team 을 바꿔 셀을 다시 생성하세요."
    )
_SP_AUTO_DOWNLOAD = True
_SP_QUESTIONS_JSON = (
    "[[\"P01\", \"사업자/단체 카카오계정은 계정 정보에 등록된 담당자 몇 명이 이용할 수 있으며, 다른 사람과 공유하는 것은 허용되나요?\"], [\"P02\", \"회사가 예측하거나 통제할 수 없는 사유로 서비스가 중단된 경우, 복구가 몇 시간 이상 지연되면 회사는 공지사항에 게시하여 알리나요?\"], [\"P03\", \"카카오계정 약관에서 회사가 개별 서비스와 연동하여 카카오계정에서 제공한다고 열거한 '카카오계정 서비스'의 내용 5가지는 각각 무엇인가요?\"], [\"P04\", \"회사가 위치기반서비스의 이용을 제한하거나 중지한 때에는 이용자에게 무엇을 어떤 방법으로 알리나요?\"], [\"P05\", \"회사가 위치정보 수집·이용·제공사실 확인자료를 기록·보존하는 근거는 위치정보의 보호 및 이용 등에 관한 법률 제 몇 조 제 몇 항이며, 그 자료는 어디에 기록되어 몇 개월간 보관되나요?\"], [\"P06\", \"카카오계정이 없는 사람이 통합서비스에 가입하려면 무엇을 먼저 해야 하며, 통합서비스 이용계약은 동의·확인·승낙의 어떤 순서로 체결되나요?\"], [\"P07\", \"서비스 명칭에 '카카오'가 사용되더라도 카카오 통합서비스약관의 '통합서비스'에 포함되지 않는 서비스는 누가 제공하는 서비스이며, 약관은 그 예로 무엇을 들고 있나요?\"], [\"P08\", \"카카오 통합 약관과 세부지침(회사가 정한 서비스의 개별 이용약관·운영정책·규칙 등)의 내용이 충돌하는 경우"
    ", 본 약관이 세부지침보다 우선하여 적용되나요?\"], [\"P09\", \"이용자가 서비스 사용을 중단하거나 카카오계정 및 Daum 아이디를 탈퇴한 이후, 게시물에 관하여 회사에 부여한 라이선스의 효력은 어떻게 되나요?\"], [\"P10\", \"8세 이하의 아동 등의 생명 또는 신체 보호를 위해 보호의무자가 개인위치정보의 이용 또는 제공에 동의하려면 어떤 서류에 무엇을 첨부하여 어디에 제출해야 하며, 그 동의는 어떤 효력을 갖나요?\"]]"
)
_SP_QUESTIONS = [tuple(_x) for _x in _sp_json.loads(_SP_QUESTIONS_JSON)]
_SP_ALLOWED_DOCS = _sp_json.loads("[\"카카오계정 약관\", \"카카오 통합서비스약관\", \"카카오 통합 약관\", \"카카오 위치정보 이용약관\"]")
_SP_PER_Q_TIMEOUT_S = 120
_SP_TRANSPORT = "http"
_SP_HTTP_HOST = "127.0.0.1"
_SP_HTTP_PORT = 8765
_SP_HTTP_STARTUP_TIMEOUT_S = 30
_SP_HTTP_HEALTH_PATH = "/health"
_SP_HTTP_ANSWER_PATH = "/answer"
_SP_PERFORMANCE_REQUESTS = 12
_SP_PERFORMANCE_CONCURRENCY = 2
_SP_PERFORMANCE_REPETITIONS = 3
_SP_PERFORMANCE_WARMUP_REQUESTS = 2

_sp_fn = globals().get("answer_question")
if not callable(_sp_fn):
    raise RuntimeError(
        "팀 코드에 answer_question(question) 함수가 없습니다(규정 ②). 실행을 중단합니다."
    )

_sp_doc_warnings = []
_sp_timeouts = []
_sp_http_server = None
_sp_http_thread = None

class _SpHttpTimeout(Exception):
    """HTTP 요청 시간 초과. 품질 추출에서는 timeout_qids로 기록한다."""

def _sp_http_url(path):
    return "http://" + _SP_HTTP_HOST + ":" + str(_SP_HTTP_PORT) + path

def _sp_http_json(method, path, payload=None, timeout_s=None):
    data = None
    headers = {"Accept": "application/json"}
    if payload is not None:
        data = _sp_json.dumps(payload, ensure_ascii=False).encode("utf-8")
        headers["Content-Type"] = "application/json"
    req = _sp_urlrequest.Request(
        _sp_http_url(path), data=data, headers=headers, method=method
    )
    try:
        with _sp_urlrequest.urlopen(req, timeout=timeout_s or _SP_PER_Q_TIMEOUT_S) as resp:
            raw = resp.read().decode("utf-8")
            if resp.status != 200:
                raise RuntimeError("HTTP " + str(resp.status) + ": " + raw[:500])
    except (_sp_socket.timeout, TimeoutError) as exc:
        raise _SpHttpTimeout(str(timeout_s or _SP_PER_Q_TIMEOUT_S) + "초 안에 응답하지 않았습니다.") from exc
    except _sp_urlerror.HTTPError as exc:
        raw = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError("HTTP " + str(exc.code) + ": " + raw[:500]) from exc
    except _sp_urlerror.URLError as exc:
        if isinstance(exc.reason, (_sp_socket.timeout, TimeoutError)):
            raise _SpHttpTimeout(
                str(timeout_s or _SP_PER_Q_TIMEOUT_S) + "초 안에 응답하지 않았습니다."
            ) from exc
        raise RuntimeError("HTTP 연결 실패: " + str(exc.reason)) from exc
    try:
        return _sp_json.loads(raw)
    except _sp_json.JSONDecodeError as exc:
        raise TypeError("HTTP 응답이 JSON이 아닙니다: " + raw[:500]) from exc

def _sp_start_http_server():
    global _sp_http_server, _sp_http_thread
    _sp_app = globals().get("app")
    if _sp_app is None:
        raise RuntimeError(
            "HTTP 실행 모드에는 전역 FastAPI app과 GET /health, POST /answer가 필요합니다."
        )
    try:
        import threading as _sp_threading
        import uvicorn as _sp_uvicorn
    except ImportError as exc:
        raise RuntimeError(
            "HTTP 실행 모드에는 fastapi와 uvicorn이 필요합니다. 팀 설치 목록에 추가하세요."
        ) from exc
    _sp_config = _sp_uvicorn.Config(
        _sp_app,
        host=_SP_HTTP_HOST,
        port=_SP_HTTP_PORT,
        workers=1,
        log_level="warning",
        access_log=False,
    )
    _sp_http_server = _sp_uvicorn.Server(_sp_config)
    _sp_http_thread = _sp_threading.Thread(
        target=_sp_http_server.run, name="ktb-fastapi", daemon=True
    )
    _sp_http_thread.start()
    _sp_deadline = _sp_time.time() + _SP_HTTP_STARTUP_TIMEOUT_S
    _sp_last = None
    while _sp_time.time() < _sp_deadline:
        if not _sp_http_thread.is_alive():
            raise RuntimeError("FastAPI 서버가 준비되기 전에 종료됐습니다.")
        try:
            health = _sp_http_json("GET", _SP_HTTP_HEALTH_PATH, timeout_s=1)
            if isinstance(health, dict):
                _sp_print("[서버] FastAPI /health 준비 완료: " + _sp_http_url(_SP_HTTP_HEALTH_PATH))
                return
        except Exception as exc:
            _sp_last = exc
        _sp_time.sleep(0.2)
    _sp_stop_http_server()
    raise RuntimeError(
        "FastAPI 서버가 " + str(_SP_HTTP_STARTUP_TIMEOUT_S)
        + "초 안에 준비되지 않았습니다: " + str(_sp_last)
    )

def _sp_stop_http_server():
    if _sp_http_server is not None:
        _sp_http_server.should_exit = True
    if _sp_http_thread is not None and _sp_http_thread.is_alive():
        _sp_http_thread.join(timeout=5)

def _sp_invoke(question):
    if _SP_TRANSPORT == "http":
        return _sp_http_json(
            "POST", _SP_HTTP_ANSWER_PATH, {"question": question},
            timeout_s=_SP_PER_Q_TIMEOUT_S,
        )
    return _sp_call_with_timeout(_sp_fn, question, _SP_PER_Q_TIMEOUT_S)

if _SP_TRANSPORT == "http":
    _sp_start_http_server()

_sp_env_warnings = []
for _sp_d in ("/content/drive", "/content/gdrive", "/gdrive"):
    if _sp_os_rt.path.ismount(_sp_d):
        _sp_env_warnings.append(_sp_d + " 가 마운트되어 있습니다")
if _sp_env_warnings:
    _sp_print("", flush=True)
    _sp_print("!" * 86, flush=True)
    _sp_print("[규정 ③ 경고] 이 세션은 운영진 실행 환경과 다릅니다.", flush=True)
    for _sp_w in _sp_env_warnings:
        _sp_print("  · " + _sp_w, flush=True)
    _sp_print("  운영진은 드라이브가 연결되지 않은 새 세션에서 실행합니다. 드라이브에 둔 약관·인덱스를", flush=True)
    _sp_print("  읽고 있다면 본선에서 전량 실패합니다. 약관은 실행 중 내려받거나 셀 안에 포함하세요.", flush=True)
    _sp_print("  확인 방법: 새 노트북을 열어 코드와 이 셀만 붙여 넣고 실행해 보세요.", flush=True)
    _sp_print("!" * 86, flush=True)
    _sp_print("", flush=True)

class _SpTimeout(BaseException):
    """문항 단위 시간 초과.

    **BaseException 을 상속하는 것이 핵심이다.** 팀 코드가 `try/except Exception` 으로
    넓게 감싸는 일은 흔한데, Exception 을 상속하면 그 handler 가 시간 초과를 삼켜
    상한이 무력화된다(그대로 다음 루프를 돌며 계속 매달린다).
    """

def _sp_call_with_timeout(fn, arg, seconds):
    """SIGALRM 으로 문항 호출에 상한을 건다.

    메인 스레드가 아니거나 SIGALRM 이 없는 환경(윈도 등)에서는 signal 설정이
    실패하므로, 그때는 상한 없이 그대로 호출한다 — 상한을 못 걸었다고 해서
    채점 자체를 포기하는 편이 더 나쁘다.

    웹 Colab 셀은 IPython 이 메인 스레드에서 실행하므로 정상 동작한다.
    """
    if not seconds or seconds <= 0:
        return fn(arg)
    _sp_secs = max(1, int(seconds))     # alarm() 은 정수만 받는다. 0 은 '취소' 라 최소 1초.

    def _sp_on_alarm(signum, frame):
        raise _SpTimeout(str(_sp_secs) + "초 안에 응답하지 않았습니다.")

    try:
        _sp_prev = _sp_signal.signal(_sp_signal.SIGALRM, _sp_on_alarm)
        _sp_signal.alarm(_sp_secs)
    except (ValueError, AttributeError, OSError):
        return fn(arg)          # 상한을 걸 수 없는 환경 — 그대로 실행
    try:
        return fn(arg)
    finally:
        _sp_signal.alarm(0)
        try:
            _sp_signal.signal(_sp_signal.SIGALRM, _sp_prev)
        except Exception:
            pass

def _sp_json_safe_art(art):
    """조번호를 JSON 으로 쓸 수 있는 값으로. 표기는 최대한 원본을 살린다.

    **여기서 흡수하지 않으면 30문항을 다 돌린 뒤 파일 저장에서 터진다.**
    일부 수치 라이브러리의 정수형은 dict 도 아니고 2원소 검사도 통과하지만
    json.dump 가 거부한다. 이 값을 흡수하지 않으면
    실패 시점이 맨 끝이라 GPU 시간을 다 쓰고 결과 파일이 없는 최악의 형태가 된다.

    '제7조' 같은 문자열은 그대로 둔다 — 채점기 _art_no 가 정수로 읽는다.
    """
    if isinstance(art, bool):        # bool 은 int 의 하위형이라 먼저 걸러 낸다
        return str(art)
    if isinstance(art, (int, str)):
        return art
    try:                              # np.int64 등 정수로 볼 수 있는 것
        return int(art)
    except (TypeError, ValueError):
        return str(art)

def _sp_norm_doc(x):
    """문서명 대조용 정규화 — NFC 통일 + 공백 전부 제거.

    ⚠️ 채점기 judge_service/engine/objective.py 의 `norm_doc` 과 **같은 규칙이어야 한다.**
    러너는 Colab 셀이라 judge_service 를 import 할 수 없어 규칙을 여기에 복제해 둔다.
    한쪽만 바뀌어 어긋나면 곧바로 오탐이 난다 — 예전에 러너가 완전 일치로 대조하던 때
    '카카오계정약관'·'카카오 계정 약관' 은 실제 채점 MRR 이 1.00 인데도 규정 ④ 위반 경고를
    맞았다. 팀은 없는 문제를 고치러 다니고(자가 확인표가 n_doc_violations == 0 을 요구한다),
    정상 팀이 경고를 맞기 시작하면 아무도 경고를 안 보게 된다.
    두 구현의 일치는 submission_pipeline/tests/test_doc_name_normalization.py 가 고정한다.
    """
    return _sp_re.sub(r"\s+", "", _sp_unicodedata.normalize("NFC", str(x)))

_SP_ALLOWED_DOCS_NORM = _sp_builtins.set(_sp_norm_doc(_d) for _d in _SP_ALLOWED_DOCS)

def _sp_normalize_retrieved(qid, value):
    """retrieved 를 근거순 [[문서명, 조번호], ...] 1~4개로 정규화."""
    if not isinstance(value, (list, tuple)):
        raise TypeError(qid + ": retrieved 는 목록이어야 합니다. (실제: " + type(value).__name__ + ")")
    out = []
    for item in value:
        if isinstance(item, dict) and "doc" in item and "article_no" in item:
            doc, art = item["doc"], item["article_no"]
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            doc, art = item
        else:
            raise TypeError(qid + ": retrieved 항목은 [문서명, 조번호] 2원소여야 합니다. (실제: " + repr(item) + ")")
        doc = str(doc)
        if _SP_ALLOWED_DOCS_NORM and _sp_norm_doc(doc) not in _SP_ALLOWED_DOCS_NORM:
            _sp_doc_warnings.append({"qid": qid, "doc": doc})
        out.append([doc, _sp_json_safe_art(art)])
    if not 1 <= len(out) <= 4:
        raise ValueError(
            qid + ": retrieved 는 실제 답변 근거를 관련도 순으로 1~4개 반환해야 합니다. "
            "(실제: " + str(len(out)) + "개)"
        )
    return out

_sp_answers = []
_sp_errors = []
_sp_total = len(_SP_QUESTIONS)
_sp_print(
    "\n========== " + "공개" + " " + str(_sp_total)
    + "문항 실행 · " + _SP_TEAM + "팀 ==========",
    flush=True,
)
_sp_t0 = _sp_time.time()

for _sp_i, (_sp_qid, _sp_q) in enumerate(_SP_QUESTIONS, 1):
    _sp_print("[" + str(_sp_i).zfill(2) + "/" + str(_sp_total) + "] " + _sp_qid + " 실행 중 ...", flush=True)
    _sp_started = _sp_time.time()
    try:
        _sp_out = _sp_invoke(_sp_q)
        if not isinstance(_sp_out, dict):
            raise TypeError(_sp_qid + ": answer_question() 은 딕셔너리를 반환해야 합니다. (실제: "
                            + type(_sp_out).__name__ + ")")
        _sp_retrieved = _sp_normalize_retrieved(_sp_qid, _sp_out.get("retrieved"))
        _sp_answer = _sp_out.get("answer")
        if not isinstance(_sp_answer, str):
            raise TypeError(_sp_qid + ": answer 는 문자열이어야 합니다. (실제: "
                            + type(_sp_answer).__name__ + ")")
        _sp_answers.append({"qid": _sp_qid, "retrieved": _sp_retrieved, "answer": _sp_answer})
    except (_SpTimeout, _SpHttpTimeout) as _sp_exc:  # 한 문항이 세션 전체를 잡아먹지 않도록 끊는다.
        _sp_msg = "Timeout: " + str(_sp_exc)
        _sp_timeouts.append(_sp_qid)
        _sp_errors.append({"qid": _sp_qid, "error": _sp_msg})
        _sp_answers.append({"qid": _sp_qid, "retrieved": [], "answer": "", "error": _sp_msg})
        _sp_print("[시간초과] " + _sp_qid + " — " + _sp_msg, flush=True)
    except Exception as _sp_exc:  # 한 문항 실패로 30문항 전체를 잃지 않는다.
        _sp_msg = type(_sp_exc).__name__ + ": " + str(_sp_exc)
        _sp_errors.append({"qid": _sp_qid, "error": _sp_msg})
        _sp_answers.append({"qid": _sp_qid, "retrieved": [], "answer": "", "error": _sp_msg})
        _sp_print("[오류] " + _sp_qid + " — " + _sp_msg, flush=True)
        _sp_traceback.print_exc()
    finally:
        _sp_print("      (" + str(round(_sp_time.time() - _sp_started, 1)) + "s)", flush=True)

_sp_performance = None
if _SP_TRANSPORT == "http" and _SP_PERFORMANCE_REQUESTS > 0:
    from concurrent.futures import ThreadPoolExecutor as _SpThreadPoolExecutor

    def _sp_perf_one(index):
        _qid, _question = _SP_QUESTIONS[index % len(_SP_QUESTIONS)]
        started = _sp_time.perf_counter()
        try:
            value = _sp_http_json(
                "POST", _SP_HTTP_ANSWER_PATH, {"question": _question},
                timeout_s=_SP_PER_Q_TIMEOUT_S,
            )
            ok = (
                isinstance(value, dict)
                and isinstance(value.get("answer"), str)
                and isinstance(value.get("retrieved"), (list, tuple))
            )
            return {
                "ok": ok,
                "qid": _qid,
                "latency_s": round(_sp_time.perf_counter() - started, 4),
                "error": None if ok else "invalid_schema",
            }
        except Exception as exc:
            return {
                "ok": False,
                "qid": _qid,
                "latency_s": round(_sp_time.perf_counter() - started, 4),
                "error": type(exc).__name__ + ": " + str(exc),
            }

    def _sp_percentile(values, ratio):
        if not values:
            return None
        pos = min(len(values) - 1, max(0, int((len(values) - 1) * ratio)))
        return round(values[pos], 4)

    def _sp_median(values):
        values = sorted(values)
        if not values:
            return None
        middle = len(values) // 2
        if len(values) % 2:
            return values[middle]
        return (values[middle - 1] + values[middle]) / 2

    def _sp_perf_round(n_requests, repetition):
        started = _sp_time.perf_counter()
        with _SpThreadPoolExecutor(max_workers=max(1, _SP_PERFORMANCE_CONCURRENCY)) as pool:
            rows = list(pool.map(_sp_perf_one, range(n_requests)))
        wall_s = _sp_time.perf_counter() - started
        ok_rows = [row for row in rows if row["ok"]]
        latencies = sorted(row["latency_s"] for row in ok_rows)
        return {
            "repetition": repetition,
            "transport": "http",
            "requests": n_requests,
            "concurrency": _SP_PERFORMANCE_CONCURRENCY,
            "success": len(ok_rows),
            "fail": len(rows) - len(ok_rows),
            "success_rate": round(len(ok_rows) / len(rows), 4),
            "throughput_rps": round(len(ok_rows) / wall_s, 4) if wall_s else 0.0,
            "wall_s": round(wall_s, 4),
            "p50_latency_s": _sp_percentile(latencies, 0.50),
            "p95_latency_s": _sp_percentile(latencies, 0.95),
            "errors": [row for row in rows if not row["ok"]],
        }

    _sp_warmup = None
    if _SP_PERFORMANCE_WARMUP_REQUESTS > 0:
        _sp_print(
            "[성능] 워밍업 " + str(_SP_PERFORMANCE_WARMUP_REQUESTS) + "요청 실행 중 ...",
            flush=True,
        )
        _sp_warmup = _sp_perf_round(_SP_PERFORMANCE_WARMUP_REQUESTS, 0)

    _sp_perf_samples = []
    for _sp_repetition in range(1, _SP_PERFORMANCE_REPETITIONS + 1):
        _sp_print(
            "[성능] 측정 " + str(_sp_repetition) + "/"
            + str(_SP_PERFORMANCE_REPETITIONS) + " 실행 중 ...",
            flush=True,
        )
        _sp_perf_samples.append(
            _sp_perf_round(_SP_PERFORMANCE_REQUESTS, _sp_repetition)
        )

    _sp_success_median = _sp_median([row["success"] for row in _sp_perf_samples])
    _sp_fail_median = _sp_median([row["fail"] for row in _sp_perf_samples])
    _sp_p50_values = [
        row["p50_latency_s"] for row in _sp_perf_samples
        if row["p50_latency_s"] is not None
    ]
    _sp_p95_values = [
        row["p95_latency_s"] for row in _sp_perf_samples
        if row["p95_latency_s"] is not None
    ]
    _sp_performance = {
        "version": 2,
        "transport": "http",
        "requests": _SP_PERFORMANCE_REQUESTS,
        "concurrency": _SP_PERFORMANCE_CONCURRENCY,
        "success": int(_sp_success_median),
        "fail": int(_sp_fail_median),
        "success_rate": round(_sp_median(
            [row["success_rate"] for row in _sp_perf_samples]
        ), 4),
        "throughput_rps": round(_sp_median(
            [row["throughput_rps"] for row in _sp_perf_samples]
        ), 4),
        "wall_s": round(_sp_median(
            [row["wall_s"] for row in _sp_perf_samples]
        ), 4),
        "p50_latency_s": (
            round(_sp_median(_sp_p50_values), 4) if _sp_p50_values else None
        ),
        "p95_latency_s": (
            round(_sp_median(_sp_p95_values), 4) if _sp_p95_values else None
        ),
        "errors": [
            dict(error, repetition=sample["repetition"])
            for sample in _sp_perf_samples
            for error in sample["errors"]
        ],
        "summary_method": "median",
        "protocol": {
            "requests_per_run": _SP_PERFORMANCE_REQUESTS,
            "concurrency": _SP_PERFORMANCE_CONCURRENCY,
            "warmup_requests": _SP_PERFORMANCE_WARMUP_REQUESTS,
            "repetitions": _SP_PERFORMANCE_REPETITIONS,
        },
        "samples": _sp_perf_samples,
    }
    if _sp_warmup is not None:
        _sp_performance["warmup"] = _sp_warmup
    _sp_print(
        "[성능] closed-loop 중앙값 · "
        + str(_SP_PERFORMANCE_REQUESTS) + "요청 × "
        + str(_SP_PERFORMANCE_REPETITIONS) + "회 · 동시성 "
        + str(_SP_PERFORMANCE_CONCURRENCY) + " · 대표 성공 "
        + str(_sp_performance["success"]) + " · "
        + str(_sp_performance["throughput_rps"]) + " req/s · p95 "
        + str(_sp_performance["p95_latency_s"]) + "s",
        flush=True,
    )

_sp_stop_http_server()

_sp_submission = {"team": _SP_TEAM, "answers": _sp_answers}
if _sp_doc_warnings or _sp_timeouts or _sp_env_warnings or _sp_performance:
    _sp_submission["meta"] = {"doc_name_violations": _sp_doc_warnings,
                              "timeout_qids": _sp_timeouts,
                              "env_warnings": _sp_env_warnings,
                              "transport": _SP_TRANSPORT}
    if _sp_performance:
        _sp_submission["meta"]["performance"] = _sp_performance
_sp_text = _sp_json.dumps(_sp_submission, ensure_ascii=False, indent=2, default=str)
with _sp_open(_SP_OUTPUT_PATH, "w", encoding="utf-8") as _sp_f:
    _sp_f.write(_sp_text)

_sp_print("[완료] " + str(len(_sp_answers)) + "문항 저장: " + _SP_OUTPUT_PATH
      + "  (총 " + str(round(_sp_time.time() - _sp_t0, 1)) + "s)", flush=True)
if _sp_errors:
    _sp_print("[경고] 실패 문항 " + str(len(_sp_errors)) + "건: "
          + ", ".join(_e["qid"] for _e in _sp_errors), flush=True)
if _sp_doc_warnings:
    _sp_print("[경고] 규정 ④ 위반 — 허용 목록 밖 문서명 " + str(len(_sp_doc_warnings)) + "건: "
          + ", ".join(sorted(set(_w["doc"] for _w in _sp_doc_warnings)))
          + "  → 해당 항목은 검색 점수가 0으로 채점됩니다. 허용(띄어쓰기 차이는 무관): "
          + ", ".join(_SP_ALLOWED_DOCS), flush=True)

if _SP_AUTO_DOWNLOAD:
    try:
        from google.colab import files as _sp_files
        _sp_files.download(_SP_OUTPUT_PATH)
        _sp_print("[다운로드] 브라우저 다운로드를 시작했습니다: " + _SP_OUTPUT_PATH, flush=True)
    except Exception as _sp_dl_exc:
        _sp_print("[다운로드] 자동 다운로드 실패(" + type(_sp_dl_exc).__name__ + ": " + str(_sp_dl_exc)
                  + ") — 좌측 파일 탭에서 " + _SP_OUTPUT_PATH + " 를 직접 내려받으세요.", flush=True)

_sp_print("SUBMISSION_RUNNER_DONE " + _sp_json.dumps(
    {"team": _SP_TEAM, "output_path": _SP_OUTPUT_PATH, "n_answers": len(_sp_answers),
     "n_errors": len(_sp_errors), "failed_qids": [_e["qid"] for _e in _sp_errors],
     "n_doc_violations": len(_sp_doc_warnings), "timeout_qids": _sp_timeouts,
     "env_warnings": _sp_env_warnings, "transport": _SP_TRANSPORT,
     "performance": _sp_performance},
    ensure_ascii=False), flush=True)
